**DEFINE PREDICTION POINT**

In [2]:
import pandas as pd

accepted = pd.read_csv("data/accepted_2007_to_2018Q4.csv")

print("Dataset shape:", accepted.shape)

C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\4294227807.py:3: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  accepted = pd.read_csv("data/accepted_2007_to_2018Q4.csv")


Dataset shape: (2260701, 151)


In [3]:
prediction_point = (
    "The model evaluates a newly submitted loan application "
    "using only information available at or before the "
    "application/underwriting decision."
)

print(prediction_point)

The model evaluates a newly submitted loan application using only information available at or before the application/underwriting decision.


In [4]:
print("Number of rows:", len(accepted))
print("Number of columns:", len(accepted.columns))

Number of rows: 2260701
Number of columns: 151


**INVESTIGATING loan_status**

In [5]:
print("Column name:", "loan_status")
print("Data type:", accepted["loan_status"].dtype)
print("Missing values:", accepted["loan_status"].isna().sum())

Column name: loan_status
Data type: object
Missing values: 33


In [6]:
loan_status_counts = accepted["loan_status"].value_counts(dropna=False)

print(loan_status_counts)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64


In [7]:
print("Number of unique loan statuses:", accepted["loan_status"].nunique(dropna=True))

Number of unique loan statuses: 9


In [8]:
print("Unique loan statuses:")

for status in accepted["loan_status"].dropna().unique():
    print("-", status)

Unique loan statuses:
- Fully Paid
- Current
- Charged Off
- In Grace Period
- Late (31-120 days)
- Late (16-30 days)
- Default
- Does not meet the credit policy. Status:Fully Paid
- Does not meet the credit policy. Status:Charged Off


In [9]:
special_statuses = accepted[
    accepted["loan_status"].astype("string").str.startswith(
        "Does not meet the credit policy",
        na=False
    )
]["loan_status"].value_counts()

print(special_statuses)

loan_status
Does not meet the credit policy. Status:Fully Paid     1988
Does not meet the credit policy. Status:Charged Off     761
Name: count, dtype: int64


In [10]:
unresolved_statuses = [
    "Current",
    "Late (16-30 days)",
    "Late (31-120 days)",
    "In Grace Period"
]

print(accepted["loan_status"].isin(unresolved_statuses).value_counts())

loan_status
False    1348132
True      912569
Name: count, dtype: int64


**Step 3: Define the Target.**

In [11]:
accepted_target = accepted.copy()

print("Working dataset shape:", accepted_target.shape)

Working dataset shape: (2260701, 151)


In [12]:
target_mapping = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Does not meet the credit policy. Status:Charged Off": 1
}

print("Target mapping:")
for status, target in target_mapping.items():
    print(f"{status} → {target}")

Target mapping:
Fully Paid → 0
Charged Off → 1
Default → 1
Does not meet the credit policy. Status:Fully Paid → 0
Does not meet the credit policy. Status:Charged Off → 1


In [13]:
accepted_target["target"] = accepted_target["loan_status"].map(target_mapping)

print(accepted_target[["loan_status", "target"]].head(20))

    loan_status  target
0    Fully Paid     0.0
1    Fully Paid     0.0
2    Fully Paid     0.0
3       Current     NaN
4    Fully Paid     0.0
5    Fully Paid     0.0
6    Fully Paid     0.0
7    Fully Paid     0.0
8    Fully Paid     0.0
9    Fully Paid     0.0
10      Current     NaN
11      Current     NaN
12   Fully Paid     0.0
13  Charged Off     1.0
14   Fully Paid     0.0
15   Fully Paid     0.0
16   Fully Paid     0.0
17   Fully Paid     0.0
18      Current     NaN
19   Fully Paid     0.0


In [14]:
target_status_summary = (
    accepted_target
    .groupby("loan_status", dropna=False)
    .agg(
        row_count=("loan_status", "size"),
        target=("target", "first")
    )
)

print(target_status_summary)

                                                    row_count  target
loan_status                                                          
Charged Off                                            268559     1.0
Current                                                878317     NaN
Default                                                    40     1.0
Does not meet the credit policy. Status:Charged...        761     1.0
Does not meet the credit policy. Status:Fully Paid       1988     0.0
Fully Paid                                            1076751     0.0
In Grace Period                                          8436     NaN
Late (16-30 days)                                        4349     NaN
Late (31-120 days)                                      21467     NaN
NaN                                                        33     NaN


In [15]:
print("Target distribution:")
print(accepted_target["target"].value_counts(dropna=False))

Target distribution:
target
0.0    1078739
NaN     912602
1.0     269360
Name: count, dtype: int64


In [16]:
target_defined = accepted_target["target"].notna()

print("Rows with defined target:", target_defined.sum())
print("Rows without defined target:", (~target_defined).sum())

Rows with defined target: 1348099
Rows without defined target: 912602


In [17]:
unresolved_statuses = accepted_target.loc[
    accepted_target["target"].isna(),
    "loan_status"
].value_counts(dropna=False)

print("Statuses without a final target:")
print(unresolved_statuses)

Statuses without a final target:
loan_status
Current               878317
Late (31-120 days)     21467
In Grace Period         8436
Late (16-30 days)       4349
NaN                       33
Name: count, dtype: int64


**Step 4 — Define Eligible Population.**

In [18]:
eligible_mask = accepted_target["target"].notna()

print("Eligible rows:", eligible_mask.sum())
print("Ineligible rows:", (~eligible_mask).sum())

Eligible rows: 1348099
Ineligible rows: 912602


In [19]:
ineligible_statuses = (
    accepted_target.loc[~eligible_mask, "loan_status"]
    .value_counts(dropna=False)
)

print("Ineligible loan statuses:")
print(ineligible_statuses)

Ineligible loan statuses:
loan_status
Current               878317
Late (31-120 days)     21467
In Grace Period         8436
Late (16-30 days)       4349
NaN                       33
Name: count, dtype: int64


In [20]:
accepted_eligible = accepted_target.loc[eligible_mask].copy()

print("Eligible dataset shape:", accepted_eligible.shape)

Eligible dataset shape: (1348099, 152)


In [21]:
print("Missing targets in eligible dataset:",
      accepted_eligible["target"].isna().sum())

print("\nTarget values present:")
print(accepted_eligible["target"].value_counts())

Missing targets in eligible dataset: 0

Target values present:
target
0.0    1078739
1.0     269360
Name: count, dtype: int64


In [22]:
print("Loan statuses in eligible population:")

print(
    accepted_eligible["loan_status"]
    .value_counts(dropna=False)
)

Loan statuses in eligible population:
loan_status
Fully Paid                                             1076751
Charged Off                                             268559
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [23]:
target_distribution = (
    accepted_eligible["target"]
    .value_counts()
    .sort_index()
)

print("Eligible population target distribution:")
print(target_distribution)

print("\nTarget proportions:")
print(accepted_eligible["target"].value_counts(normalize=True).sort_index())

Eligible population target distribution:
target
0.0    1078739
1.0     269360
Name: count, dtype: int64

Target proportions:
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64


In [24]:
total_rows = len(accepted_target)
eligible_rows = len(accepted_eligible)
ineligible_rows = total_rows - eligible_rows

print("Eligibility Summary")
print("-------------------")
print("Total accepted rows :", total_rows)
print("Eligible rows       :", eligible_rows)
print("Ineligible rows     :", ineligible_rows)
print("Eligibility rate    :", f"{eligible_rows / total_rows:.2%}")

Eligibility Summary
-------------------
Total accepted rows : 2260701
Eligible rows       : 1348099
Ineligible rows     : 912602
Eligibility rate    : 59.63%


**STEP 5:-AUDIT EVERY COLUMN**

In [25]:
print("Dataset being audited: accepted_eligible")
print("Rows:", accepted_eligible.shape[0])
print("Columns:", accepted_eligible.shape[1])

Dataset being audited: accepted_eligible
Rows: 1348099
Columns: 152


In [26]:
print("All columns in accepted_eligible:\n")

for i, column in enumerate(accepted_eligible.columns, start=1):
    print(f"{i:3}. {column}")

All columns in accepted_eligible:

  1. id
  2. member_id
  3. loan_amnt
  4. funded_amnt
  5. funded_amnt_inv
  6. term
  7. int_rate
  8. installment
  9. grade
 10. sub_grade
 11. emp_title
 12. emp_length
 13. home_ownership
 14. annual_inc
 15. verification_status
 16. issue_d
 17. loan_status
 18. pymnt_plan
 19. url
 20. desc
 21. purpose
 22. title
 23. zip_code
 24. addr_state
 25. dti
 26. delinq_2yrs
 27. earliest_cr_line
 28. fico_range_low
 29. fico_range_high
 30. inq_last_6mths
 31. mths_since_last_delinq
 32. mths_since_last_record
 33. open_acc
 34. pub_rec
 35. revol_bal
 36. revol_util
 37. total_acc
 38. initial_list_status
 39. out_prncp
 40. out_prncp_inv
 41. total_pymnt
 42. total_pymnt_inv
 43. total_rec_prncp
 44. total_rec_int
 45. total_rec_late_fee
 46. recoveries
 47. collection_recovery_fee
 48. last_pymnt_d
 49. last_pymnt_amnt
 50. next_pymnt_d
 51. last_credit_pull_d
 52. last_fico_range_high
 53. last_fico_range_low
 54. collections_12_mths_ex_med
 55

In [27]:
column_audit = pd.DataFrame({
    "column": accepted_eligible.columns,
    "dtype": accepted_eligible.dtypes.astype(str).values,
    "non_null_count": accepted_eligible.notna().sum().values,
    "missing_count": accepted_eligible.isna().sum().values,
    "missing_percentage": (
        accepted_eligible.isna().mean().values * 100
    ),
    "unique_values": accepted_eligible.nunique(dropna=True).values
})

column_audit

,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
0,id,object,1348099,0,0.000000,1348099
1,member_id,float64,0,1348099,100.000000,0
2,loan_amnt,float64,1348099,0,0.000000,1560
3,funded_amnt,float64,1348099,0,0.000000,1560
4,funded_amnt_inv,float64,1348099,0,0.000000,10041
...,...,...,...,...,...,...
147,settlement_date,object,33292,1314807,97.530448,90
148,settlement_amount,float64,33292,1314807,97.530448,21762
149,settlement_percentage,float64,33292,1314807,97.530448,2062
150,settlement_term,float64,33292,1314807,97.530448,40


In [28]:
column_audit_by_order = column_audit.copy()

column_audit_by_order.index = range(1, len(column_audit_by_order) + 1)

column_audit_by_order

,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
1,id,object,1348099,0,0.000000,1348099
2,member_id,float64,0,1348099,100.000000,0
3,loan_amnt,float64,1348099,0,0.000000,1560
4,funded_amnt,float64,1348099,0,0.000000,1560
5,funded_amnt_inv,float64,1348099,0,0.000000,10041
...,...,...,...,...,...,...
148,settlement_date,object,33292,1314807,97.530448,90
149,settlement_amount,float64,33292,1314807,97.530448,21762
150,settlement_percentage,float64,33292,1314807,97.530448,2062
151,settlement_term,float64,33292,1314807,97.530448,40


In [29]:
print("Data type distribution:")
print(accepted_eligible.dtypes.value_counts())

Data type distribution:
float64    114
object      38
Name: count, dtype: int64


In [30]:
missing_summary = (
    column_audit[
        column_audit["missing_count"] > 0
    ]
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
)

missing_summary

,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
1,member_id,float64,0,1348099,100.000000,0
49,next_pymnt_d,object,2789,1345310,99.793116,99
140,orig_projected_additional_accrued_interest,float64,3761,1344338,99.721015,3482
131,hardship_status,object,5756,1342343,99.573028,2
137,hardship_length,float64,5756,1342343,99.573028,1
...,...,...,...,...,...,...
25,delinq_2yrs,float64,1348070,29,0.002151,31
60,acc_now_delinq,float64,1348070,29,0.002151,8
83,delinq_amnt,float64,1348070,29,0.002151,2006
13,annual_inc,float64,1348095,4,0.000297,64462


In [31]:
high_missing = column_audit[
    column_audit["missing_percentage"] >= 80
].sort_values(
    by="missing_percentage",
    ascending=False
)

print("Columns with >= 80% missing values:")
print("Count:", len(high_missing))

high_missing

Columns with >= 80% missing values:
Count: 40


,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
1,member_id,float64,0,1348099,100.000000,0
49,next_pymnt_d,object,2789,1345310,99.793116,99
140,orig_projected_additional_accrued_interest,float64,3761,1344338,99.721015,3482
130,hardship_reason,object,5756,1342343,99.573028,9
142,hardship_last_payment_amount,float64,5756,1342343,99.573028,5050
141,hardship_payoff_balance_amount,float64,5756,1342343,99.573028,5750
139,hardship_loan_status,object,5756,1342343,99.573028,5
138,hardship_dpd,float64,5756,1342343,99.573028,34
137,hardship_length,float64,5756,1342343,99.573028,1
136,payment_plan_start_date,object,5756,1342343,99.573028,25


In [32]:
cardinality_summary = column_audit[
    ["column", "dtype", "unique_values"]
].sort_values(
    by="unique_values",
    ascending=True
)

cardinality_summary

,column,dtype,unique_values
1,member_id,float64,0
129,hardship_type,object,1
128,hardship_flag,object,1
55,policy_code,float64,1
137,hardship_length,float64,1
...,...,...,...
48,last_pymnt_amnt,float64,693799
41,total_pymnt_inv,float64,1014400
40,total_pymnt,float64,1264380
18,url,object,1348099


In [33]:
constant_columns = column_audit[
    column_audit["unique_values"] <= 1
]

print("Constant / single-value columns:")
print("Count:", len(constant_columns))

constant_columns

Constant / single-value columns:
Count: 7


,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
1,member_id,float64,0,1348099,100.000000,0
17,pymnt_plan,object,1348099,0,0.000000,1
55,policy_code,float64,1348099,0,0.000000,1
128,hardship_flag,object,1348099,0,0.000000,1
129,hardship_type,object,5756,1342343,99.573028,1
132,deferral_term,float64,5756,1342343,99.573028,1
137,hardship_length,float64,5756,1342343,99.573028,1


In [34]:
sample_values = pd.DataFrame({
    "column": accepted_eligible.columns,
    "sample_values": [
        accepted_eligible[column]
        .dropna()
        .head(5)
        .tolist()
        for column in accepted_eligible.columns
    ]
})

sample_values

,column,sample_values
0,id,"[68407277, 68355089, 68341763, 68476807, 68426..."
1,member_id,[]
2,loan_amnt,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
3,funded_amnt,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
4,funded_amnt_inv,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
...,...,...
147,settlement_date,"[Sep-2017, Nov-2017, Jan-2018, May-2018, Sep-2..."
148,settlement_amount,"[9915.0, 2761.0, 4382.0, 3513.0, 13403.0]"
149,settlement_percentage,"[55.0, 50.01, 44.99, 45.01, 45.0]"
150,settlement_term,"[1.0, 10.0, 12.0, 16.0, 18.0]"


In [35]:
column_audit_full = column_audit.merge(
    sample_values,
    on="column",
    how="left"
)

column_audit_full

,column,dtype,non_null_count,missing_count,missing_percentage,unique_values,sample_values
0,id,object,1348099,0,0.000000,1348099,"[68407277, 68355089, 68341763, 68476807, 68426..."
1,member_id,float64,0,1348099,100.000000,0,[]
2,loan_amnt,float64,1348099,0,0.000000,1560,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
3,funded_amnt,float64,1348099,0,0.000000,1560,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
4,funded_amnt_inv,float64,1348099,0,0.000000,10041,"[3600.0, 24700.0, 20000.0, 10400.0, 11950.0]"
...,...,...,...,...,...,...,...
147,settlement_date,object,33292,1314807,97.530448,90,"[Sep-2017, Nov-2017, Jan-2018, May-2018, Sep-2..."
148,settlement_amount,float64,33292,1314807,97.530448,21762,"[9915.0, 2761.0, 4382.0, 3513.0, 13403.0]"
149,settlement_percentage,float64,33292,1314807,97.530448,2062,"[55.0, 50.01, 44.99, 45.01, 45.0]"
150,settlement_term,float64,33292,1314807,97.530448,40,"[1.0, 10.0, 12.0, 16.0, 18.0]"


In [36]:
column_audit_full.to_csv(
    "reports/preprocessing_v1_audits/accepted_column_audit.csv",
    index=False
)

print("Column audit saved to accepted_column_audit.csv")

Column audit saved to accepted_column_audit.csv


**Step 6 — Application-Time vs Future-Time Audit**

In [37]:
# Step 6 — Full Timing & Feature-Role Audit
# Cell 1

timing_audit = pd.DataFrame({
    "column": accepted_eligible.columns
})

print("Total columns to classify:", len(timing_audit))
print("Expected columns:", 152)

Total columns to classify: 152
Expected columns: 152


In [38]:
# Cell 2

target_columns = [
    "loan_status",
    "target"
]

print("Target columns:")
for column in target_columns:
    print("-", column)

Target columns:
- loan_status
- target


In [39]:
# Cell 3

identifier_columns = [
    "id",
    "member_id"
]

print("Identifier columns:")
for column in identifier_columns:
    print("-", column)

Identifier columns:
- id
- member_id


In [40]:
# Cell 4

administrative_columns = [
    "url",
    "policy_code",
    "initial_list_status"
]

print("Administrative / constant / non-feature columns:")
for column in administrative_columns:
    print("-", column)

Administrative / constant / non-feature columns:
- url
- policy_code
- initial_list_status


In [41]:
# Cell 5

lc_derived_columns = [
    "int_rate",
    "installment",
    "grade",
    "sub_grade"
]

print("LendingClub-derived / potential leakage-risk columns:")
for column in lc_derived_columns:
    print("-", column)

LendingClub-derived / potential leakage-risk columns:
- int_rate
- installment
- grade
- sub_grade


In [42]:
# Cell 6

text_columns = [
    "emp_title",
    "desc",
    "title",
    "zip_code"
]

print("Text / high-cardinality columns:")
for column in text_columns:
    print("-", column)

Text / high-cardinality columns:
- emp_title
- desc
- title
- zip_code


In [43]:
# Cell 7

date_columns = [
    "earliest_cr_line"
]

print("Application-time / historical date columns:")
for column in date_columns:
    print("-", column)

Application-time / historical date columns:
- earliest_cr_line


In [44]:
# Cell 8

post_prediction_columns = [
    "funded_amnt",
    "funded_amnt_inv",
    "issue_d",
    "pymnt_plan",
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "disbursement_method",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

print("Post-prediction columns:", len(post_prediction_columns))

for column in post_prediction_columns:
    print("-", column)

Post-prediction columns: 42
- funded_amnt
- funded_amnt_inv
- issue_d
- pymnt_plan
- out_prncp
- out_prncp_inv
- total_pymnt
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- recoveries
- collection_recovery_fee
- last_pymnt_d
- last_pymnt_amnt
- next_pymnt_d
- last_credit_pull_d
- last_fico_range_high
- last_fico_range_low
- hardship_flag
- hardship_type
- hardship_reason
- hardship_status
- deferral_term
- hardship_amount
- hardship_start_date
- hardship_end_date
- payment_plan_start_date
- hardship_length
- hardship_dpd
- hardship_loan_status
- orig_projected_additional_accrued_interest
- hardship_payoff_balance_amount
- hardship_last_payment_amount
- disbursement_method
- debt_settlement_flag
- debt_settlement_flag_date
- settlement_status
- settlement_date
- settlement_amount
- settlement_percentage
- settlement_term


In [45]:
# Cell 9

all_columns = set(accepted_eligible.columns)

already_classified = (
    set(target_columns)
    | set(identifier_columns)
    | set(administrative_columns)
    | set(lc_derived_columns)
    | set(text_columns)
    | set(date_columns)
    | set(post_prediction_columns)
)

application_time_columns = sorted(
    all_columns - already_classified
)

print("Application-time feature candidates:", len(application_time_columns))

for column in application_time_columns:
    print("-", column)

Application-time feature candidates: 94
- acc_now_delinq
- acc_open_past_24mths
- addr_state
- all_util
- annual_inc
- annual_inc_joint
- application_type
- avg_cur_bal
- bc_open_to_buy
- bc_util
- chargeoff_within_12_mths
- collections_12_mths_ex_med
- delinq_2yrs
- delinq_amnt
- dti
- dti_joint
- emp_length
- fico_range_high
- fico_range_low
- home_ownership
- il_util
- inq_fi
- inq_last_12m
- inq_last_6mths
- loan_amnt
- max_bal_bc
- mo_sin_old_il_acct
- mo_sin_old_rev_tl_op
- mo_sin_rcnt_rev_tl_op
- mo_sin_rcnt_tl
- mort_acc
- mths_since_last_delinq
- mths_since_last_major_derog
- mths_since_last_record
- mths_since_rcnt_il
- mths_since_recent_bc
- mths_since_recent_bc_dlq
- mths_since_recent_inq
- mths_since_recent_revol_delinq
- num_accts_ever_120_pd
- num_actv_bc_tl
- num_actv_rev_tl
- num_bc_sats
- num_bc_tl
- num_il_tl
- num_op_rev_tl
- num_rev_accts
- num_rev_tl_bal_gt_0
- num_sats
- num_tl_120dpd_2m
- num_tl_30dpd
- num_tl_90g_dpd_24m
- num_tl_op_past_12m
- open_acc
- open_a

In [46]:
# Cell 10

classification_map = {}

for column in target_columns:
    classification_map[column] = "target"

for column in identifier_columns:
    classification_map[column] = "identifier"

for column in administrative_columns:
    classification_map[column] = "administrative / constant / non-feature"

for column in lc_derived_columns:
    classification_map[column] = "LC-derived / potential leakage risk"

for column in text_columns:
    classification_map[column] = "text / high-cardinality"

for column in date_columns:
    classification_map[column] = "application-time date / historical date"

for column in post_prediction_columns:
    classification_map[column] = "post-prediction"

for column in application_time_columns:
    classification_map[column] = "application-time feature"

timing_audit["timing_class"] = (
    timing_audit["column"].map(classification_map)
)

print(timing_audit)

                    column              timing_class
0                       id                identifier
1                member_id                identifier
2                loan_amnt  application-time feature
3              funded_amnt           post-prediction
4          funded_amnt_inv           post-prediction
..                     ...                       ...
147        settlement_date           post-prediction
148      settlement_amount           post-prediction
149  settlement_percentage           post-prediction
150        settlement_term           post-prediction
151                 target                    target

[152 rows x 2 columns]


In [47]:
# Cell 11

unclassified = timing_audit[
    timing_audit["timing_class"].isna()
]

print("Total columns:", len(timing_audit))
print("Classified columns:", timing_audit["timing_class"].notna().sum())
print("Unclassified columns:", len(unclassified))

if len(unclassified) == 0:
    print("\n✅ Every column has exactly one classification.")
else:
    print("\n❌ Unclassified columns:")
    print(unclassified["column"].tolist())

Total columns: 152
Classified columns: 152
Unclassified columns: 0

✅ Every column has exactly one classification.


In [48]:
# Cell 12

category_sets = {
    "target": set(target_columns),
    "identifier": set(identifier_columns),
    "administrative": set(administrative_columns),
    "LC-derived": set(lc_derived_columns),
    "text/high-cardinality": set(text_columns),
    "date": set(date_columns),
    "post-prediction": set(post_prediction_columns),
    "application-time": set(application_time_columns)
}

all_category_columns = []

for category, columns in category_sets.items():
    all_category_columns.extend(columns)

duplicate_membership = (
    pd.Series(all_category_columns)
    .value_counts()
)

duplicate_membership = duplicate_membership[
    duplicate_membership > 1
]

print("Columns appearing in multiple categories:")

if len(duplicate_membership) == 0:
    print("None ✅")
else:
    print(duplicate_membership)

Columns appearing in multiple categories:
None ✅


In [49]:
# Cell 13

classification_counts = (
    timing_audit["timing_class"]
    .value_counts()
)

print("Classification counts:")
print(classification_counts)

Classification counts:
timing_class
application-time feature                   94
post-prediction                            42
LC-derived / potential leakage risk         4
text / high-cardinality                     4
administrative / constant / non-feature     3
identifier                                  2
target                                      2
application-time date / historical date     1
Name: count, dtype: int64


In [50]:
# Cell 14

timing_audit_by_order = timing_audit.copy()

timing_audit_by_order.index = range(
    1,
    len(timing_audit_by_order) + 1
)

timing_audit_by_order

,column,timing_class
1,id,identifier
2,member_id,identifier
3,loan_amnt,application-time feature
4,funded_amnt,post-prediction
5,funded_amnt_inv,post-prediction
...,...,...
148,settlement_date,post-prediction
149,settlement_amount,post-prediction
150,settlement_percentage,post-prediction
151,settlement_term,post-prediction


In [51]:
# Cell 15

reason_map = {}

for column in target_columns:
    reason_map[column] = (
        "Defines the supervised learning target/outcome."
    )

for column in identifier_columns:
    reason_map[column] = (
        "Identifier; does not represent borrower risk information."
    )

reason_map["url"] = (
    "Administrative LendingClub listing URL; not a borrower risk feature."
)

reason_map["policy_code"] = (
    "Constant/single-value field; no predictive variance."
)

reason_map["initial_list_status"] = (
    "Platform listing/origination attribute; not an underwriting borrower feature."
)

for column in lc_derived_columns:
    reason_map[column] = (
        "LendingClub-derived loan risk/terms associated with its "
        "underwriting/pricing process; flagged for potential leakage "
        "relative to an independent pre-underwriting model."
    )

for column in text_columns:
    reason_map[column] = (
        "Available at application but requires specialized "
        "text/high-cardinality representation."
    )

reason_map["earliest_cr_line"] = (
    "Historical credit date available at application; requires "
    "date-derived representation."
)

for column in post_prediction_columns:
    reason_map[column] = (
        "Recorded at or after origination/servicing and therefore "
        "not safely available at the pre-underwriting prediction point."
    )

for column in application_time_columns:
    reason_map[column] = (
        "Applicant/credit information that can be available at "
        "the application/underwriting stage."
    )

timing_audit_by_order["reason"] = (
    timing_audit_by_order["column"].map(reason_map)
)

timing_audit_by_order

,column,timing_class,reason
1,id,identifier,Identifier; does not represent borrower risk i...
2,member_id,identifier,Identifier; does not represent borrower risk i...
3,loan_amnt,application-time feature,Applicant/credit information that can be avail...
4,funded_amnt,post-prediction,Recorded at or after origination/servicing and...
5,funded_amnt_inv,post-prediction,Recorded at or after origination/servicing and...
...,...,...,...
148,settlement_date,post-prediction,Recorded at or after origination/servicing and...
149,settlement_amount,post-prediction,Recorded at or after origination/servicing and...
150,settlement_percentage,post-prediction,Recorded at or after origination/servicing and...
151,settlement_term,post-prediction,Recorded at or after origination/servicing and...


In [52]:
# Cell 16

print("FINAL STEP 6 VERIFICATION")
print("=========================")

print("Expected columns        :", 152)
print("Actual columns          :", len(timing_audit_by_order))
print(
    "Unclassified columns    :",
    timing_audit_by_order["timing_class"].isna().sum()
)
print(
    "Missing reasons         :",
    timing_audit_by_order["reason"].isna().sum()
)

if (
    len(timing_audit_by_order) == 152
    and timing_audit_by_order["timing_class"].notna().all()
    and timing_audit_by_order["reason"].notna().all()
):
    print("\n✅ STEP 6 STRUCTURAL AUDIT COMPLETE")
else:
    print("\n❌ STEP 6 STILL NEEDS REVIEW")

FINAL STEP 6 VERIFICATION
Expected columns        : 152
Actual columns          : 152
Unclassified columns    : 0
Missing reasons         : 0

✅ STEP 6 STRUCTURAL AUDIT COMPLETE


In [53]:
# Cell 17

timing_audit_by_order.to_csv(
    "reports/preprocessing_v1_audits/accepted_timing_audit.csv",
    index=False
)

print("Saved: accepted_timing_audit.csv")

Saved: accepted_timing_audit.csv


**STEP 7. REMOVE TARGET / LEAKAGE / IDS / UNSUITABLE**

In [54]:
# Step 7
# Cell 1 — Create modeling candidate dataset

accepted_model_candidates = accepted_eligible.copy()

print("Original accepted_eligible shape:")
print(accepted_eligible.shape)

print("\nModel candidate shape:")
print(accepted_model_candidates.shape)

Original accepted_eligible shape:
(1348099, 152)

Model candidate shape:
(1348099, 152)


In [55]:
# Cell 2 — Columns excluded from the modeling candidate dataset

step7_remove_columns = [
    # Target
    "loan_status",
    "target",

    # Identifiers
    "id",
    "member_id",

    # Administrative / non-feature
    "url",
    "policy_code",
    "initial_list_status",

    # Post-prediction / future information
    "funded_amnt",
    "funded_amnt_inv",
    "issue_d",
    "pymnt_plan",
    "out_prncp",
    "out_prncp_inv",
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "next_pymnt_d",
    "last_credit_pull_d",
    "last_fico_range_high",
    "last_fico_range_low",
    "hardship_flag",
    "hardship_type",
    "hardship_reason",
    "hardship_status",
    "deferral_term",
    "hardship_amount",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "hardship_length",
    "hardship_dpd",
    "hardship_loan_status",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount",
    "hardship_last_payment_amount",
    "disbursement_method",
    "debt_settlement_flag",
    "debt_settlement_flag_date",
    "settlement_status",
    "settlement_date",
    "settlement_amount",
    "settlement_percentage",
    "settlement_term"
]

print("Columns scheduled for removal:", len(step7_remove_columns))

Columns scheduled for removal: 49


In [56]:
# Cell 3 — Verify removal list

missing_from_dataset = [
    column
    for column in step7_remove_columns
    if column not in accepted_model_candidates.columns
]

if len(missing_from_dataset) == 0:
    print("✅ Every removal column exists in the dataset.")
else:
    print("❌ These columns were not found:")
    print(missing_from_dataset)

✅ Every removal column exists in the dataset.


In [57]:
# Cell 4 — Check duplicate removal definitions

from collections import Counter

removal_counts = Counter(step7_remove_columns)

duplicate_removals = {
    column: count
    for column, count in removal_counts.items()
    if count > 1
}

if len(duplicate_removals) == 0:
    print("✅ No duplicate columns in the removal list.")
else:
    print("❌ Duplicate removal definitions found:")
    print(duplicate_removals)

✅ No duplicate columns in the removal list.


In [58]:
# Cell 5 — Build removal audit

removal_reason_map = {}

# Target
for column in ["loan_status", "target"]:
    removal_reason_map[column] = "Target / outcome information"

# Identifiers
for column in ["id", "member_id"]:
    removal_reason_map[column] = "Identifier, not a predictive feature"

# Administrative
removal_reason_map["url"] = (
    "Administrative LendingClub listing URL"
)

removal_reason_map["policy_code"] = (
    "Constant / non-informative field"
)

removal_reason_map["initial_list_status"] = (
    "Platform listing/origination attribute, not an underwriting feature"
)

# Future information
for column in post_prediction_columns:
    removal_reason_map[column] = (
        "Post-prediction / post-origination information"
    )

step7_removal_audit = pd.DataFrame({
    "column": step7_remove_columns,
    "reason": [
        removal_reason_map[column]
        for column in step7_remove_columns
    ]
})

step7_removal_audit

,column,reason
0,loan_status,Target / outcome information
1,target,Target / outcome information
2,id,"Identifier, not a predictive feature"
3,member_id,"Identifier, not a predictive feature"
4,url,Administrative LendingClub listing URL
5,policy_code,Constant / non-informative field
6,initial_list_status,"Platform listing/origination attribute, not an..."
7,funded_amnt,Post-prediction / post-origination information
8,funded_amnt_inv,Post-prediction / post-origination information
9,issue_d,Post-prediction / post-origination information


In [59]:
# Cell 6 — Removal category summary

step7_removal_categories = {
    "Target": [
        "loan_status",
        "target"
    ],

    "Identifier": [
        "id",
        "member_id"
    ],

    "Administrative / Non-feature": [
        "url",
        "policy_code",
        "initial_list_status"
    ],

    "Post-prediction": post_prediction_columns
}

for category, columns in step7_removal_categories.items():
    print(f"\n{category}: {len(columns)} columns")
    for column in columns:
        print("  -", column)


Target: 2 columns
  - loan_status
  - target

Identifier: 2 columns
  - id
  - member_id

Administrative / Non-feature: 3 columns
  - url
  - policy_code
  - initial_list_status

Post-prediction: 42 columns
  - funded_amnt
  - funded_amnt_inv
  - issue_d
  - pymnt_plan
  - out_prncp
  - out_prncp_inv
  - total_pymnt
  - total_pymnt_inv
  - total_rec_prncp
  - total_rec_int
  - total_rec_late_fee
  - recoveries
  - collection_recovery_fee
  - last_pymnt_d
  - last_pymnt_amnt
  - next_pymnt_d
  - last_credit_pull_d
  - last_fico_range_high
  - last_fico_range_low
  - hardship_flag
  - hardship_type
  - hardship_reason
  - hardship_status
  - deferral_term
  - hardship_amount
  - hardship_start_date
  - hardship_end_date
  - payment_plan_start_date
  - hardship_length
  - hardship_dpd
  - hardship_loan_status
  - orig_projected_additional_accrued_interest
  - hardship_payoff_balance_amount
  - hardship_last_payment_amount
  - disbursement_method
  - debt_settlement_flag
  - debt_settleme

In [60]:
# Cell 7 — Verify LC-derived fields remain candidates

for column in lc_derived_columns:
    if column in step7_remove_columns:
        print(f"❌ ERROR: {column} is scheduled for removal")
    else:
        print(f"✅ {column} remains in the candidate dataset")

✅ int_rate remains in the candidate dataset
✅ installment remains in the candidate dataset
✅ grade remains in the candidate dataset
✅ sub_grade remains in the candidate dataset


In [61]:
# Cell 8 — Remove Step 7 exclusions

accepted_model_candidates = accepted_model_candidates.drop(
    columns=step7_remove_columns
)

print("Step 7 candidate dataset shape:")
print(accepted_model_candidates.shape)

Step 7 candidate dataset shape:
(1348099, 103)


In [62]:
# Cell 9 — Verify exclusions

still_present = [
    column
    for column in step7_remove_columns
    if column in accepted_model_candidates.columns
]

if len(still_present) == 0:
    print("✅ All Step 7 exclusion columns were removed.")
else:
    print("❌ These columns are still present:")
    print(still_present)

✅ All Step 7 exclusion columns were removed.


In [63]:
# Cell 10 — Target leakage check

target_still_present = [
    column
    for column in ["loan_status", "target"]
    if column in accepted_model_candidates.columns
]

if len(target_still_present) == 0:
    print("✅ Target columns are absent from model candidates.")
else:
    print("❌ Target leakage detected:")
    print(target_still_present)

✅ Target columns are absent from model candidates.


In [64]:
# Cell 11 — Identifier check

identifiers_still_present = [
    column
    for column in ["id", "member_id"]
    if column in accepted_model_candidates.columns
]

if len(identifiers_still_present) == 0:
    print("✅ Identifier columns are absent.")
else:
    print("❌ Identifier columns still present:")
    print(identifiers_still_present)

✅ Identifier columns are absent.


In [65]:
# Cell 12 — Future information leakage check

future_still_present = [
    column
    for column in post_prediction_columns
    if column in accepted_model_candidates.columns
]

if len(future_still_present) == 0:
    print("✅ No Step 6 post-prediction columns remain.")
else:
    print("❌ Post-prediction columns still present:")
    print(future_still_present)

✅ No Step 6 post-prediction columns remain.


In [66]:
# Cell 13 — LC-derived field audit

lc_derived_present = [
    column
    for column in lc_derived_columns
    if column in accepted_model_candidates.columns
]

print("LC-derived fields currently retained:")
for column in lc_derived_present:
    print("-", column)

print("\nThese remain intentionally flagged for methodological review.")# Cell 13 — LC-derived field audit

lc_derived_present = [
    column
    for column in lc_derived_columns
    if column in accepted_model_candidates.columns
]

print("LC-derived fields currently retained:")
for column in lc_derived_present:
    print("-", column)

print("\nThese remain intentionally flagged for methodological review.")

LC-derived fields currently retained:
- int_rate
- installment
- grade
- sub_grade

These remain intentionally flagged for methodological review.
LC-derived fields currently retained:
- int_rate
- installment
- grade
- sub_grade

These remain intentionally flagged for methodological review.


In [67]:
# Cell 14 — Remaining candidate columns

print("Remaining columns:", len(accepted_model_candidates.columns))

for i, column in enumerate(
    accepted_model_candidates.columns,
    start=1
):
    print(f"{i:3}. {column}")

Remaining columns: 103
  1. loan_amnt
  2. term
  3. int_rate
  4. installment
  5. grade
  6. sub_grade
  7. emp_title
  8. emp_length
  9. home_ownership
 10. annual_inc
 11. verification_status
 12. desc
 13. purpose
 14. title
 15. zip_code
 16. addr_state
 17. dti
 18. delinq_2yrs
 19. earliest_cr_line
 20. fico_range_low
 21. fico_range_high
 22. inq_last_6mths
 23. mths_since_last_delinq
 24. mths_since_last_record
 25. open_acc
 26. pub_rec
 27. revol_bal
 28. revol_util
 29. total_acc
 30. collections_12_mths_ex_med
 31. mths_since_last_major_derog
 32. application_type
 33. annual_inc_joint
 34. dti_joint
 35. verification_status_joint
 36. acc_now_delinq
 37. tot_coll_amt
 38. tot_cur_bal
 39. open_acc_6m
 40. open_act_il
 41. open_il_12m
 42. open_il_24m
 43. mths_since_rcnt_il
 44. total_bal_il
 45. il_util
 46. open_rv_12m
 47. open_rv_24m
 48. max_bal_bc
 49. all_util
 50. total_rev_hi_lim
 51. inq_fi
 52. total_cu_tl
 53. inq_last_12m
 54. acc_open_past_24mths
 55. avg_

In [68]:
# Cell 15 — Duplicate column-name check

duplicate_column_names = (
    accepted_model_candidates.columns[
        accepted_model_candidates.columns.duplicated()
    ]
    .tolist()
)

if len(duplicate_column_names) == 0:
    print("✅ No duplicate column names.")
else:
    print("❌ Duplicate column names found:")
    print(duplicate_column_names)

✅ No duplicate column names.


In [69]:
# Cell 16 — Final Step 7 verification

checks = {
    "Target removed": all(
        column not in accepted_model_candidates.columns
        for column in ["loan_status", "target"]
    ),

    "Identifiers removed": all(
        column not in accepted_model_candidates.columns
        for column in ["id", "member_id"]
    ),

    "Administrative fields removed": all(
        column not in accepted_model_candidates.columns
        for column in ["url", "policy_code", "initial_list_status"]
    ),

    "Post-prediction fields removed": all(
        column not in accepted_model_candidates.columns
        for column in post_prediction_columns
    ),

    "No duplicate column names": len(duplicate_column_names) == 0,

    "LC-derived fields retained for review": all(
        column in accepted_model_candidates.columns
        for column in lc_derived_columns
    )
}

print("STEP 7 INTEGRITY CHECK")
print("======================")

for check, result in checks.items():
    print(f"{'✅' if result else '❌'} {check}")

if all(checks.values()):
    print("\n🟢 STEP 7 STRUCTURAL CHECK PASSED")
else:
    print("\n🔴 STEP 7 NEEDS REVIEW")

STEP 7 INTEGRITY CHECK
✅ Target removed
✅ Identifiers removed
✅ Administrative fields removed
✅ Post-prediction fields removed
✅ No duplicate column names
✅ LC-derived fields retained for review

🟢 STEP 7 STRUCTURAL CHECK PASSED


In [70]:
# Cell 17 — Save Step 7 audit

step7_removal_audit.to_csv(
    "reports/preprocessing_v1_audits/accepted_step7_removal_audit.csv",
    index=False
)

print("Saved:")
print("reports/preprocessing_v1_audits/accepted_step7_removal_audit.csv")

Saved:
accepted_step7_removal_audit.csv


In [71]:
# Cell 18 — Save Step 7 candidate dataset

accepted_model_candidates.to_csv(
    "data/interim/accepted_model_candidates.csv",
    index=False
)

print("Saved:")
print("data/interim/accepted_model_candidates.csv")

print("\nShape:", accepted_model_candidates.shape)

Saved:
accepted_model_candidates.csv

Shape: (1348099, 103)


**STEP 8. DATA QUALITY INVESTIGATION**

In [72]:
# STEP 8
# Cell 1 — Verify Step 7 dataset

print("STEP 8 — DATA QUALITY AUDIT")
print("=" * 60)

print("Dataset: accepted_model_candidates")
print("Rows   :", accepted_model_candidates.shape[0])
print("Columns:", accepted_model_candidates.shape[1])

print("\nTarget column present:",
      "target" in accepted_model_candidates.columns)

print("\nStep 7 candidate dataset verified.")

STEP 8 — DATA QUALITY AUDIT
Dataset: accepted_model_candidates
Rows   : 1348099
Columns: 103

Target column present: False

Step 7 candidate dataset verified.


In [73]:
# Cell 2 — Column inventory

print("All columns in accepted_model_candidates:\n")

for i, column in enumerate(accepted_model_candidates.columns, start=1):
    print(f"{i:3}. {column}")

print("\nTotal columns:", len(accepted_model_candidates.columns))

All columns in accepted_model_candidates:

  1. loan_amnt
  2. term
  3. int_rate
  4. installment
  5. grade
  6. sub_grade
  7. emp_title
  8. emp_length
  9. home_ownership
 10. annual_inc
 11. verification_status
 12. desc
 13. purpose
 14. title
 15. zip_code
 16. addr_state
 17. dti
 18. delinq_2yrs
 19. earliest_cr_line
 20. fico_range_low
 21. fico_range_high
 22. inq_last_6mths
 23. mths_since_last_delinq
 24. mths_since_last_record
 25. open_acc
 26. pub_rec
 27. revol_bal
 28. revol_util
 29. total_acc
 30. collections_12_mths_ex_med
 31. mths_since_last_major_derog
 32. application_type
 33. annual_inc_joint
 34. dti_joint
 35. verification_status_joint
 36. acc_now_delinq
 37. tot_coll_amt
 38. tot_cur_bal
 39. open_acc_6m
 40. open_act_il
 41. open_il_12m
 42. open_il_24m
 43. mths_since_rcnt_il
 44. total_bal_il
 45. il_util
 46. open_rv_12m
 47. open_rv_24m
 48. max_bal_bc
 49. all_util
 50. total_rev_hi_lim
 51. inq_fi
 52. total_cu_tl
 53. inq_last_12m
 54. acc_open_p

In [74]:
# Cell 3 — Basic dataset information

print("Dataset information")
print("=" * 60)

print("Shape:", accepted_model_candidates.shape)
print("Memory usage:")
print(
    accepted_model_candidates.memory_usage(deep=True)
    .sum() / (1024 ** 2),
    "MB"
)

print("\nData types:")
print(accepted_model_candidates.dtypes.value_counts())

Dataset information
Shape: (1348099, 103)
Memory usage:
2018.231141090393 MB

Data types:
float64    87
object     16
Name: count, dtype: int64


In [75]:
# Cell 4 — Missing-value audit

missing_audit = pd.DataFrame({
    "column": accepted_model_candidates.columns,
    "dtype": accepted_model_candidates.dtypes.astype(str).values,
    "non_null_count": accepted_model_candidates.notna().sum().values,
    "missing_count": accepted_model_candidates.isna().sum().values,
    "missing_percentage": (
        accepted_model_candidates.isna().mean().values * 100
    ),
    "unique_values": accepted_model_candidates.nunique(
        dropna=True
    ).values
})

missing_audit = (
    missing_audit
    .sort_values("missing_percentage", ascending=False)
    .reset_index(drop=True)
)

missing_audit

,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
0,sec_app_mths_since_last_major_derog,float64,6649,1341450,99.506787,113
1,sec_app_revol_util,float64,18308,1329791,98.641940,1112
2,revol_bal_joint,float64,18635,1329464,98.617683,16157
3,sec_app_mort_acc,float64,18636,1329463,98.617609,18
4,sec_app_fico_range_high,float64,18636,1329463,98.617609,60
...,...,...,...,...,...,...
98,application_type,object,1348099,0,0.000000,2
99,fico_range_high,float64,1348099,0,0.000000,48
100,fico_range_low,float64,1348099,0,0.000000,48
101,term,object,1348099,0,0.000000,2


In [76]:
# Cell 5 — Columns containing missing values

columns_with_missing = missing_audit[
    missing_audit["missing_count"] > 0
].copy()

print("Columns containing missing values:", len(columns_with_missing))
print()

columns_with_missing

Columns containing missing values: 89



,column,dtype,non_null_count,missing_count,missing_percentage,unique_values
0,sec_app_mths_since_last_major_derog,float64,6649,1341450,99.506787,113
1,sec_app_revol_util,float64,18308,1329791,98.641940,1112
2,revol_bal_joint,float64,18635,1329464,98.617683,16157
3,sec_app_mort_acc,float64,18636,1329463,98.617609,18
4,sec_app_fico_range_high,float64,18636,1329463,98.617609,60
...,...,...,...,...,...,...
84,earliest_cr_line,object,1348070,29,0.002151,739
85,delinq_2yrs,float64,1348070,29,0.002151,31
86,delinq_amnt,float64,1348070,29,0.002151,2006
87,annual_inc,float64,1348095,4,0.000297,64462


In [77]:
# Cell 6 — High-missingness audit

high_missing_80 = missing_audit[
    missing_audit["missing_percentage"] >= 80
].copy()

high_missing_90 = missing_audit[
    missing_audit["missing_percentage"] >= 90
].copy()

high_missing_95 = missing_audit[
    missing_audit["missing_percentage"] >= 95
].copy()

print("Columns with >= 80% missing:", len(high_missing_80))
print("Columns with >= 90% missing:", len(high_missing_90))
print("Columns with >= 95% missing:", len(high_missing_95))

print("\n>= 80% missing:")
print(high_missing_80)

Columns with >= 80% missing: 18
Columns with >= 90% missing: 17
Columns with >= 95% missing: 16

>= 80% missing:
                                 column    dtype  non_null_count  \
0   sec_app_mths_since_last_major_derog  float64            6649   
1                    sec_app_revol_util  float64           18308   
2                       revol_bal_joint  float64           18635   
3                      sec_app_mort_acc  float64           18636   
4               sec_app_fico_range_high  float64           18636   
5              sec_app_earliest_cr_line   object           18636   
6                sec_app_inq_last_6mths  float64           18636   
7                sec_app_fico_range_low  float64           18636   
8                      sec_app_open_acc  float64           18636   
9                   sec_app_open_act_il  float64           18636   
10                sec_app_num_rev_accts  float64           18636   
11     sec_app_chargeoff_within_12_mths  float64           18636   
12 

In [78]:
# Cell 7 — All-missing columns

all_missing_columns = [
    column
    for column in accepted_model_candidates.columns
    if accepted_model_candidates[column].isna().all()
]

print("All-missing columns:", len(all_missing_columns))

if all_missing_columns:
    for column in all_missing_columns:
        print("-", column)
else:
    print("None")

All-missing columns: 0
None


In [79]:
# Cell 8 — Constant / zero-variance columns

constant_columns = [
    column
    for column in accepted_model_candidates.columns
    if accepted_model_candidates[column].nunique(dropna=True) <= 1
]

print("Constant / single-value columns:", len(constant_columns))

if constant_columns:
    for column in constant_columns:
        print("-", column)
else:
    print("None")

Constant / single-value columns: 0
None


In [80]:
# Cell 9 — Duplicate column-name check

duplicate_column_names = (
    accepted_model_candidates.columns[
        accepted_model_candidates.columns.duplicated()
    ]
    .tolist()
)

print("Duplicate column names:", len(duplicate_column_names))

if duplicate_column_names:
    print(duplicate_column_names)
else:
    print("None")

Duplicate column names: 0
None


In [81]:
# Cell 10 — Duplicate columns by identical values

duplicate_columns_by_content = []

columns = accepted_model_candidates.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        col_a = columns[i]
        col_b = columns[j]

        if accepted_model_candidates[col_a].equals(
            accepted_model_candidates[col_b]
        ):
            duplicate_columns_by_content.append(
                (col_a, col_b)
            )

print(
    "Duplicate columns with identical content:",
    len(duplicate_columns_by_content)
)

if duplicate_columns_by_content:
    for pair in duplicate_columns_by_content:
        print("-", pair)
else:
    print("None")

Duplicate columns with identical content: 0
None


In [82]:
# Cell 11 — Duplicate row audit

duplicate_row_count = accepted_model_candidates.duplicated().sum()

print("Duplicate rows:", duplicate_row_count)

print(
    "Duplicate row percentage:",
    f"{duplicate_row_count / len(accepted_model_candidates) * 100:.4f}%"
)

Duplicate rows: 0
Duplicate row percentage: 0.0000%


In [83]:
import numpy as np

In [84]:
# Cell 12 — Infinite-value audit

numeric_columns = accepted_model_candidates.select_dtypes(
    include=np.number
).columns

infinite_counts = {}

for column in numeric_columns:
    values = pd.to_numeric(
        accepted_model_candidates[column],
        errors="coerce"
    )

    infinite_count = np.isinf(values).sum()

    if infinite_count > 0:
        infinite_counts[column] = int(infinite_count)

infinite_counts = pd.Series(
    infinite_counts,
    dtype="int64"
).sort_values(ascending=False)

print(
    "Numeric columns containing infinite values:",
    len(infinite_counts)
)

if len(infinite_counts) > 0:
    print(infinite_counts)
else:
    print("None")

Numeric columns containing infinite values: 0
None


In [85]:
# Cell 13 — Numeric feature summary

numeric_summary = (
    accepted_model_candidates
    .select_dtypes(include=np.number)
    .describe()
    .T
)

numeric_summary

,count,mean,std,min,25%,50%,75%,max
loan_amnt,1348099.0,14408.998913,8716.137925,500.00,7975.00,12000.00,20000.00,40000.00
int_rate,1348099.0,13.241562,4.765685,5.31,9.75,12.74,15.99,30.99
installment,1348099.0,437.777843,261.497190,4.93,248.28,375.04,580.22,1719.83
annual_inc,1348095.0,76237.743295,69922.741975,0.00,45750.00,65000.00,90000.00,10999200.00
dti,1347725.0,18.274253,11.155495,-1.00,11.79,17.61,24.05,999.00
...,...,...,...,...,...,...,...,...
sec_app_open_act_il,18636.0,2.945160,3.194239,0.00,1.00,2.00,4.00,38.00
sec_app_num_rev_accts,18636.0,12.798508,8.338989,0.00,7.00,11.00,17.00,92.00
sec_app_chargeoff_within_12_mths,18636.0,0.056825,0.463119,0.00,0.00,0.00,0.00,20.00
sec_app_collections_12_mths_ex_med,18636.0,0.087304,0.429709,0.00,0.00,0.00,0.00,16.00


In [86]:
# Cell 14 — Numeric range audit

numeric_range_audit = pd.DataFrame({
    "column": numeric_columns,
    "min": [
        accepted_model_candidates[column].min()
        for column in numeric_columns
    ],
    "max": [
        accepted_model_candidates[column].max()
        for column in numeric_columns
    ],
    "mean": [
        accepted_model_candidates[column].mean()
        for column in numeric_columns
    ],
    "median": [
        accepted_model_candidates[column].median()
        for column in numeric_columns
    ]
})

numeric_range_audit

,column,min,max,mean,median
0,loan_amnt,500.00,40000.00,14408.998913,12000.00
1,int_rate,5.31,30.99,13.241562,12.74
2,installment,4.93,1719.83,437.777843,375.04
3,annual_inc,0.00,10999200.00,76237.743295,65000.00
4,dti,-1.00,999.00,18.274253,17.61
...,...,...,...,...,...
82,sec_app_open_act_il,0.00,38.00,2.945160,2.00
83,sec_app_num_rev_accts,0.00,92.00,12.798508,11.00
84,sec_app_chargeoff_within_12_mths,0.00,20.00,0.056825,0.00
85,sec_app_collections_12_mths_ex_med,0.00,16.00,0.087304,0.00


In [87]:
# Cell 15 — Categorical / object cardinality audit

categorical_columns = accepted_model_candidates.select_dtypes(
    include=["object", "category"]
).columns

categorical_audit = pd.DataFrame({
    "column": categorical_columns,
    "missing_count": [
        accepted_model_candidates[column].isna().sum()
        for column in categorical_columns
    ],
    "unique_values": [
        accepted_model_candidates[column].nunique(dropna=True)
        for column in categorical_columns
    ]
}).sort_values(
    "unique_values",
    ascending=False
)

print("Categorical columns:", len(categorical_columns))

categorical_audit

Categorical columns: 16


,column,missing_count,unique_values
3,emp_title,85950,379863
7,desc,1222146,124398
9,title,16662,63151
10,zip_code,1,945
12,earliest_cr_line,29,739
15,sec_app_earliest_cr_line,1329463,570
11,addr_state,0,51
2,sub_grade,0,35
8,purpose,0,14
4,emp_length,78550,11


In [88]:
# Cell 16 — High-cardinality categorical audit

high_cardinality_categorical = categorical_audit[
    categorical_audit["unique_values"] > 100
].copy()

print(
    "Categorical columns with >100 unique values:",
    len(high_cardinality_categorical)
)

high_cardinality_categorical

Categorical columns with >100 unique values: 6


,column,missing_count,unique_values
3,emp_title,85950,379863
7,desc,1222146,124398
9,title,16662,63151
10,zip_code,1,945
12,earliest_cr_line,29,739
15,sec_app_earliest_cr_line,1329463,570


In [89]:
# Cell 17 — Low-cardinality categorical audit

low_cardinality_categorical = categorical_audit[
    categorical_audit["unique_values"] <= 20
].copy()

print(
    "Categorical columns with <=20 unique values:",
    len(low_cardinality_categorical)
)

low_cardinality_categorical

Categorical columns with <=20 unique values: 8


,column,missing_count,unique_values
8,purpose,0,14
4,emp_length,78550,11
1,grade,0,7
5,home_ownership,0,6
6,verification_status,0,3
14,verification_status_joint,1322498,3
0,term,0,2
13,application_type,0,2


In [90]:
# Cell 18 — Date-like columns

date_like_columns = []

for column in accepted_model_candidates.columns:
    name = column.lower()

    if any(
        keyword in name
        for keyword in [
            "_d",
            "date",
            "month",
            "issue",
            "cr_line"
        ]
    ):
        date_like_columns.append(column)

print("Potential date-like columns:")

for column in date_like_columns:
    print("-", column)

Potential date-like columns:
- earliest_cr_line
- mths_since_last_delinq
- mths_since_last_major_derog
- acc_now_delinq
- mths_since_recent_bc_dlq
- mths_since_recent_revol_delinq
- num_tl_90g_dpd_24m
- pct_tl_nvr_dlq
- sec_app_earliest_cr_line
- sec_app_mths_since_last_major_derog


In [91]:
# Cell 19 — Inspect object columns

object_columns = accepted_model_candidates.select_dtypes(
    include=["object"]
).columns

print("Object columns:", len(object_columns))
print()

for column in object_columns:
    print("=" * 70)
    print("COLUMN:", column)
    print("Unique:", accepted_model_candidates[column].nunique(dropna=True))
    print(
        accepted_model_candidates[column]
        .dropna()
        .head(10)
        .tolist()
    )

Object columns: 16

COLUMN: term
Unique: 2
[' 36 months', ' 36 months', ' 60 months', ' 60 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months']
COLUMN: grade
Unique: 7
['C', 'C', 'B', 'F', 'C', 'B', 'B', 'A', 'B', 'C']
COLUMN: sub_grade
Unique: 35
['C4', 'C1', 'B4', 'F1', 'C3', 'B2', 'B1', 'A2', 'B5', 'C2']
COLUMN: emp_title
Unique: 379863
['leadman', 'Engineer', 'truck driver', 'Contract Specialist', 'Veterinary Tecnician', 'Vice President of Recruiting Operations', 'road driver', 'SERVICE MANAGER', 'Vendor liaison', 'Logistics Manager']
COLUMN: emp_length
Unique: 11
['10+ years', '10+ years', '10+ years', '3 years', '4 years', '10+ years', '10+ years', '6 years', '10+ years', '3 years']
COLUMN: home_ownership
Unique: 6
['MORTGAGE', 'MORTGAGE', 'MORTGAGE', 'MORTGAGE', 'RENT', 'MORTGAGE', 'MORTGAGE', 'RENT', 'MORTGAGE', 'MORTGAGE']
COLUMN: verification_status
Unique: 3
['Not Verified', 'Not Verified', 'Not Verified', 'Source Verified', 'Source Ve

In [92]:
# Cell 20 — Target integrity reference

print("TARGET INTEGRITY")
print("=" * 60)

print("Target is intentionally NOT present in accepted_model_candidates.")
print("Step 7 removed target before creating the feature-candidate dataset.")

print("\nTarget column present:",
      "target" in accepted_model_candidates.columns)

if "target" not in accepted_model_candidates.columns:
    print("✅ Correct: target is excluded from the feature dataset.")
else:
    print("❌ ERROR: target is present in the feature dataset.")

TARGET INTEGRITY
Target is intentionally NOT present in accepted_model_candidates.
Step 7 removed target before creating the feature-candidate dataset.

Target column present: False
✅ Correct: target is excluded from the feature dataset.


In [93]:
# Cell 21 — Verify target exclusion

target_present = "target" in accepted_model_candidates.columns

if target_present:
    print("❌ ERROR: target is present in accepted_model_candidates.")
else:
    print("✅ Target is correctly excluded from accepted_model_candidates.")

✅ Target is correctly excluded from accepted_model_candidates.


In [94]:
# Cell 22 — Compact data-quality overview

data_quality_summary = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "columns_with_missing",
        "all_missing_columns",
        "constant_columns",
        "duplicate_column_names",
        "duplicate_columns_by_content",
        "duplicate_rows",
        "numeric_columns",
        "categorical_columns",
        "high_cardinality_categorical",
        "infinite_numeric_values"
    ],
    "value": [
        len(accepted_model_candidates),
        len(accepted_model_candidates.columns),
        len(columns_with_missing),
        len(all_missing_columns),
        len(constant_columns),
        len(duplicate_column_names),
        len(duplicate_columns_by_content),
        duplicate_row_count,
        len(numeric_columns),
        len(categorical_columns),
        len(high_cardinality_categorical),
        int(infinite_counts.sum())
    ]
})

data_quality_summary

,metric,value
0,rows,1348099
1,columns,103
2,columns_with_missing,89
3,all_missing_columns,0
4,constant_columns,0
5,duplicate_column_names,0
6,duplicate_columns_by_content,0
7,duplicate_rows,0
8,numeric_columns,87
9,categorical_columns,16


In [95]:
# Cell 23 — Define objectively removable columns

quality_removal_columns = set()

# All-missing columns
quality_removal_columns.update(all_missing_columns)

# Constant columns
quality_removal_columns.update(constant_columns)

# Duplicate columns by identical content
for col_a, col_b in duplicate_columns_by_content:
    quality_removal_columns.add(col_b)

quality_removal_columns = sorted(
    quality_removal_columns
)

print("Objectively removable quality columns:")
print("Count:", len(quality_removal_columns))

for column in quality_removal_columns:
    print("-", column)

Objectively removable quality columns:
Count: 0


In [96]:
# Cell 24 — Review quality-removal candidates

if quality_removal_columns:
    quality_removal_review = missing_audit[
        missing_audit["column"].isin(quality_removal_columns)
    ].copy()

    quality_removal_review
else:
    print("No objectively removable quality columns identified.")

No objectively removable quality columns identified.


In [97]:
# Cell 25 — Apply objective data-quality removals

accepted_quality_clean = accepted_model_candidates.drop(
    columns=quality_removal_columns,
    errors="ignore"
).copy()

print("Before quality cleanup:")
print(accepted_model_candidates.shape)

print("\nAfter quality cleanup:")
print(accepted_quality_clean.shape)

Before quality cleanup:
(1348099, 103)

After quality cleanup:
(1348099, 103)


In [98]:
# Cell 26 — Verify quality cleanup

remaining_removed_columns = [
    column
    for column in quality_removal_columns
    if column in accepted_quality_clean.columns
]

if len(remaining_removed_columns) == 0:
    print("✅ All objective quality-removal columns were removed.")
else:
    print("❌ Some removal columns remain:")
    print(remaining_removed_columns)

✅ All objective quality-removal columns were removed.


In [99]:
# Cell 27 — Re-audit missingness

post_cleanup_missing = (
    accepted_quality_clean
    .isna()
    .sum()
    .sort_values(ascending=False)
)

post_cleanup_missing = post_cleanup_missing[
    post_cleanup_missing > 0
]

print(
    "Columns still containing missing values:",
    len(post_cleanup_missing)
)

post_cleanup_missing

Columns still containing missing values: 89


sec_app_mths_since_last_major_derog    1341450
sec_app_revol_util                     1329791
revol_bal_joint                        1329464
sec_app_mort_acc                       1329463
sec_app_fico_range_high                1329463
                                        ...   
earliest_cr_line                            29
delinq_2yrs                                 29
delinq_amnt                                 29
annual_inc                                   4
zip_code                                     1
Length: 89, dtype: int64

In [100]:
# Cell 28 — Final duplicate check

duplicate_rows_after_cleanup = (
    accepted_quality_clean
    .duplicated()
    .sum()
)

duplicate_columns_after_cleanup = (
    accepted_quality_clean
    .columns[
        accepted_quality_clean.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate rows:",
    duplicate_rows_after_cleanup
)

print(
    "Duplicate column names:",
    len(duplicate_columns_after_cleanup)
)

if duplicate_columns_after_cleanup:
    print(duplicate_columns_after_cleanup)

Duplicate rows: 0
Duplicate column names: 0


In [101]:
# Cell 29 — Final Step 8 structural audit

print("STEP 8 — FINAL STRUCTURAL AUDIT")
print("=" * 60)

print("Rows:", accepted_quality_clean.shape[0])
print("Columns:", accepted_quality_clean.shape[1])

target_present = "target" in accepted_quality_clean.columns

print(
    "Target present:",
    target_present
)

if not target_present:
    print("✅ Target correctly excluded from feature dataset.")
else:
    print("❌ ERROR: Target is present in feature dataset.")

print(
    "Duplicate rows:",
    accepted_quality_clean.duplicated().sum()
)

print(
    "Duplicate column names:",
    accepted_quality_clean.columns.duplicated().sum()
)

# Safe infinite-value check
numeric_clean_columns = accepted_quality_clean.select_dtypes(
    include=np.number
).columns

infinite_value_count = 0

for column in numeric_clean_columns:
    infinite_value_count += np.isinf(
        pd.to_numeric(
            accepted_quality_clean[column],
            errors="coerce"
        )
    ).sum()

print(
    "Infinite numeric values:",
    int(infinite_value_count)
)

STEP 8 — FINAL STRUCTURAL AUDIT
Rows: 1348099
Columns: 103
Target present: False
✅ Target correctly excluded from feature dataset.
Duplicate rows: 0
Duplicate column names: 0
Infinite numeric values: 0


In [102]:
# Cell 30 — Confirm no Step 7 exclusions returned

step7_exclusions = set(
    step7_remove_columns
)

returned_step7_columns = [
    column
    for column in step7_exclusions
    if column in accepted_quality_clean.columns
]

print("Step 7 exclusion columns returned:", len(returned_step7_columns))

if returned_step7_columns:
    print("❌ Returned columns:")
    for column in returned_step7_columns:
        print("-", column)
else:
    print("✅ No Step 7 exclusion columns returned.")

Step 7 exclusion columns returned: 0
✅ No Step 7 exclusion columns returned.


In [103]:
# Cell 31 — Final Step 8 verification

step8_checks = {
    "Target correctly excluded": (
        "target" not in accepted_quality_clean.columns
    ),

    "No duplicate column names": (
        accepted_quality_clean.columns.duplicated().sum() == 0
    ),

    "No duplicate rows": (
        accepted_quality_clean.duplicated().sum() == 0
    ),

    "No infinite numeric values": (
        infinite_value_count == 0
    ),

    "Step 7 exclusions absent": (
        len(returned_step7_columns) == 0
    )
}

print("STEP 8 INTEGRITY CHECK")
print("=" * 60)

for check, result in step8_checks.items():
    print(
        "✅" if result else "❌",
        check
    )

if all(step8_checks.values()):
    print("\n🟢 STEP 8 STRUCTURAL CHECK PASSED")
else:
    print("\n🔴 STEP 8 NEEDS REVIEW")

STEP 8 INTEGRITY CHECK
✅ Target correctly excluded
✅ No duplicate column names
✅ No duplicate rows
✅ No infinite numeric values
✅ Step 7 exclusions absent

🟢 STEP 8 STRUCTURAL CHECK PASSED


In [104]:
# Cell 32 — Save Step 8 data-quality audit

step8_quality_audit = missing_audit.copy()

step8_quality_audit.to_csv(
    "reports/preprocessing_v1_audits/accepted_step8_quality_audit.csv",
    index=False
)

print("Saved:")
print("reports/preprocessing_v1_audits/accepted_step8_quality_audit.csv")

Saved:
accepted_step8_quality_audit.csv


In [105]:
# Cell 33 — Save Step 8 cleaned candidate dataset

accepted_quality_clean.to_csv(
    "data/interim/accepted_quality_clean.csv",
    index=False
)

print("Saved:")
print("data/interim/accepted_quality_clean.csv")

print("\nFinal shape:")
print(accepted_quality_clean.shape)

Saved:
accepted_quality_clean.csv

Final shape:
(1348099, 103)


**STEP 9 — FEATURE SEMANTICS & REPRESENTATION REVIEW**

In [106]:
# STEP 9
# Cell 1 — Load Step 8 cleaned candidate dataset

import pandas as pd
import numpy as np

accepted_quality_clean = pd.read_csv(
    "data/interim/accepted_quality_clean.csv",
    low_memory=False
)

print("STEP 9 — FEATURE SEMANTICS & REPRESENTATION REVIEW")
print("=" * 60)

print("Rows   :", accepted_quality_clean.shape[0])
print("Columns:", accepted_quality_clean.shape[1])
print("Target present:", "target" in accepted_quality_clean.columns)

STEP 9 — FEATURE SEMANTICS & REPRESENTATION REVIEW
Rows   : 1348099
Columns: 103
Target present: False


In [107]:
# Cell 2 — Feature inventory

feature_inventory = pd.DataFrame({
    "column": accepted_quality_clean.columns,
    "dtype": accepted_quality_clean.dtypes.astype(str).values,
    "missing_count": accepted_quality_clean.isna().sum().values,
    "missing_percentage": (
        accepted_quality_clean.isna().mean().values * 100
    ),
    "unique_values": accepted_quality_clean.nunique(
        dropna=True
    ).values
})

feature_inventory = feature_inventory.sort_values(
    "column"
).reset_index(drop=True)

feature_inventory

,column,dtype,missing_count,missing_percentage,unique_values
0,acc_now_delinq,float64,29,0.002151,8
1,acc_open_past_24mths,float64,50030,3.711152,55
2,addr_state,object,0,0.000000,51
3,all_util,float64,810516,60.122884,171
4,annual_inc,float64,4,0.000297,64462
...,...,...,...,...,...
98,total_il_high_credit_limit,float64,70276,5.212970,162550
99,total_rev_hi_lim,float64,70276,5.212970,26799
100,verification_status,object,0,0.000000,3
101,verification_status_joint,object,1322498,98.100955,3


In [108]:
# Cell 3 — Raw data-type groups

numeric_features = accepted_quality_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = accepted_quality_clean.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical/object features:", len(categorical_features))

print("\nNumeric:")
for column in numeric_features:
    print("-", column)

print("\nCategorical/Object:")
for column in categorical_features:
    print("-", column)

Numeric features: 87
Categorical/object features: 16

Numeric:
- loan_amnt
- int_rate
- installment
- annual_inc
- dti
- delinq_2yrs
- fico_range_low
- fico_range_high
- inq_last_6mths
- mths_since_last_delinq
- mths_since_last_record
- open_acc
- pub_rec
- revol_bal
- revol_util
- total_acc
- collections_12_mths_ex_med
- mths_since_last_major_derog
- annual_inc_joint
- dti_joint
- acc_now_delinq
- tot_coll_amt
- tot_cur_bal
- open_acc_6m
- open_act_il
- open_il_12m
- open_il_24m
- mths_since_rcnt_il
- total_bal_il
- il_util
- open_rv_12m
- open_rv_24m
- max_bal_bc
- all_util
- total_rev_hi_lim
- inq_fi
- total_cu_tl
- inq_last_12m
- acc_open_past_24mths
- avg_cur_bal
- bc_open_to_buy
- bc_util
- chargeoff_within_12_mths
- delinq_amnt
- mo_sin_old_il_acct
- mo_sin_old_rev_tl_op
- mo_sin_rcnt_rev_tl_op
- mo_sin_rcnt_tl
- mort_acc
- mths_since_recent_bc
- mths_since_recent_bc_dlq
- mths_since_recent_inq
- mths_since_recent_revol_delinq
- num_accts_ever_120_pd
- num_actv_bc_tl
- num_actv_

In [109]:
# Cell 4 — Date feature candidates

date_keywords = [
    "date",
    "_d",
    "month",
    "issue",
    "cr_line"
]

date_candidates = []

for column in accepted_quality_clean.columns:
    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in date_keywords
    ):
        date_candidates.append(column)

print("Potential date features:")
print("=" * 60)

for column in date_candidates:
    print("-", column)

Potential date features:
- earliest_cr_line
- mths_since_last_delinq
- mths_since_last_major_derog
- acc_now_delinq
- mths_since_recent_bc_dlq
- mths_since_recent_revol_delinq
- num_tl_90g_dpd_24m
- pct_tl_nvr_dlq
- sec_app_earliest_cr_line
- sec_app_mths_since_last_major_derog


In [110]:
# Cell 5 — Inspect date candidates

for column in date_candidates:
    print("=" * 70)
    print("COLUMN:", column)
    print("dtype :", accepted_quality_clean[column].dtype)
    print(
        accepted_quality_clean[column]
        .dropna()
        .head(10)
        .tolist()
    )

COLUMN: earliest_cr_line
dtype : object
['Aug-2003', 'Dec-1999', 'Aug-2000', 'Jun-1998', 'Oct-1987', 'Jun-1990', 'Feb-1999', 'Apr-2002', 'Nov-1994', 'Jun-1996']
COLUMN: mths_since_last_delinq
dtype : float64
[30.0, 6.0, 12.0, 49.0, 3.0, 75.0, 42.0, 33.0, 15.0, 18.0]
COLUMN: mths_since_last_major_derog
dtype : float64
[30.0, 3.0, 75.0, 33.0, 55.0, 69.0, 38.0, 70.0, 18.0, 56.0]
COLUMN: acc_now_delinq
dtype : float64
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
COLUMN: mths_since_recent_bc_dlq
dtype : float64
[69.0, 12.0, 48.0, 76.0, 15.0, 33.0, 69.0, 45.0, 46.0, 39.0]
COLUMN: mths_since_recent_revol_delinq
dtype : float64
[69.0, 6.0, 12.0, 42.0, 76.0, 15.0, 18.0, 47.0, 33.0, 69.0]
COLUMN: num_tl_90g_dpd_24m
dtype : float64
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
COLUMN: pct_tl_nvr_dlq
dtype : float64
[76.9, 97.4, 100.0, 96.6, 100.0, 96.3, 93.3, 95.7, 94.4, 100.0]
COLUMN: sec_app_earliest_cr_line
dtype : object
['Feb-2007', 'Sep-2008', 'Aug-2006', 'Feb-2009', 'May-2004', 

In [111]:
# Cell 6 — Text-like feature candidates

text_keywords = [
    "desc",
    "description",
    "title",
    "emp_title",
    "purpose",
    "url"
]

text_candidates = []

for column in accepted_quality_clean.columns:
    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in text_keywords
    ):
        text_candidates.append(column)

print("Potential text-like features:")
print("=" * 60)

for column in text_candidates:
    print("-", column)

Potential text-like features:
- emp_title
- desc
- purpose
- title


In [112]:
# Cell 7 — Inspect text-like fields

text_audit = pd.DataFrame({
    "column": text_candidates,
    "dtype": [
        str(accepted_quality_clean[column].dtype)
        for column in text_candidates
    ],
    "unique_values": [
        accepted_quality_clean[column].nunique(dropna=True)
        for column in text_candidates
    ],
    "missing_percentage": [
        accepted_quality_clean[column].isna().mean() * 100
        for column in text_candidates
    ]
})

text_audit.sort_values(
    "unique_values",
    ascending=False
)

,column,dtype,unique_values,missing_percentage
0,emp_title,object,379863,6.375645
1,desc,object,124398,90.656992
3,title,object,63151,1.235963
2,purpose,object,14,0.000000


In [113]:
# Cell 8 — High-cardinality audit

high_cardinality_features = feature_inventory[
    feature_inventory["unique_values"] > 100
].copy()

print(
    "Features with more than 100 unique values:",
    len(high_cardinality_features)
)

high_cardinality_features

Features with more than 100 unique values: 49


,column,dtype,missing_count,missing_percentage,unique_values
3,all_util,float64,810516,60.122884,171
4,annual_inc,float64,4,0.000297,64462
5,annual_inc_joint,float64,1322293,98.085749,5161
7,avg_cur_bal,float64,70298,5.214602,76864
8,bc_open_to_buy,float64,63894,4.739563,74924
9,bc_util,float64,64663,4.796606,1444
13,delinq_amnt,float64,29,0.002151,2006
14,desc,object,1222146,90.656992,124398
15,dti,float64,374,0.027743,7067
16,dti_joint,float64,1322296,98.085971,3658


In [114]:
# Cell 9 — Low-cardinality categorical audit

categorical_inventory = feature_inventory[
    feature_inventory["column"].isin(categorical_features)
].copy()

low_cardinality_categorical = categorical_inventory[
    categorical_inventory["unique_values"] <= 20
].copy()

print(
    "Categorical features with <=20 unique values:",
    len(low_cardinality_categorical)
)

low_cardinality_categorical

Categorical features with <=20 unique values: 8


,column,dtype,missing_count,missing_percentage,unique_values
6,application_type,object,0,0.000000,2
18,emp_length,object,78550,5.826723,11
22,grade,object,0,0.000000,7
23,home_ownership,object,0,0.000000,6
70,purpose,object,0,0.000000,14
88,term,object,0,0.000000,2
100,verification_status,object,0,0.000000,3
101,verification_status_joint,object,1322498,98.100955,3


In [115]:
# Cell 10 — Inspect categorical values

for column in categorical_features:
    print("=" * 70)
    print("COLUMN:", column)

    print(
        accepted_quality_clean[column]
        .value_counts(dropna=False)
        .head(20)
    )

COLUMN: term
term
36 months    1023206
60 months     324893
Name: count, dtype: int64
COLUMN: grade
grade
B    393102
C    382323
A    235193
D    201657
E     94192
F     32306
G      9326
Name: count, dtype: int64
COLUMN: sub_grade
sub_grade
C1    85618
B4    83276
B5    82639
B3    81901
C2    79358
C3    75129
C4    74554
B2    74080
B1    71206
C5    67664
A5    64056
A4    52255
D1    51445
D2    44984
A1    43682
D3    39466
A3    38010
A2    37190
D4    35722
D5    30040
Name: count, dtype: int64
COLUMN: emp_title
emp_title
NaN                 85950
Teacher             21268
Manager             19472
Owner               10302
Registered Nurse     8774
RN                   8522
Supervisor           8290
Driver               7559
Sales                7488
Project Manager      6381
Office Manager       5526
General Manager      5191
Director             5032
owner                4585
manager              4511
Engineer             4294
President            4162
teacher             

In [116]:
# Cell 11 — LendingClub-derived / underwriting-related fields

lc_derived_candidates = [
    column
    for column in [
        "int_rate",
        "grade",
        "sub_grade",
        "installment"
    ]
    if column in accepted_quality_clean.columns
]

print("LendingClub-derived / underwriting-related candidates:")
print("=" * 60)

for column in lc_derived_candidates:
    print("-", column)

LendingClub-derived / underwriting-related candidates:
- int_rate
- grade
- sub_grade
- installment


In [117]:
# Cell 12 — Inspect LendingClub-derived fields

for column in lc_derived_candidates:
    print("=" * 70)
    print("COLUMN:", column)
    print("dtype:", accepted_quality_clean[column].dtype)
    print("unique:", accepted_quality_clean[column].nunique(dropna=True))
    print()
    print(
        accepted_quality_clean[column]
        .value_counts(dropna=False)
        .head(20)
    )

COLUMN: int_rate
dtype: float64
unique: 672

int_rate
10.99    37666
11.99    33112
13.99    29360
5.32     28820
9.17     26048
12.99    25590
7.89     25054
15.61    22752
16.99    22640
9.99     19470
11.49    18388
12.69    16701
14.99    16613
12.29    16462
8.18     16284
17.57    15723
13.33    15177
11.53    14893
16.29    14831
15.31    14173
Name: count, dtype: int64
COLUMN: grade
dtype: object
unique: 7

grade
B    393102
C    382323
A    235193
D    201657
E     94192
F     32306
G      9326
Name: count, dtype: int64
COLUMN: sub_grade
dtype: object
unique: 35

sub_grade
C1    85618
B4    83276
B5    82639
B3    81901
C2    79358
C3    75129
C4    74554
B2    74080
B1    71206
C5    67664
A5    64056
A4    52255
D1    51445
D2    44984
A1    43682
D3    39466
A3    38010
A2    37190
D4    35722
D5    30040
Name: count, dtype: int64
COLUMN: installment
dtype: float64
unique: 83531

installment
327.34    3064
301.15    2690
332.10    2670
318.79    2295
312.86    2067
491.01  

In [118]:
# Cell 13 — Inspect grade / sub_grade / int_rate relationship

rate_grade_columns = [
    column
    for column in [
        "grade",
        "sub_grade",
        "int_rate"
    ]
    if column in accepted_quality_clean.columns
]

if len(rate_grade_columns) > 0:
    print(
        accepted_quality_clean[rate_grade_columns]
        .head(20)
    )
else:
    print("No grade/rate fields found.")

   grade sub_grade  int_rate
0      C        C4     13.99
1      C        C1     11.99
2      B        B4     10.78
3      F        F1     22.45
4      C        C3     13.44
5      B        B2      9.17
6      B        B1      8.49
7      A        A2      6.49
8      B        B5     11.48
9      C        C2     12.88
10     E        E2     19.48
11     A        A2      6.49
12     A        A4      7.49
13     A        A4      7.49
14     C        C1     11.99
15     E        E3     19.89
16     C        C2     12.88
17     C        C5     14.85
18     C        C3     13.44
19     C        C3     13.44


In [119]:
# Cell 14 — Numeric feature audit

numeric_feature_audit = pd.DataFrame({
    "column": numeric_features,
    "dtype": [
        str(accepted_quality_clean[column].dtype)
        for column in numeric_features
    ],
    "missing_percentage": [
        accepted_quality_clean[column].isna().mean() * 100
        for column in numeric_features
    ],
    "unique_values": [
        accepted_quality_clean[column].nunique(dropna=True)
        for column in numeric_features
    ],
    "min": [
        accepted_quality_clean[column].min()
        for column in numeric_features
    ],
    "median": [
        accepted_quality_clean[column].median()
        for column in numeric_features
    ],
    "max": [
        accepted_quality_clean[column].max()
        for column in numeric_features
    ]
})

numeric_feature_audit

,column,dtype,missing_percentage,unique_values,min,median,max
0,loan_amnt,float64,0.000000,1560,500.00,12000.00,40000.00
1,int_rate,float64,0.000000,672,5.31,12.74,30.99
2,installment,float64,0.000000,83531,4.93,375.04,1719.83
3,annual_inc,float64,0.000297,64462,0.00,65000.00,10999200.00
4,dti,float64,0.027743,7067,-1.00,17.61,999.00
...,...,...,...,...,...,...,...
82,sec_app_open_act_il,float64,98.617609,35,0.00,2.00,38.00
83,sec_app_num_rev_accts,float64,98.617609,68,0.00,11.00,92.00
84,sec_app_chargeoff_within_12_mths,float64,98.617609,15,0.00,0.00,20.00
85,sec_app_collections_12_mths_ex_med,float64,98.617609,12,0.00,0.00,16.00


In [120]:
# Cell 15 — Percentage/rate feature candidates

percentage_keywords = [
    "rate",
    "percent",
    "util",
    "dti"
]

percentage_candidates = []

for column in accepted_quality_clean.columns:
    column_lower = column.lower()

    if any(
        keyword in column_lower
        for keyword in percentage_keywords
    ):
        percentage_candidates.append(column)

print("Potential percentage/rate features:")
print("=" * 60)

for column in percentage_candidates:
    print("-", column)

Potential percentage/rate features:
- int_rate
- dti
- revol_util
- dti_joint
- il_util
- all_util
- bc_util
- percent_bc_gt_75
- sec_app_revol_util


In [121]:
# Cell 16 — Inspect percentage/rate ranges

percentage_range_audit = []

for column in percentage_candidates:

    if pd.api.types.is_numeric_dtype(
        accepted_quality_clean[column]
    ):
        percentage_range_audit.append({
            "column": column,
            "min": accepted_quality_clean[column].min(),
            "max": accepted_quality_clean[column].max(),
            "median": accepted_quality_clean[column].median()
        })

percentage_range_audit = pd.DataFrame(
    percentage_range_audit
)

percentage_range_audit

,column,min,max,median
0,int_rate,5.31,30.99,12.74
1,dti,-1.00,999.00,17.61
2,revol_util,0.00,892.30,52.20
3,dti_joint,0.00,69.49,18.46
4,il_util,0.00,558.00,75.00
5,all_util,0.00,198.00,59.00
6,bc_util,0.00,339.60,63.20
7,percent_bc_gt_75,0.00,100.00,42.90
8,sec_app_revol_util,0.00,212.60,59.30


In [122]:
# Cell 17 — Feature representation decision table

representation_plan = pd.DataFrame({
    "column": accepted_quality_clean.columns,
    "raw_dtype": [
        str(accepted_quality_clean[column].dtype)
        for column in accepted_quality_clean.columns
    ],
    "representation_category": "REVIEW",
    "planned_transformation": "REVIEW",
    "reason": "REVIEW"
})

representation_plan.head()

,column,raw_dtype,representation_category,planned_transformation,reason
0,loan_amnt,float64,REVIEW,REVIEW,REVIEW
1,term,object,REVIEW,REVIEW,REVIEW
2,int_rate,float64,REVIEW,REVIEW,REVIEW
3,installment,float64,REVIEW,REVIEW,REVIEW
4,grade,object,REVIEW,REVIEW,REVIEW


In [123]:
# Cell 18 — Initial representation classification

for index, row in representation_plan.iterrows():

    column = row["column"]
    column_lower = column.lower()

    if column in date_candidates:
        representation_plan.loc[
            index,
            "representation_category"
        ] = "DATE"

        representation_plan.loc[
            index,
            "planned_transformation"
        ] = "DATE_FEATURE_ENGINEERING"

        representation_plan.loc[
            index,
            "reason"
        ] = "Application-time date requiring temporal representation"

    elif column in text_candidates:
        representation_plan.loc[
            index,
            "representation_category"
        ] = "TEXT_OR_HIGH_CARDINALITY"

        representation_plan.loc[
            index,
            "planned_transformation"
        ] = "REVIEW"

        representation_plan.loc[
            index,
            "reason"
        ] = "Text/high-cardinality field requires explicit treatment"

    elif column in lc_derived_candidates:
        representation_plan.loc[
            index,
            "representation_category"
        ] = "LC_DERIVED_REVIEW"

        representation_plan.loc[
            index,
            "planned_transformation"
        ] = "METHODOLOGICAL_DECISION"

        representation_plan.loc[
            index,
            "reason"
        ] = "Potential underwriting-derived information"

    elif column in categorical_features:
        representation_plan.loc[
            index,
            "representation_category"
        ] = "CATEGORICAL"

        representation_plan.loc[
            index,
            "planned_transformation"
        ] = "ENCODING"

        representation_plan.loc[
            index,
            "reason"
        ] = "Categorical application-time feature"

    elif column in numeric_features:
        representation_plan.loc[
            index,
            "representation_category"
        ] = "NUMERIC"

        representation_plan.loc[
            index,
            "planned_transformation"
        ] = "NUMERIC_PREPROCESSING"

        representation_plan.loc[
            index,
            "reason"
        ] = "Numeric application-time feature"

representation_plan

,column,raw_dtype,representation_category,planned_transformation,reason
0,loan_amnt,float64,NUMERIC,NUMERIC_PREPROCESSING,Numeric application-time feature
1,term,object,CATEGORICAL,ENCODING,Categorical application-time feature
2,int_rate,float64,LC_DERIVED_REVIEW,METHODOLOGICAL_DECISION,Potential underwriting-derived information
3,installment,float64,LC_DERIVED_REVIEW,METHODOLOGICAL_DECISION,Potential underwriting-derived information
4,grade,object,LC_DERIVED_REVIEW,METHODOLOGICAL_DECISION,Potential underwriting-derived information
...,...,...,...,...,...
98,sec_app_open_act_il,float64,NUMERIC,NUMERIC_PREPROCESSING,Numeric application-time feature
99,sec_app_num_rev_accts,float64,NUMERIC,NUMERIC_PREPROCESSING,Numeric application-time feature
100,sec_app_chargeoff_within_12_mths,float64,NUMERIC,NUMERIC_PREPROCESSING,Numeric application-time feature
101,sec_app_collections_12_mths_ex_med,float64,NUMERIC,NUMERIC_PREPROCESSING,Numeric application-time feature


In [124]:
# Cell 19 — Features requiring explicit review

review_features = representation_plan[
    representation_plan["planned_transformation"] == "REVIEW"
].copy()

print(
    "Features requiring explicit representation review:",
    len(review_features)
)

review_features

Features requiring explicit representation review: 4


,column,raw_dtype,representation_category,planned_transformation,reason
6,emp_title,object,TEXT_OR_HIGH_CARDINALITY,REVIEW,Text/high-cardinality field requires explicit ...
11,desc,object,TEXT_OR_HIGH_CARDINALITY,REVIEW,Text/high-cardinality field requires explicit ...
12,purpose,object,TEXT_OR_HIGH_CARDINALITY,REVIEW,Text/high-cardinality field requires explicit ...
13,title,object,TEXT_OR_HIGH_CARDINALITY,REVIEW,Text/high-cardinality field requires explicit ...


In [125]:
# Cell 20 — Step 9 summary

print("STEP 9 SUMMARY")
print("=" * 60)

print("Total features:", len(accepted_quality_clean.columns))
print("Numeric:", len(numeric_features))
print("Categorical/Object:", len(categorical_features))
print("Date candidates:", len(date_candidates))
print("Text-like candidates:", len(text_candidates))
print("LC-derived review candidates:", len(lc_derived_candidates))
print(
    "High-cardinality features:",
    len(high_cardinality_features)
)

print("\nRepresentation categories:")
print(
    representation_plan[
        "representation_category"
    ].value_counts()
)

STEP 9 SUMMARY
Total features: 103
Numeric: 87
Categorical/Object: 16
Date candidates: 10
Text-like candidates: 4
LC-derived review candidates: 4
High-cardinality features: 49

Representation categories:
representation_category
NUMERIC                     77
DATE                        10
CATEGORICAL                  8
LC_DERIVED_REVIEW            4
TEXT_OR_HIGH_CARDINALITY     4
Name: count, dtype: int64


In [126]:
# Cell 21 — Save Step 9 representation plan

representation_plan.to_csv(
    "reports/preprocessing_v1_audits/accepted_step9_representation_plan.csv",
    index=False
)

print(
    "Saved: accepted_step9_representation_plan.csv"
)

Saved: accepted_step9_representation_plan.csv


In [127]:
# Cell 22 — Final date classification

# Only fields that represent actual calendar dates
actual_date_features = [
    column
    for column in [
        "earliest_cr_line",
        "sec_app_earliest_cr_line"
    ]
    if column in accepted_quality_clean.columns
]

# Numeric "months since" / count / percentage fields are NOT dates
numeric_non_date_candidates = [
    column
    for column in date_candidates
    if column not in actual_date_features
]

print("Actual date features:")
for column in actual_date_features:
    print("✅", column)

print("\nPreviously detected but confirmed NOT to be dates:")
for column in numeric_non_date_candidates:
    print("✅", column)

Actual date features:
✅ earliest_cr_line
✅ sec_app_earliest_cr_line

Previously detected but confirmed NOT to be dates:
✅ mths_since_last_delinq
✅ mths_since_last_major_derog
✅ acc_now_delinq
✅ mths_since_recent_bc_dlq
✅ mths_since_recent_revol_delinq
✅ num_tl_90g_dpd_24m
✅ pct_tl_nvr_dlq
✅ sec_app_mths_since_last_major_derog


In [128]:
# Cell 23 — Final text representation decisions

text_representation_decisions = {
    "emp_title": {
        "category": "HIGH_CARDINALITY_CATEGORICAL",
        "transformation": "REVIEW_ENCODER",
        "reason": (
            "Applicant employment title has high cardinality and "
            "requires controlled representation rather than raw one-hot encoding."
        )
    },

    "desc": {
        "category": "FREE_TEXT",
        "transformation": "DROP_INITIAL_VERSION",
        "reason": (
            "Free-form loan description requires a separate NLP pipeline. "
            "It will not be included in the initial tabular model."
        )
    },

    "purpose": {
        "category": "LOW_CARDINALITY_CATEGORICAL",
        "transformation": "ENCODING",
        "reason": (
            "Loan purpose is a structured categorical application-time field."
        )
    },

    "title": {
        "category": "HIGH_CARDINALITY_TEXT",
        "transformation": "DROP_INITIAL_VERSION",
        "reason": (
            "Free-form loan title has high cardinality and requires "
            "specialized text processing. It will not enter the initial "
            "tabular model."
        )
    }
}

for column, decision in text_representation_decisions.items():

    if column in representation_plan["column"].values:

        representation_plan.loc[
            representation_plan["column"] == column,
            "representation_category"
        ] = decision["category"]

        representation_plan.loc[
            representation_plan["column"] == column,
            "planned_transformation"
        ] = decision["transformation"]

        representation_plan.loc[
            representation_plan["column"] == column,
            "reason"
        ] = decision["reason"]

print("Text representation decisions applied.")

Text representation decisions applied.


In [129]:
# Cell 24 — Final date representation decisions

for column in actual_date_features:

    representation_plan.loc[
        representation_plan["column"] == column,
        "representation_category"
    ] = "DATE"

    representation_plan.loc[
        representation_plan["column"] == column,
        "planned_transformation"
    ] = "DATE_FEATURE_ENGINEERING"

    representation_plan.loc[
        representation_plan["column"] == column,
        "reason"
    ] = (
        "Historical/application-time credit date. "
        "Convert to meaningful temporal features during transformation."
    )

# Correct the falsely detected numeric date candidates
for column in numeric_non_date_candidates:

    representation_plan.loc[
        representation_plan["column"] == column,
        "representation_category"
    ] = "NUMERIC"

    representation_plan.loc[
        representation_plan["column"] == column,
        "planned_transformation"
    ] = "NUMERIC_PREPROCESSING"

    representation_plan.loc[
        representation_plan["column"] == column,
        "reason"
    ] = (
        "Numeric count/months-since/percentage feature. "
        "It is not a calendar date."
    )

print("Date classification corrected.")

Date classification corrected.


In [130]:
# Cell 25 — LC-derived feature methodological decision

lc_decision = {
    "int_rate": (
        "RETAIN_FOR_REVIEW",
        "Interest rate is available in the LendingClub record but is "
        "an underwriting-derived variable. Its use requires explicit "
        "methodological justification."
    ),

    "grade": (
        "RETAIN_FOR_REVIEW",
        "LendingClub risk grade is underwriting-derived and requires "
        "explicit methodological justification."
    ),

    "sub_grade": (
        "RETAIN_FOR_REVIEW",
        "LendingClub sub-grade is underwriting-derived and requires "
        "explicit methodological justification."
    ),

    "installment": (
        "RETAIN_FOR_REVIEW",
        "Installment is associated with loan terms and requires "
        "explicit review of its availability at the prediction point."
    )
}

for column, (decision, reason) in lc_decision.items():

    if column in representation_plan["column"].values:

        representation_plan.loc[
            representation_plan["column"] == column,
            "representation_category"
        ] = "LC_DERIVED_REVIEW"

        representation_plan.loc[
            representation_plan["column"] == column,
            "planned_transformation"
        ] = decision

        representation_plan.loc[
            representation_plan["column"] == column,
            "reason"
        ] = reason

print("LC-derived methodological decisions documented.")

LC-derived methodological decisions documented.


In [131]:
# Cell 26 — Final unresolved-feature check

review_features = representation_plan[
    representation_plan["planned_transformation"].isin([
        "REVIEW",
        "METHODOLOGICAL_DECISION"
    ])
].copy()

print("UNRESOLVED REPRESENTATION DECISIONS")
print("=" * 60)

if len(review_features) == 0:
    print("None ✅")
else:
    print(
        "Features still requiring explicit decision:",
        len(review_features)
    )
    print(review_features.to_string(index=False))

UNRESOLVED REPRESENTATION DECISIONS
None ✅


In [132]:
# Cell 27 — Final Step 9 representation audit

print("STEP 9 — FINAL REPRESENTATION AUDIT")
print("=" * 60)

print("Total features:", len(representation_plan))

print("\nRepresentation categories:")
print(
    representation_plan[
        "representation_category"
    ].value_counts()
)

print("\nTransformation categories:")
print(
    representation_plan[
        "planned_transformation"
    ].value_counts()
)

STEP 9 — FINAL REPRESENTATION AUDIT
Total features: 103

Representation categories:
representation_category
NUMERIC                         85
CATEGORICAL                      8
LC_DERIVED_REVIEW                4
DATE                             2
HIGH_CARDINALITY_CATEGORICAL     1
FREE_TEXT                        1
LOW_CARDINALITY_CATEGORICAL      1
HIGH_CARDINALITY_TEXT            1
Name: count, dtype: int64

Transformation categories:
planned_transformation
NUMERIC_PREPROCESSING       85
ENCODING                     9
RETAIN_FOR_REVIEW            4
DROP_INITIAL_VERSION         2
DATE_FEATURE_ENGINEERING     2
REVIEW_ENCODER               1
Name: count, dtype: int64


In [133]:
# Cell 28 — Date classification verification

print("DATE CLASSIFICATION CHECK")
print("=" * 60)

for column in actual_date_features:

    row = representation_plan[
        representation_plan["column"] == column
    ].iloc[0]

    print(
        f"✅ {column} → "
        f"{row['representation_category']} → "
        f"{row['planned_transformation']}"
    )

print("\nNumeric fields previously mistaken for dates:")

for column in numeric_non_date_candidates:

    row = representation_plan[
        representation_plan["column"] == column
    ].iloc[0]

    print(
        f"✅ {column} → "
        f"{row['representation_category']} → "
        f"{row['planned_transformation']}"
    )

DATE CLASSIFICATION CHECK
✅ earliest_cr_line → DATE → DATE_FEATURE_ENGINEERING
✅ sec_app_earliest_cr_line → DATE → DATE_FEATURE_ENGINEERING

Numeric fields previously mistaken for dates:
✅ mths_since_last_delinq → NUMERIC → NUMERIC_PREPROCESSING
✅ mths_since_last_major_derog → NUMERIC → NUMERIC_PREPROCESSING
✅ acc_now_delinq → NUMERIC → NUMERIC_PREPROCESSING
✅ mths_since_recent_bc_dlq → NUMERIC → NUMERIC_PREPROCESSING
✅ mths_since_recent_revol_delinq → NUMERIC → NUMERIC_PREPROCESSING
✅ num_tl_90g_dpd_24m → NUMERIC → NUMERIC_PREPROCESSING
✅ pct_tl_nvr_dlq → NUMERIC → NUMERIC_PREPROCESSING
✅ sec_app_mths_since_last_major_derog → NUMERIC → NUMERIC_PREPROCESSING


In [134]:
# Cell 29 — Text representation verification

print("TEXT REPRESENTATION CHECK")
print("=" * 60)

for column in text_representation_decisions:

    row = representation_plan[
        representation_plan["column"] == column
    ].iloc[0]

    print(
        f"\n{column}"
        f"\n  Category      : {row['representation_category']}"
        f"\n  Transformation: {row['planned_transformation']}"
        f"\n  Reason        : {row['reason']}"
    )

TEXT REPRESENTATION CHECK

emp_title
  Category      : HIGH_CARDINALITY_CATEGORICAL
  Transformation: REVIEW_ENCODER
  Reason        : Applicant employment title has high cardinality and requires controlled representation rather than raw one-hot encoding.

desc
  Category      : FREE_TEXT
  Transformation: DROP_INITIAL_VERSION
  Reason        : Free-form loan description requires a separate NLP pipeline. It will not be included in the initial tabular model.

purpose
  Category      : LOW_CARDINALITY_CATEGORICAL
  Transformation: ENCODING
  Reason        : Loan purpose is a structured categorical application-time field.

title
  Category      : HIGH_CARDINALITY_TEXT
  Transformation: DROP_INITIAL_VERSION
  Reason        : Free-form loan title has high cardinality and requires specialized text processing. It will not enter the initial tabular model.


In [135]:
# Cell 30 — Final Step 9 integrity check

step9_checks = {
    "All features represented": (
        len(representation_plan) ==
        len(accepted_quality_clean.columns)
    ),

    "No duplicate feature entries": (
        representation_plan["column"].duplicated().sum() == 0
    ),

    "No generic REVIEW decisions": (
        (
            representation_plan["planned_transformation"]
            == "REVIEW"
        ).sum() == 0
    ),

    "Actual dates correctly classified": (
        all(
            representation_plan.loc[
                representation_plan["column"] == column,
                "representation_category"
            ].iloc[0] == "DATE"
            for column in actual_date_features
        )
    ),

    "Non-date numeric fields correctly classified": (
        all(
            representation_plan.loc[
                representation_plan["column"] == column,
                "representation_category"
            ].iloc[0] == "NUMERIC"
            for column in numeric_non_date_candidates
        )
    )
}

print("STEP 9 INTEGRITY CHECK")
print("=" * 60)

for check, result in step9_checks.items():
    print(
        "✅" if result else "❌",
        check
    )

if all(step9_checks.values()):
    print("\n🟢 STEP 9 REPRESENTATION CHECK PASSED")
else:
    print("\n🔴 STEP 9 NEEDS REVIEW")

STEP 9 INTEGRITY CHECK
✅ All features represented
✅ No duplicate feature entries
✅ No generic REVIEW decisions
✅ Actual dates correctly classified
✅ Non-date numeric fields correctly classified

🟢 STEP 9 REPRESENTATION CHECK PASSED


In [136]:
# Cell 31 — Save finalized Step 9 representation plan

representation_plan.to_csv(
    "reports/preprocessing_v1_audits/accepted_step9_representation_plan.csv",
    index=False
)

print(
    "Saved: accepted_step9_representation_plan.csv"
)

Saved: accepted_step9_representation_plan.csv


**STEP 10 — MISSING-VALUE STRATEGY**

In [137]:
# Step 10
# Cell 1 — Load Step 8 cleaned dataset and Step 9 representation plan

import pandas as pd
import numpy as np

accepted_quality_clean = pd.read_csv(
    "data/interim/accepted_quality_clean.csv",
    low_memory=False
)

representation_plan = pd.read_csv(
    "reports/preprocessing_v1_audits/accepted_step9_representation_plan.csv"
)

print("STEP 10 — MISSING-VALUE STRATEGY")
print("=" * 60)

print("Rows   :", accepted_quality_clean.shape[0])
print("Columns:", accepted_quality_clean.shape[1])
print("Target present:", "target" in accepted_quality_clean.columns)

STEP 10 — MISSING-VALUE STRATEGY
Rows   : 1348099
Columns: 103
Target present: False


In [138]:
# Cell 2 — Complete missing-value audit

missing_audit = pd.DataFrame({
    "column": accepted_quality_clean.columns,
    "missing_count": accepted_quality_clean.isna().sum().values,
    "missing_percentage": (
        accepted_quality_clean.isna().mean().values * 100
    ),
    "non_missing_count": (
        accepted_quality_clean.notna().sum().values
    ),
    "dtype": (
        accepted_quality_clean.dtypes.astype(str).values
    ),
    "unique_non_missing": (
        accepted_quality_clean.nunique(dropna=True).values
    )
})

missing_audit = missing_audit.sort_values(
    "missing_percentage",
    ascending=False
).reset_index(drop=True)

missing_audit

,column,missing_count,missing_percentage,non_missing_count,dtype,unique_non_missing
0,sec_app_mths_since_last_major_derog,1341450,99.506787,6649,float64,113
1,sec_app_revol_util,1329791,98.641940,18308,float64,1112
2,revol_bal_joint,1329464,98.617683,18635,float64,16157
3,sec_app_mort_acc,1329463,98.617609,18636,float64,18
4,sec_app_fico_range_high,1329463,98.617609,18636,float64,60
...,...,...,...,...,...,...
98,application_type,0,0.000000,1348099,object,2
99,fico_range_high,0,0.000000,1348099,float64,48
100,fico_range_low,0,0.000000,1348099,float64,48
101,term,0,0.000000,1348099,object,2


In [139]:
# Cell 3 — Features containing missing values

missing_features = missing_audit[
    missing_audit["missing_count"] > 0
].copy()

print(
    "Features containing missing values:",
    len(missing_features)
)

missing_features

Features containing missing values: 89


,column,missing_count,missing_percentage,non_missing_count,dtype,unique_non_missing
0,sec_app_mths_since_last_major_derog,1341450,99.506787,6649,float64,113
1,sec_app_revol_util,1329791,98.641940,18308,float64,1112
2,revol_bal_joint,1329464,98.617683,18635,float64,16157
3,sec_app_mort_acc,1329463,98.617609,18636,float64,18
4,sec_app_fico_range_high,1329463,98.617609,18636,float64,60
...,...,...,...,...,...,...
84,earliest_cr_line,29,0.002151,1348070,object,739
85,delinq_2yrs,29,0.002151,1348070,float64,31
86,delinq_amnt,29,0.002151,1348070,float64,2006
87,annual_inc,4,0.000297,1348095,float64,64462


In [140]:
# Cell 4 — Missingness severity buckets

def missingness_bucket(percentage):

    if percentage == 0:
        return "NO_MISSING"

    elif percentage < 5:
        return "LOW_<5%"

    elif percentage < 20:
        return "MODERATE_5-20%"

    elif percentage < 50:
        return "HIGH_20-50%"

    elif percentage < 80:
        return "VERY_HIGH_50-80%"

    else:
        return "EXTREME_>=80%"


missing_audit["missingness_bucket"] = (
    missing_audit["missing_percentage"]
    .apply(missingness_bucket)
)

print(
    missing_audit[
        "missingness_bucket"
    ].value_counts()
)

missingness_bucket
LOW_<5%             27
MODERATE_5-20%      26
EXTREME_>=80%       18
VERY_HIGH_50-80%    18
NO_MISSING          14
Name: count, dtype: int64


In [141]:
# Cell 5 — Missingness summary

missingness_summary = (
    missing_audit[
        missing_audit["missing_count"] > 0
    ][
        [
            "column",
            "missing_count",
            "missing_percentage",
            "missingness_bucket",
            "dtype"
        ]
    ]
)

missingness_summary

,column,missing_count,missing_percentage,missingness_bucket,dtype
0,sec_app_mths_since_last_major_derog,1341450,99.506787,EXTREME_>=80%,float64
1,sec_app_revol_util,1329791,98.641940,EXTREME_>=80%,float64
2,revol_bal_joint,1329464,98.617683,EXTREME_>=80%,float64
3,sec_app_mort_acc,1329463,98.617609,EXTREME_>=80%,float64
4,sec_app_fico_range_high,1329463,98.617609,EXTREME_>=80%,float64
...,...,...,...,...,...
84,earliest_cr_line,29,0.002151,LOW_<5%,object
85,delinq_2yrs,29,0.002151,LOW_<5%,float64
86,delinq_amnt,29,0.002151,LOW_<5%,float64
87,annual_inc,4,0.000297,LOW_<5%,float64


In [142]:
# Cell 6 — Completely missing columns

completely_missing_columns = (
    missing_audit[
        missing_audit["missing_percentage"] == 100
    ]["column"]
    .tolist()
)

print(
    "Completely missing columns:",
    len(completely_missing_columns)
)

for column in completely_missing_columns:
    print("-", column)

Completely missing columns: 0


In [143]:
# Cell 7 — Extremely high missingness

extreme_missing_columns = (
    missing_audit[
        missing_audit["missing_percentage"] >= 80
    ][
        [
            "column",
            "missing_count",
            "missing_percentage",
            "dtype",
            "unique_non_missing"
        ]
    ]
)

print(
    "Features with >=80% missing values:",
    len(extreme_missing_columns)
)

extreme_missing_columns

Features with >=80% missing values: 18


,column,missing_count,missing_percentage,dtype,unique_non_missing
0,sec_app_mths_since_last_major_derog,1341450,99.506787,float64,113
1,sec_app_revol_util,1329791,98.641940,float64,1112
2,revol_bal_joint,1329464,98.617683,float64,16157
3,sec_app_mort_acc,1329463,98.617609,float64,18
4,sec_app_fico_range_high,1329463,98.617609,float64,60
5,sec_app_earliest_cr_line,1329463,98.617609,object,570
6,sec_app_inq_last_6mths,1329463,98.617609,float64,7
7,sec_app_fico_range_low,1329463,98.617609,float64,60
8,sec_app_open_acc,1329463,98.617609,float64,54
9,sec_app_open_act_il,1329463,98.617609,float64,35


In [144]:
# Cell 8 — Numeric missing-value audit

numeric_columns = accepted_quality_clean.select_dtypes(
    include=np.number
).columns.tolist()

numeric_missing_audit = missing_audit[
    missing_audit["column"].isin(numeric_columns)
].copy()

numeric_missing_audit = numeric_missing_audit[
    numeric_missing_audit["missing_count"] > 0
].copy()

numeric_missing_audit

,column,missing_count,missing_percentage,non_missing_count,dtype,unique_non_missing,missingness_bucket
0,sec_app_mths_since_last_major_derog,1341450,99.506787,6649,float64,113,EXTREME_>=80%
1,sec_app_revol_util,1329791,98.641940,18308,float64,1112,EXTREME_>=80%
2,revol_bal_joint,1329464,98.617683,18635,float64,16157,EXTREME_>=80%
3,sec_app_mort_acc,1329463,98.617609,18636,float64,18,EXTREME_>=80%
4,sec_app_fico_range_high,1329463,98.617609,18636,float64,60,EXTREME_>=80%
...,...,...,...,...,...,...,...
82,acc_now_delinq,29,0.002151,1348070,float64,8,LOW_<5%
83,pub_rec,29,0.002151,1348070,float64,37,LOW_<5%
85,delinq_2yrs,29,0.002151,1348070,float64,31,LOW_<5%
86,delinq_amnt,29,0.002151,1348070,float64,2006,LOW_<5%


In [145]:
# Cell 9 — Categorical missing-value audit

categorical_columns = accepted_quality_clean.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_missing_audit = missing_audit[
    missing_audit["column"].isin(categorical_columns)
].copy()

categorical_missing_audit = categorical_missing_audit[
    categorical_missing_audit["missing_count"] > 0
].copy()

categorical_missing_audit

,column,missing_count,missing_percentage,non_missing_count,dtype,unique_non_missing,missingness_bucket
5,sec_app_earliest_cr_line,1329463,98.617609,18636,object,570,EXTREME_>=80%
13,verification_status_joint,1322498,98.100955,25601,object,3,EXTREME_>=80%
16,desc,1222146,90.656992,125953,object,124398,EXTREME_>=80%
39,emp_title,85950,6.375645,1262149,object,379863,MODERATE_5-20%
40,emp_length,78550,5.826723,1269549,object,11,MODERATE_5-20%
72,title,16662,1.235963,1331437,object,63151,LOW_<5%
84,earliest_cr_line,29,0.002151,1348070,object,739,LOW_<5%
88,zip_code,1,0.000074,1348098,object,945,LOW_<5%


In [146]:
# Cell 10 — Missingness semantic review

semantic_columns = [
    "mths_since_last_delinq",
    "mths_since_last_major_derog",
    "mths_since_last_record",
    "mths_since_recent_bc",
    "mths_since_recent_bc_dlq",
    "mths_since_recent_inq",
    "mths_since_recent_revol_delinq",
    "sec_app_mths_since_last_major_derog",
    "sec_app_mths_since_last_record",
    "sec_app_mths_since_last_major_derog"
]

semantic_columns = [
    column
    for column in semantic_columns
    if column in accepted_quality_clean.columns
]

print("Potential structural-missingness fields:")
print("=" * 60)

for column in semantic_columns:
    print(
        column,
        "| missing:",
        int(accepted_quality_clean[column].isna().sum()),
        "| missing %:",
        round(
            accepted_quality_clean[column].isna().mean() * 100,
            2
        )
    )

Potential structural-missingness fields:
mths_since_last_delinq | missing: 679960 | missing %: 50.44
mths_since_last_major_derog | missing: 994342 | missing %: 73.76
mths_since_last_record | missing: 1118675 | missing %: 82.98
mths_since_recent_bc | missing: 62972 | missing %: 4.67
mths_since_recent_bc_dlq | missing: 1029070 | missing %: 76.33
mths_since_recent_inq | missing: 176822 | missing %: 13.12
mths_since_recent_revol_delinq | missing: 898123 | missing %: 66.62
sec_app_mths_since_last_major_derog | missing: 1341450 | missing %: 99.51
sec_app_mths_since_last_major_derog | missing: 1341450 | missing %: 99.51


In [147]:
# Cell 11 — Row-level missingness

row_missing_count = accepted_quality_clean.isna().sum(axis=1)

row_missing_percentage = (
    row_missing_count /
    accepted_quality_clean.shape[1] *
    100
)

print("Row-level missingness")
print("=" * 60)

print(
    "Minimum missing fields:",
    row_missing_count.min()
)

print(
    "Median missing fields:",
    row_missing_count.median()
)

print(
    "Maximum missing fields:",
    row_missing_count.max()
)

print(
    "Average missing fields:",
    round(row_missing_count.mean(), 2)
)

print(
    "Rows with zero missing values:",
    int((row_missing_count == 0).sum())
)

print(
    "Rows with at least one missing value:",
    int((row_missing_count > 0).sum())
)

Row-level missingness
Minimum missing fields: 1
Median missing fields: 33.0
Maximum missing fields: 84
Average missing fields: 30.62
Rows with zero missing values: 0
Rows with at least one missing value: 1348099


In [148]:
# Cell 12 — Row-level missingness distribution

row_missing_distribution = (
    row_missing_count
    .value_counts()
    .sort_index()
)

row_missing_distribution.head(30)

1        117
2        983
3       1311
4       2114
5       2016
6       3382
7       4702
8       1828
9        968
10       609
11       321
12       161
13        79
14       189
15       744
16       844
17      9781
18     53998
19     68634
20     71860
21     79195
22    150396
23     50917
24     15565
25     10508
26      4189
27      1516
28       534
29       151
30      1001
Name: count, dtype: int64

In [149]:
# Cell 13 — Extreme row-level missingness

extreme_row_missing_threshold = 50

extreme_missing_rows = (
    row_missing_count >= extreme_row_missing_threshold
)

print(
    "Rows with at least",
    extreme_row_missing_threshold,
    "missing fields:",
    int(extreme_missing_rows.sum())
)

print(
    "Percentage of dataset:",
    round(
        extreme_missing_rows.mean() * 100,
        4
    ),
    "%"
)

Rows with at least 50 missing fields: 70276
Percentage of dataset: 5.213 %


In [150]:
# Cell 14 — Combine missingness with Step 9 representation

missing_strategy_audit = missing_audit.merge(
    representation_plan[
        [
            "column",
            "representation_category",
            "planned_transformation"
        ]
    ],
    on="column",
    how="left"
)

missing_strategy_audit = missing_strategy_audit[
    missing_strategy_audit["missing_count"] > 0
].copy()

missing_strategy_audit

,column,missing_count,missing_percentage,non_missing_count,dtype,unique_non_missing,missingness_bucket,representation_category,planned_transformation
0,sec_app_mths_since_last_major_derog,1341450,99.506787,6649,float64,113,EXTREME_>=80%,NUMERIC,NUMERIC_PREPROCESSING
1,sec_app_revol_util,1329791,98.641940,18308,float64,1112,EXTREME_>=80%,NUMERIC,NUMERIC_PREPROCESSING
2,revol_bal_joint,1329464,98.617683,18635,float64,16157,EXTREME_>=80%,NUMERIC,NUMERIC_PREPROCESSING
3,sec_app_mort_acc,1329463,98.617609,18636,float64,18,EXTREME_>=80%,NUMERIC,NUMERIC_PREPROCESSING
4,sec_app_fico_range_high,1329463,98.617609,18636,float64,60,EXTREME_>=80%,NUMERIC,NUMERIC_PREPROCESSING
...,...,...,...,...,...,...,...,...,...
84,earliest_cr_line,29,0.002151,1348070,object,739,LOW_<5%,DATE,DATE_FEATURE_ENGINEERING
85,delinq_2yrs,29,0.002151,1348070,float64,31,LOW_<5%,NUMERIC,NUMERIC_PREPROCESSING
86,delinq_amnt,29,0.002151,1348070,float64,2006,LOW_<5%,NUMERIC,NUMERIC_PREPROCESSING
87,annual_inc,4,0.000297,1348095,float64,64462,LOW_<5%,NUMERIC,NUMERIC_PREPROCESSING


In [151]:
# Cell 15 — Initial missing-value strategy classification

missing_strategy_audit["missing_value_strategy"] = "REVIEW"

for index, row in missing_strategy_audit.iterrows():

    column = row["column"]
    category = row["representation_category"]

    if row["missing_percentage"] == 100:
        strategy = "DROP_COLUMN_REVIEW"

    elif category == "NUMERIC":
        strategy = "NUMERIC_IMPUTATION_REVIEW"

    elif category == "CATEGORICAL":
        strategy = "CATEGORICAL_IMPUTATION_REVIEW"

    elif category == "DATE":
        strategy = "DATE_MISSINGNESS_REVIEW"

    elif category == "TEXT_OR_HIGH_CARDINALITY":
        strategy = "TEXT_MISSINGNESS_REVIEW"

    else:
        strategy = "SEMANTIC_REVIEW"

    missing_strategy_audit.loc[
        index,
        "missing_value_strategy"
    ] = strategy

missing_strategy_audit[
    [
        "column",
        "missing_percentage",
        "representation_category",
        "missing_value_strategy"
    ]
]

,column,missing_percentage,representation_category,missing_value_strategy
0,sec_app_mths_since_last_major_derog,99.506787,NUMERIC,NUMERIC_IMPUTATION_REVIEW
1,sec_app_revol_util,98.641940,NUMERIC,NUMERIC_IMPUTATION_REVIEW
2,revol_bal_joint,98.617683,NUMERIC,NUMERIC_IMPUTATION_REVIEW
3,sec_app_mort_acc,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
4,sec_app_fico_range_high,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
...,...,...,...,...
84,earliest_cr_line,0.002151,DATE,DATE_MISSINGNESS_REVIEW
85,delinq_2yrs,0.002151,NUMERIC,NUMERIC_IMPUTATION_REVIEW
86,delinq_amnt,0.002151,NUMERIC,NUMERIC_IMPUTATION_REVIEW
87,annual_inc,0.000297,NUMERIC,NUMERIC_IMPUTATION_REVIEW


In [152]:
# Cell 16 — Strategy summary

print("INITIAL MISSING-VALUE STRATEGY SUMMARY")
print("=" * 60)

print(
    missing_strategy_audit[
        "missing_value_strategy"
    ].value_counts()
)

INITIAL MISSING-VALUE STRATEGY SUMMARY
missing_value_strategy
NUMERIC_IMPUTATION_REVIEW        81
CATEGORICAL_IMPUTATION_REVIEW     3
SEMANTIC_REVIEW                   3
DATE_MISSINGNESS_REVIEW           2
Name: count, dtype: int64


In [153]:
# Cell 17 — High-missingness feature review

high_missing_review = missing_strategy_audit[
    missing_strategy_audit["missing_percentage"] >= 50
].copy()

high_missing_review[
    [
        "column",
        "missing_count",
        "missing_percentage",
        "representation_category",
        "missing_value_strategy"
    ]
]

,column,missing_count,missing_percentage,representation_category,missing_value_strategy
0,sec_app_mths_since_last_major_derog,1341450,99.506787,NUMERIC,NUMERIC_IMPUTATION_REVIEW
1,sec_app_revol_util,1329791,98.641940,NUMERIC,NUMERIC_IMPUTATION_REVIEW
2,revol_bal_joint,1329464,98.617683,NUMERIC,NUMERIC_IMPUTATION_REVIEW
3,sec_app_mort_acc,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
4,sec_app_fico_range_high,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
5,sec_app_earliest_cr_line,1329463,98.617609,DATE,DATE_MISSINGNESS_REVIEW
6,sec_app_inq_last_6mths,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
7,sec_app_fico_range_low,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
8,sec_app_open_acc,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW
9,sec_app_open_act_il,1329463,98.617609,NUMERIC,NUMERIC_IMPUTATION_REVIEW


In [154]:
# Cell 18 — Numeric strategy review

numeric_strategy_review = missing_strategy_audit[
    missing_strategy_audit["representation_category"] == "NUMERIC"
].copy()

numeric_strategy_review[
    [
        "column",
        "missing_count",
        "missing_percentage",
        "unique_non_missing",
        "missing_value_strategy"
    ]
]

,column,missing_count,missing_percentage,unique_non_missing,missing_value_strategy
0,sec_app_mths_since_last_major_derog,1341450,99.506787,113,NUMERIC_IMPUTATION_REVIEW
1,sec_app_revol_util,1329791,98.641940,1112,NUMERIC_IMPUTATION_REVIEW
2,revol_bal_joint,1329464,98.617683,16157,NUMERIC_IMPUTATION_REVIEW
3,sec_app_mort_acc,1329463,98.617609,18,NUMERIC_IMPUTATION_REVIEW
4,sec_app_fico_range_high,1329463,98.617609,60,NUMERIC_IMPUTATION_REVIEW
...,...,...,...,...,...
82,acc_now_delinq,29,0.002151,8,NUMERIC_IMPUTATION_REVIEW
83,pub_rec,29,0.002151,37,NUMERIC_IMPUTATION_REVIEW
85,delinq_2yrs,29,0.002151,31,NUMERIC_IMPUTATION_REVIEW
86,delinq_amnt,29,0.002151,2006,NUMERIC_IMPUTATION_REVIEW


In [155]:
# Cell 19 — Categorical strategy review

categorical_strategy_review = missing_strategy_audit[
    missing_strategy_audit["representation_category"] == "CATEGORICAL"
].copy()

categorical_strategy_review[
    [
        "column",
        "missing_count",
        "missing_percentage",
        "unique_non_missing",
        "missing_value_strategy"
    ]
]

,column,missing_count,missing_percentage,unique_non_missing,missing_value_strategy
13,verification_status_joint,1322498,98.100955,3,CATEGORICAL_IMPUTATION_REVIEW
40,emp_length,78550,5.826723,11,CATEGORICAL_IMPUTATION_REVIEW
88,zip_code,1,0.000074,945,CATEGORICAL_IMPUTATION_REVIEW


In [156]:
# Cell 20 — Final missing-value strategy rules

missing_strategy_rules = {
    "numeric": (
        "Evaluate training-set imputation using a statistic learned "
        "from training data only. Preserve meaningful missingness "
        "indicators where domain semantics justify them."
    ),

    "categorical": (
        "Treat missingness explicitly where appropriate, using a "
        "training-fitted categorical strategy."
    ),

    "date": (
        "Do not blindly replace missing dates. Preserve missingness "
        "meaning and derive date features using training-fitted rules."
    ),

    "text": (
        "Treat missing text as absence of text or exclude the field "
        "from the initial tabular model according to the Step 9 plan."
    ),

    "extreme_missingness": (
        "Review each feature individually. Do not automatically "
        "delete a feature solely because its missingness is high."
    ),

    "row_missingness": (
        "Do not automatically remove applicants based only on the "
        "number of missing fields. Investigate whether missingness "
        "represents valid applicant characteristics."
    )
}

for category, rule in missing_strategy_rules.items():
    print(f"\n{category.upper()}")
    print(rule)


NUMERIC
Evaluate training-set imputation using a statistic learned from training data only. Preserve meaningful missingness indicators where domain semantics justify them.

CATEGORICAL
Treat missingness explicitly where appropriate, using a training-fitted categorical strategy.

DATE
Do not blindly replace missing dates. Preserve missingness meaning and derive date features using training-fitted rules.

TEXT
Treat missing text as absence of text or exclude the field from the initial tabular model according to the Step 9 plan.

EXTREME_MISSINGNESS
Review each feature individually. Do not automatically delete a feature solely because its missingness is high.

ROW_MISSINGNESS
Do not automatically remove applicants based only on the number of missing fields. Investigate whether missingness represents valid applicant characteristics.


In [157]:
# Cell 21 — Document final strategy categories

strategy_documentation = {
    "NUMERIC_IMPUTATION_REVIEW":
        "Numeric imputation must be fitted using training data only.",

    "CATEGORICAL_IMPUTATION_REVIEW":
        "Categorical missingness requires training-fitted handling.",

    "DATE_MISSINGNESS_REVIEW":
        "Date missingness requires semantic handling before date transformation.",

    "TEXT_MISSINGNESS_REVIEW":
        "Missing text represents absent text and follows the Step 9 text decision.",

    "DROP_COLUMN_REVIEW":
        "Column requires explicit review before removal.",

    "SEMANTIC_REVIEW":
        "Feature requires domain-specific missingness interpretation."
}

missing_strategy_audit[
    "strategy_documentation"
] = missing_strategy_audit[
    "missing_value_strategy"
].map(
    strategy_documentation
)

missing_strategy_audit[
    [
        "column",
        "missing_percentage",
        "missing_value_strategy",
        "strategy_documentation"
    ]
]

,column,missing_percentage,missing_value_strategy,strategy_documentation
0,sec_app_mths_since_last_major_derog,99.506787,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
1,sec_app_revol_util,98.641940,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
2,revol_bal_joint,98.617683,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
3,sec_app_mort_acc,98.617609,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
4,sec_app_fico_range_high,98.617609,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
...,...,...,...,...
84,earliest_cr_line,0.002151,DATE_MISSINGNESS_REVIEW,Date missingness requires semantic handling be...
85,delinq_2yrs,0.002151,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
86,delinq_amnt,0.002151,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...
87,annual_inc,0.000297,NUMERIC_IMPUTATION_REVIEW,Numeric imputation must be fitted using traini...


In [158]:
# Cell 22 — Unresolved missing-value decisions

unresolved_missing = missing_strategy_audit[
    missing_strategy_audit[
        "missing_value_strategy"
    ].isin([
        "REVIEW",
        "SEMANTIC_REVIEW",
        "DROP_COLUMN_REVIEW"
    ])
].copy()

print(
    "Features requiring explicit missing-value review:",
    len(unresolved_missing)
)

unresolved_missing[
    [
        "column",
        "missing_percentage",
        "representation_category",
        "missing_value_strategy"
    ]
]

Features requiring explicit missing-value review: 3


,column,missing_percentage,representation_category,missing_value_strategy
16,desc,90.656992,FREE_TEXT,SEMANTIC_REVIEW
39,emp_title,6.375645,HIGH_CARDINALITY_CATEGORICAL,SEMANTIC_REVIEW
72,title,1.235963,HIGH_CARDINALITY_TEXT,SEMANTIC_REVIEW


In [159]:
# Cell 23 — Save Step 10 missing-value audit

missing_audit.to_csv(
    "reports/preprocessing_v1_audits/accepted_step10_missing_audit.csv",
    index=False
)

missing_strategy_audit.to_csv(
    "reports/preprocessing_v1_audits/accepted_step10_missing_strategy.csv",
    index=False
)

print("Saved:")
print("- accepted_step10_missing_audit.csv")
print("- accepted_step10_missing_strategy.csv")

Saved:
- accepted_step10_missing_audit.csv
- accepted_step10_missing_strategy.csv


In [160]:
# Cell 24 — Step 10 structural verification

step10_checks = {
    "Missing audit created": (
        len(missing_audit) ==
        len(accepted_quality_clean.columns)
    ),

    "Only existing features audited": (
        set(missing_audit["column"]) ==
        set(accepted_quality_clean.columns)
    ),

    "Missing counts are non-negative": (
        (missing_audit["missing_count"] >= 0).all()
    ),

    "Missing percentages valid": (
        (
            (missing_audit["missing_percentage"] >= 0) &
            (missing_audit["missing_percentage"] <= 100)
        ).all()
    ),

    "No data modified": (
        accepted_quality_clean.shape[1] ==
        len(missing_audit)
    )
}

print("STEP 10 STRUCTURAL CHECK")
print("=" * 60)

for check, result in step10_checks.items():
    print(
        "✅" if result else "❌",
        check
    )

if all(step10_checks.values()):
    print("\n🟢 STEP 10 AUDIT STRUCTURE PASSED")
else:
    print("\n🔴 STEP 10 NEEDS REVIEW")

STEP 10 STRUCTURAL CHECK
✅ Missing audit created
✅ Only existing features audited
✅ Missing counts are non-negative
✅ Missing percentages valid
✅ No data modified

🟢 STEP 10 AUDIT STRUCTURE PASSED


In [161]:
# Cell 25 — Step 10 final summary

print("STEP 10 — MISSING-VALUE STRATEGY SUMMARY")
print("=" * 60)

print(
    "Total features:",
    len(accepted_quality_clean.columns)
)

print(
    "Features with missing values:",
    len(missing_features)
)

print(
    "Completely missing features:",
    len(completely_missing_columns)
)

print(
    "Features with >=80% missing:",
    len(extreme_missing_columns)
)

print(
    "Rows with at least one missing value:",
    int((row_missing_count > 0).sum())
)

print(
    "Rows with >=50 missing fields:",
    int(extreme_missing_rows.sum())
)

print("\nMissing-value strategy categories:")
print(
    missing_strategy_audit[
        "missing_value_strategy"
    ].value_counts()
)

STEP 10 — MISSING-VALUE STRATEGY SUMMARY
Total features: 103
Features with missing values: 89
Completely missing features: 0
Features with >=80% missing: 18
Rows with at least one missing value: 1348099
Rows with >=50 missing fields: 70276

Missing-value strategy categories:
missing_value_strategy
NUMERIC_IMPUTATION_REVIEW        81
CATEGORICAL_IMPUTATION_REVIEW     3
SEMANTIC_REVIEW                   3
DATE_MISSINGNESS_REVIEW           2
Name: count, dtype: int64


In [162]:
# Cell 26 — Final semantic missing-value decisions

final_missing_decisions = {
    "desc": {
        "strategy": "DROP_INITIAL_TABULAR_MODEL",
        "reason": (
            "desc is free-form text and 90.66% of values are missing. "
            "The initial model is tabular, so this field will not be "
            "imputed or used as a raw feature."
        )
    },

    "emp_title": {
        "strategy": "MISSING_CATEGORY",
        "reason": (
            "emp_title is a high-cardinality categorical feature. "
            "Missing employment title represents unavailable information. "
            "Treat missingness as an explicit category during categorical "
            "encoding."
        )
    },

    "title": {
        "strategy": "DROP_INITIAL_TABULAR_MODEL",
        "reason": (
            "title is high-cardinality text. The initial model will not "
            "perform text modeling, so missing values will not be imputed. "
            "The field is excluded from the initial tabular feature set."
        )
    }
}

for column, decision in final_missing_decisions.items():

    mask = missing_strategy_audit["column"] == column

    missing_strategy_audit.loc[
        mask,
        "missing_value_strategy"
    ] = decision["strategy"]

    missing_strategy_audit.loc[
        mask,
        "strategy_documentation"
    ] = decision["reason"]

print("Final semantic decisions applied.")

Final semantic decisions applied.


In [163]:
# Cell 27 — Verify final semantic decisions

resolved_fields = [
    "desc",
    "emp_title",
    "title"
]

print("FINAL SEMANTIC MISSING-VALUE DECISIONS")
print("=" * 60)

print(
    missing_strategy_audit[
        missing_strategy_audit["column"].isin(resolved_fields)
    ][
        [
            "column",
            "missing_percentage",
            "representation_category",
            "missing_value_strategy",
            "strategy_documentation"
        ]
    ].to_string(index=False)
)

FINAL SEMANTIC MISSING-VALUE DECISIONS
   column  missing_percentage      representation_category     missing_value_strategy                                                                                                                                                                   strategy_documentation
     desc           90.656992                    FREE_TEXT DROP_INITIAL_TABULAR_MODEL                                       desc is free-form text and 90.66% of values are missing. The initial model is tabular, so this field will not be imputed or used as a raw feature.
emp_title            6.375645 HIGH_CARDINALITY_CATEGORICAL           MISSING_CATEGORY emp_title is a high-cardinality categorical feature. Missing employment title represents unavailable information. Treat missingness as an explicit category during categorical encoding.
    title            1.235963        HIGH_CARDINALITY_TEXT DROP_INITIAL_TABULAR_MODEL     title is high-cardinality text. The initial model will not

In [164]:
# Cell 28 — Final unresolved missing-value check

unresolved_missing = missing_strategy_audit[
    missing_strategy_audit["missing_value_strategy"].isin([
        "REVIEW",
        "SEMANTIC_REVIEW",
        "DROP_COLUMN_REVIEW"
    ])
].copy()

print("UNRESOLVED MISSING-VALUE DECISIONS")
print("=" * 60)

print(
    "Count:",
    len(unresolved_missing)
)

if len(unresolved_missing) > 0:
    print(
        unresolved_missing[
            [
                "column",
                "missing_percentage",
                "representation_category",
                "missing_value_strategy"
            ]
        ].to_string(index=False)
    )
else:
    print("None ✅")

UNRESOLVED MISSING-VALUE DECISIONS
Count: 0
None ✅


In [165]:
# Cell 29 — Final Step 10 strategy audit

step10_final_checks = {
    "No unresolved semantic reviews": (
        len(unresolved_missing) == 0
    ),

    "desc has explicit strategy": (
        missing_strategy_audit.loc[
            missing_strategy_audit["column"] == "desc",
            "missing_value_strategy"
        ].iloc[0] == "DROP_INITIAL_TABULAR_MODEL"
    ),

    "emp_title has explicit strategy": (
        missing_strategy_audit.loc[
            missing_strategy_audit["column"] == "emp_title",
            "missing_value_strategy"
        ].iloc[0] == "MISSING_CATEGORY"
    ),

    "title has explicit strategy": (
        missing_strategy_audit.loc[
            missing_strategy_audit["column"] == "title",
            "missing_value_strategy"
        ].iloc[0] == "DROP_INITIAL_TABULAR_MODEL"
    ),

    "No completely missing features": (
        len(completely_missing_columns) == 0
    )
}

print("STEP 10 — FINAL STRATEGY AUDIT")
print("=" * 60)

for check, result in step10_final_checks.items():
    print(
        "✅" if result else "❌",
        check
    )

if all(step10_final_checks.values()):
    print("\n🟢 STEP 10 FINAL STRATEGY AUDIT PASSED")
else:
    print("\n🔴 STEP 10 STILL NEEDS REVIEW")

STEP 10 — FINAL STRATEGY AUDIT
✅ No unresolved semantic reviews
✅ desc has explicit strategy
✅ emp_title has explicit strategy
✅ title has explicit strategy
✅ No completely missing features

🟢 STEP 10 FINAL STRATEGY AUDIT PASSED


In [166]:
# Cell 30 — Save final Step 10 strategy

missing_strategy_audit.to_csv(
    "reports/preprocessing_v1_audits/accepted_step10_missing_strategy.csv",
    index=False
)

print(
    "Saved final strategy:"
    "\naccepted_step10_missing_strategy.csv"
)

Saved final strategy:
accepted_step10_missing_strategy.csv


In [167]:
# Cell 31 — Final Step 10 summary

print("STEP 10 — FINAL SUMMARY")
print("=" * 60)

print("Total features:", len(accepted_quality_clean.columns))
print("Features with missing values:", len(missing_features))
print("Completely missing features:", len(completely_missing_columns))
print("Features with >=80% missing:", len(extreme_missing_columns))

print("\nFinal strategy distribution:")
print(
    missing_strategy_audit[
        "missing_value_strategy"
    ].value_counts()
)

print("\nFinal unresolved decisions:", len(unresolved_missing))

if len(unresolved_missing) == 0:
    print("\n🟢 STEP 10 IS COMPLETE")
else:
    print("\n🔴 STEP 10 NEEDS REVIEW")

STEP 10 — FINAL SUMMARY
Total features: 103
Features with missing values: 89
Completely missing features: 0
Features with >=80% missing: 18

Final strategy distribution:
missing_value_strategy
NUMERIC_IMPUTATION_REVIEW        81
CATEGORICAL_IMPUTATION_REVIEW     3
DATE_MISSINGNESS_REVIEW           2
DROP_INITIAL_TABULAR_MODEL        2
MISSING_CATEGORY                  1
Name: count, dtype: int64

Final unresolved decisions: 0

🟢 STEP 10 IS COMPLETE


**STEP 11 — TRAIN / VALIDATION / TEST SPLIT**

In [168]:
# Cell 1 — Load Step 10 dataset

import pandas as pd
import numpy as np

accepted_quality_clean = pd.read_csv(
    "data/interim/accepted_quality_clean.csv",
    low_memory=False
)

missing_strategy_audit = pd.read_csv(
    "reports/preprocessing_v1_audits/accepted_step10_missing_strategy.csv",
    low_memory=False
)

print("STEP 11 — DATASET SPLIT")
print("=" * 60)

print("Rows   :", accepted_quality_clean.shape[0])
print("Columns:", accepted_quality_clean.shape[1])

print(
    "Target present:",
    "target" in accepted_quality_clean.columns
)

STEP 11 — DATASET SPLIT
Rows   : 1348099
Columns: 103
Target present: False


In [169]:
# Cell 2 — Diagnose target columns

print("AVAILABLE COLUMNS")
print("=" * 60)

print("Number of columns:", len(accepted_quality_clean.columns))

print("\nColumns containing 'status':")
for column in accepted_quality_clean.columns:
    if "status" in column.lower():
        print("-", column)

print("\nTarget-related columns:")
for column in accepted_quality_clean.columns:
    if any(
        word in column.lower()
        for word in ["target", "loan_status", "status"]
    ):
        print("-", column)

AVAILABLE COLUMNS
Number of columns: 103

Columns containing 'status':
- verification_status
- verification_status_joint

Target-related columns:
- verification_status
- verification_status_joint


In [170]:
# Cell 3 — Safely restore target by row position

print("TARGET RESTORATION CHECK")
print("=" * 60)

# Basic row-count check
print("accepted_eligible rows:", len(accepted_eligible))
print("accepted_quality_clean rows:", len(accepted_quality_clean))

if len(accepted_eligible) != len(accepted_quality_clean):
    raise ValueError(
        "Row counts differ. Target cannot be restored safely by position."
    )

# Columns that exist in both datasets and should preserve row order
verification_columns = [
    "loan_amnt",
    "annual_inc",
    "dti"
]

print("\nChecking row alignment using retained columns...")

for column in verification_columns:

    if column not in accepted_eligible.columns:
        raise ValueError(
            f"{column} missing from accepted_eligible."
        )

    if column not in accepted_quality_clean.columns:
        raise ValueError(
            f"{column} missing from accepted_quality_clean."
        )

    original_values = accepted_eligible[column].reset_index(drop=True)
    clean_values = accepted_quality_clean[column].reset_index(drop=True)

    if not original_values.equals(clean_values):
        raise ValueError(
            f"Row alignment check failed for column: {column}"
        )

    print(f"✅ {column}: row order matches")

# Restore target by POSITION, not index
accepted_quality_clean["target"] = (
    accepted_eligible["target"]
    .reset_index(drop=True)
    .to_numpy()
)

print("\nTarget restored by row position.")

print("Target dtype:", accepted_quality_clean["target"].dtype)

print(
    "Missing target:",
    accepted_quality_clean["target"].isna().sum()
)

print("\nTarget distribution:")
print(
    accepted_quality_clean["target"]
    .value_counts(dropna=False)
    .sort_index()
)

TARGET RESTORATION CHECK
accepted_eligible rows: 1348099
accepted_quality_clean rows: 1348099

Checking row alignment using retained columns...
✅ loan_amnt: row order matches
✅ annual_inc: row order matches
✅ dti: row order matches

Target restored by row position.
Target dtype: float64
Missing target: 0

Target distribution:
target
0.0    1078739
1.0     269360
Name: count, dtype: int64


In [171]:
# Cell 4 — Verify target restoration

print("TARGET RESTORATION VERIFICATION")
print("=" * 60)

print("Target present:", "target" in accepted_quality_clean.columns)
print("Total columns:", accepted_quality_clean.shape[1])
print("Target missing:", accepted_quality_clean["target"].isna().sum())

if "target" in accepted_quality_clean.columns:
    print("\nTarget values:")
    print(
        accepted_quality_clean["target"]
        .value_counts(dropna=False)
        .sort_index()
    )

if (
    "target" in accepted_quality_clean.columns
    and accepted_quality_clean["target"].isna().sum() == 0
):
    print("\n✅ Target successfully restored.")
else:
    print("\n❌ Target restoration needs review.")

TARGET RESTORATION VERIFICATION
Target present: True
Total columns: 104
Target missing: 0

Target values:
target
0.0    1078739
1.0     269360
Name: count, dtype: int64

✅ Target successfully restored.


In [172]:
# Cell 5 — Stratified Train / Validation / Test Split

from sklearn.model_selection import train_test_split

print("STEP 11 — DATA SPLIT")
print("=" * 60)

# Separate features and target
X = accepted_quality_clean.drop(columns=["target"])
y = accepted_quality_clean["target"]

print("Total rows:", len(accepted_quality_clean))
print("Feature columns:", X.shape[1])
print("Target column: target")

# First split:
# 80% training
# 20% temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Second split:
# Temporary 20% -> 10% validation + 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("\nSplit completed.")

print("\nDataset sizes:")
print("Training   :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

STEP 11 — DATA SPLIT
Total rows: 1348099
Feature columns: 103
Target column: target

Split completed.

Dataset sizes:
Training   : (1078479, 103)
Validation : (134810, 103)
Test       : (134810, 103)


In [173]:
# Cell 6 — Verify split integrity and target distribution

print("SPLIT INTEGRITY CHECK")
print("=" * 60)

print("Training rows   :", len(X_train))
print("Validation rows :", len(X_val))
print("Test rows       :", len(X_test))

print(
    "\nTotal rows:",
    len(X_train) + len(X_val) + len(X_test)
)

print("\nTarget distribution:")
print("-" * 60)

print("\nTRAIN")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nVALIDATION")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTEST")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

SPLIT INTEGRITY CHECK
Training rows   : 1078479
Validation rows : 134810
Test rows       : 134810

Total rows: 1348099

Target distribution:
------------------------------------------------------------

TRAIN
target
0.0    862991
1.0    215488
Name: count, dtype: int64
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64

VALIDATION
target
0.0    107874
1.0     26936
Name: count, dtype: int64
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64

TEST
target
0.0    107874
1.0     26936
Name: count, dtype: int64
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64


In [174]:
# Cell 7 — Check for accidental overlap between splits

print("SPLIT OVERLAP CHECK")
print("=" * 60)

train_indices = set(X_train.index)
validation_indices = set(X_val.index)
test_indices = set(X_test.index)

train_val_overlap = train_indices & validation_indices
train_test_overlap = train_indices & test_indices
validation_test_overlap = validation_indices & test_indices

print("Train ∩ Validation:", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(validation_test_overlap))

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(validation_test_overlap) == 0
):
    print("\n✅ No row overlap between splits.")
else:
    print("\n❌ Overlap detected. Review the split.")

SPLIT OVERLAP CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

✅ No row overlap between splits.


In [175]:
# Cell 8 — Training-set missing-value audit

print("STEP 11 — MISSING-VALUE AUDIT")
print("=" * 60)

# Audit missing values using TRAINING data only
train_missing = X_train.isna().sum()

train_missing = train_missing[
    train_missing > 0
].sort_values(ascending=False)

print(
    "Features with missing values in training:",
    len(train_missing)
)

if len(train_missing) > 0:

    train_missing_audit = pd.DataFrame({
        "column": train_missing.index,
        "missing_count": train_missing.values,
        "missing_percentage": (
            train_missing.values / len(X_train) * 100
        )
    })

    print("\nTraining missing-value audit:")
    display(train_missing_audit)

else:
    print("\n✅ No missing values in training features.")

STEP 11 — MISSING-VALUE AUDIT
Features with missing values in training: 89

Training missing-value audit:


,column,missing_count,missing_percentage
0,sec_app_mths_since_last_major_derog,1073173,99.508011
1,sec_app_revol_util,1063807,98.639566
2,sec_app_fico_range_high,1063550,98.615736
3,sec_app_collections_12_mths_ex_med,1063550,98.615736
4,sec_app_chargeoff_within_12_mths,1063550,98.615736
...,...,...,...
84,pub_rec,24,0.002225
85,total_acc,24,0.002225
86,delinq_amnt,24,0.002225
87,annual_inc,4,0.000371


In [176]:
# Cell 9 — Missingness severity classification

print("MISSINGNESS SEVERITY AUDIT")
print("=" * 60)

missingness_audit = train_missing_audit.copy()

def classify_missingness(percentage):
    if percentage == 0:
        return "No missing"
    elif percentage < 1:
        return "Very low (<1%)"
    elif percentage < 5:
        return "Low (1-5%)"
    elif percentage < 20:
        return "Moderate (5-20%)"
    elif percentage < 50:
        return "High (20-50%)"
    elif percentage < 80:
        return "Very high (50-80%)"
    else:
        return "Extreme (80%+)"

missingness_audit["severity"] = (
    missingness_audit["missing_percentage"]
    .apply(classify_missingness)
)

print("Missingness categories:")
print(
    missingness_audit["severity"]
    .value_counts()
)

print("\nFull missingness audit:")
display(
    missingness_audit.sort_values(
        "missing_percentage",
        ascending=False
    )
)

MISSINGNESS SEVERITY AUDIT
Missingness categories:
severity
Moderate (5-20%)      26
Extreme (80%+)        18
Very high (50-80%)    18
Very low (<1%)        16
Low (1-5%)            11
Name: count, dtype: int64

Full missingness audit:


,column,missing_count,missing_percentage,severity
0,sec_app_mths_since_last_major_derog,1073173,99.508011,Extreme (80%+)
1,sec_app_revol_util,1063807,98.639566,Extreme (80%+)
7,sec_app_open_acc,1063550,98.615736,Extreme (80%+)
12,sec_app_earliest_cr_line,1063550,98.615736,Extreme (80%+)
10,sec_app_fico_range_low,1063550,98.615736,Extreme (80%+)
...,...,...,...,...
84,pub_rec,24,0.002225,Very low (<1%)
85,total_acc,24,0.002225,Very low (<1%)
86,delinq_amnt,24,0.002225,Very low (<1%)
87,annual_inc,4,0.000371,Very low (<1%)


In [177]:
# Cell 10 — Identify extreme-missingness features

EXTREME_MISSINGNESS_THRESHOLD = 80.0

extreme_missing_columns = (
    missingness_audit[
        missingness_audit["missing_percentage"]
        >= EXTREME_MISSINGNESS_THRESHOLD
    ]
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

print("EXTREME-MISSINGNESS FEATURES")
print("=" * 60)

print(
    "Threshold:",
    f"{EXTREME_MISSINGNESS_THRESHOLD}%"
)

print(
    "Number of features:",
    len(extreme_missing_columns)
)

display(extreme_missing_columns)

EXTREME-MISSINGNESS FEATURES
Threshold: 80.0%
Number of features: 18


,column,missing_count,missing_percentage,severity
0,sec_app_mths_since_last_major_derog,1073173,99.508011,Extreme (80%+)
1,sec_app_revol_util,1063807,98.639566,Extreme (80%+)
2,sec_app_open_acc,1063550,98.615736,Extreme (80%+)
3,sec_app_earliest_cr_line,1063550,98.615736,Extreme (80%+)
4,sec_app_inq_last_6mths,1063550,98.615736,Extreme (80%+)
5,sec_app_fico_range_low,1063550,98.615736,Extreme (80%+)
6,sec_app_mort_acc,1063550,98.615736,Extreme (80%+)
7,revol_bal_joint,1063550,98.615736,Extreme (80%+)
8,sec_app_open_act_il,1063550,98.615736,Extreme (80%+)
9,sec_app_num_rev_accts,1063550,98.615736,Extreme (80%+)


In [178]:
# Cell 11 — Inspect extreme-missingness feature types

print("EXTREME-MISSINGNESS FEATURE TYPE AUDIT")
print("=" * 60)

extreme_columns = extreme_missing_columns["column"].tolist()

extreme_type_audit = pd.DataFrame({
    "column": extreme_columns,
    "dtype": [
        X_train[column].dtype
        for column in extreme_columns
    ],
    "unique_values": [
        X_train[column].nunique(dropna=True)
        for column in extreme_columns
    ],
    "missing_count": [
        X_train[column].isna().sum()
        for column in extreme_columns
    ],
    "missing_percentage": [
        X_train[column].isna().mean() * 100
        for column in extreme_columns
    ]
})

display(
    extreme_type_audit.sort_values(
        "missing_percentage",
        ascending=False
    )
)

EXTREME-MISSINGNESS FEATURE TYPE AUDIT


,column,dtype,unique_values,missing_count,missing_percentage
0,sec_app_mths_since_last_major_derog,float64,110,1073173,99.508011
1,sec_app_revol_util,float64,1097,1063807,98.639566
7,revol_bal_joint,float64,13268,1063550,98.615736
12,sec_app_fico_range_high,float64,60,1063550,98.615736
11,sec_app_collections_12_mths_ex_med,float64,12,1063550,98.615736
10,sec_app_chargeoff_within_12_mths,float64,15,1063550,98.615736
8,sec_app_open_act_il,float64,33,1063550,98.615736
9,sec_app_num_rev_accts,float64,65,1063550,98.615736
6,sec_app_mort_acc,float64,18,1063550,98.615736
5,sec_app_fico_range_low,float64,60,1063550,98.615736


In [179]:
# Cell 12 — Extreme-missingness pattern audit

print("EXTREME-MISSINGNESS PATTERN AUDIT")
print("=" * 60)

# Create a binary missingness matrix for the 18 extreme-missingness features
extreme_missing_flags = X_train[extreme_columns].isna().astype("int8")

print("Missingness flag matrix shape:")
print(extreme_missing_flags.shape)

print("\nMissing-value count by feature:")
print(
    extreme_missing_flags.sum()
    .sort_values(ascending=False)
)

print("\nFeatures with identical missingness patterns:")

pattern_groups = {}

for column in extreme_columns:
    pattern = tuple(extreme_missing_flags[column].values)

    pattern_groups.setdefault(pattern, []).append(column)

identical_groups = [
    columns
    for columns in pattern_groups.values()
    if len(columns) > 1
]

if len(identical_groups) == 0:
    print("None")
else:
    for group_number, columns in enumerate(
        identical_groups,
        start=1
    ):
        print(f"\nPattern group {group_number}:")
        for column in columns:
            print(" -", column)

print("\nMissingness correlation matrix:")
display(
    extreme_missing_flags.corr()
)

EXTREME-MISSINGNESS PATTERN AUDIT
Missingness flag matrix shape:
(1078479, 18)

Missing-value count by feature:
sec_app_mths_since_last_major_derog    1073173
sec_app_revol_util                     1063807
revol_bal_joint                        1063550
sec_app_fico_range_high                1063550
sec_app_collections_12_mths_ex_med     1063550
sec_app_chargeoff_within_12_mths       1063550
sec_app_open_act_il                    1063550
sec_app_num_rev_accts                  1063550
sec_app_mort_acc                       1063550
sec_app_fico_range_low                 1063550
sec_app_inq_last_6mths                 1063550
sec_app_earliest_cr_line               1063550
sec_app_open_acc                       1063550
verification_status_joint              1057964
dti_joint                              1057800
annual_inc_joint                       1057797
desc                                    977658
mths_since_last_record                  895269
dtype: int64

Features with identical miss

,sec_app_mths_since_last_major_derog,sec_app_revol_util,sec_app_open_acc,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_fico_range_low,sec_app_mort_acc,revol_bal_joint,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_fico_range_high,verification_status_joint,dti_joint,annual_inc_joint,desc,mths_since_last_record
sec_app_mths_since_last_major_derog,1.000000,0.580089,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.593489,0.500488,0.502905,0.502868,-0.022580,-0.006330
sec_app_revol_util,0.580089,1.000000,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.991235,0.833633,0.839944,0.839882,-0.037713,-0.001779
sec_app_open_acc,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_earliest_cr_line,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_inq_last_6mths,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_fico_range_low,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_mort_acc,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
revol_bal_joint,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_open_act_il,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010
sec_app_num_rev_accts,0.593489,0.991235,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.841115,0.847371,0.847308,-0.038047,-0.002010


In [180]:
# Cell 13 — Structural missingness group audit

print("STRUCTURAL MISSINGNESS GROUP AUDIT")
print("=" * 60)

# Group 1: Secondary applicant / joint-account information
secondary_app_columns = [
    "sec_app_open_acc",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_fico_range_low",
    "sec_app_mort_acc",
    "revol_bal_joint",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_fico_range_high"
]

# Group 2: Joint application information
joint_application_columns = [
    "verification_status_joint",
    "dti_joint",
    "annual_inc_joint"
]

# Other extreme-missingness fields
other_extreme_columns = [
    "sec_app_mths_since_last_major_derog",
    "sec_app_revol_util",
    "desc",
    "mths_since_last_record"
]

def audit_group(columns, group_name):

    available_columns = [
        column
        for column in columns
        if column in X_train.columns
    ]

    if len(available_columns) == 0:
        print(f"\n{group_name}")
        print("-" * 60)
        print("No columns found.")
        return

    group_present = X_train[available_columns].notna().any(axis=1)

    group_complete = X_train[available_columns].notna().all(axis=1)

    print(f"\n{group_name}")
    print("-" * 60)

    print("Columns:", len(available_columns))
    print("Rows with at least one value:", int(group_present.sum()))
    print("Rows with all values missing:", int((~group_present).sum()))
    print("Rows with all values present:", int(group_complete.sum()))

    print(
        "Percentage with at least one value:",
        round(group_present.mean() * 100, 4),
        "%"
    )

    print(
        "Percentage completely missing:",
        round((~group_present).mean() * 100, 4),
        "%"
    )


audit_group(
    secondary_app_columns,
    "GROUP 1 — Secondary Applicant / Joint Account"
)

audit_group(
    joint_application_columns,
    "GROUP 2 — Joint Application"
)

audit_group(
    other_extreme_columns,
    "GROUP 3 — Other Extreme-Missingness Fields"
)

STRUCTURAL MISSINGNESS GROUP AUDIT

GROUP 1 — Secondary Applicant / Joint Account
------------------------------------------------------------
Columns: 11
Rows with at least one value: 14929
Rows with all values missing: 1063550
Rows with all values present: 14929
Percentage with at least one value: 1.3843 %
Percentage completely missing: 98.6157 %

GROUP 2 — Joint Application
------------------------------------------------------------
Columns: 3
Rows with at least one value: 20682
Rows with all values missing: 1057797
Rows with all values present: 20512
Percentage with at least one value: 1.9177 %
Percentage completely missing: 98.0823 %

GROUP 3 — Other Extreme-Missingness Fields
------------------------------------------------------------
Columns: 4
Rows with at least one value: 286191
Rows with all values missing: 792288
Rows with all values present: 0
Percentage with at least one value: 26.5365 %
Percentage completely missing: 73.4635 %


In [181]:
# Cell 14 — Define treatment for extreme-missingness features

print("EXTREME-MISSINGNESS TREATMENT PLAN")
print("=" * 60)

# Fields belonging to secondary-applicant information.
# Their values exist only for a small fraction of observations.
drop_structural_columns = [
    "sec_app_mths_since_last_major_derog",
    "sec_app_revol_util",
    "sec_app_open_acc",
    "sec_app_earliest_cr_line",
    "sec_app_inq_last_6mths",
    "sec_app_fico_range_low",
    "sec_app_mort_acc",
    "revol_bal_joint",
    "sec_app_open_act_il",
    "sec_app_num_rev_accts",
    "sec_app_chargeoff_within_12_mths",
    "sec_app_collections_12_mths_ex_med",
    "sec_app_fico_range_high"
]

# Joint-application fields are also structurally sparse.
drop_structural_columns.extend([
    "verification_status_joint",
    "dti_joint",
    "annual_inc_joint"
])

# Description is free text and has over 90% missingness.
# It requires separate NLP handling rather than ordinary imputation.
drop_special_handling_columns = [
    "desc"
]

# This is a legitimate credit-history feature.
# Missingness itself carries information, so retain it.
retain_with_missing_indicator = [
    "mths_since_last_record"
]

print("\nDROP — STRUCTURAL SECONDARY/JOIN APPLICANT FIELDS:")
for column in drop_structural_columns:
    print(" -", column)

print("\nDROP — SPECIAL HANDLING:")
for column in drop_special_handling_columns:
    print(" -", column)

print("\nRETAIN + MISSINGNESS INDICATOR:")
for column in retain_with_missing_indicator:
    print(" -", column)

print("\nTotal extreme columns reviewed:")
print(len(extreme_columns))

print(
    "\nTotal planned for removal:",
    len(drop_structural_columns) +
    len(drop_special_handling_columns)
)

print(
    "Total planned for retention:",
    len(retain_with_missing_indicator)
)

EXTREME-MISSINGNESS TREATMENT PLAN

DROP — STRUCTURAL SECONDARY/JOIN APPLICANT FIELDS:
 - sec_app_mths_since_last_major_derog
 - sec_app_revol_util
 - sec_app_open_acc
 - sec_app_earliest_cr_line
 - sec_app_inq_last_6mths
 - sec_app_fico_range_low
 - sec_app_mort_acc
 - revol_bal_joint
 - sec_app_open_act_il
 - sec_app_num_rev_accts
 - sec_app_chargeoff_within_12_mths
 - sec_app_collections_12_mths_ex_med
 - sec_app_fico_range_high
 - verification_status_joint
 - dti_joint
 - annual_inc_joint

DROP — SPECIAL HANDLING:
 - desc

RETAIN + MISSINGNESS INDICATOR:
 - mths_since_last_record

Total extreme columns reviewed:
18

Total planned for removal: 17
Total planned for retention: 1


In [182]:
# Cell 15 — Apply structural missingness decisions

print("APPLYING EXTREME-MISSINGNESS TREATMENT")
print("=" * 60)

# Combine all approved removal columns
step11_drop_columns = (
    drop_structural_columns
    + drop_special_handling_columns
)

print("Columns scheduled for removal:", len(step11_drop_columns))

# Verify columns exist in every split
missing_from_train = [
    column
    for column in step11_drop_columns
    if column not in X_train.columns
]

missing_from_validation = [
    column
    for column in step11_drop_columns
    if column not in X_val.columns
]

missing_from_test = [
    column
    for column in step11_drop_columns
    if column not in X_test.columns
]

if missing_from_train:
    raise ValueError(
        f"Columns missing from X_train: {missing_from_train}"
    )

if missing_from_validation:
    raise ValueError(
        f"Columns missing from X_val: {missing_from_validation}"
    )

if missing_from_test:
    raise ValueError(
        f"Columns missing from X_test: {missing_from_test}"
    )

# Remove the exact same columns from all three splits
X_train = X_train.drop(
    columns=step11_drop_columns
)

X_val = X_val.drop(
    columns=step11_drop_columns
)

X_test = X_test.drop(
    columns=step11_drop_columns
)

print("\nRemoved columns:", len(step11_drop_columns))

print("\nRemaining shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

# Check which extreme-missingness columns remain
remaining_extreme = [
    column
    for column in extreme_columns
    if column in X_train.columns
]

print("\nRemaining extreme-missingness columns:")

if remaining_extreme:
    for column in remaining_extreme:
        print(" -", column)
else:
    print("None")

print("\nStructural missingness removal completed.")

APPLYING EXTREME-MISSINGNESS TREATMENT
Columns scheduled for removal: 17

Removed columns: 17

Remaining shapes:
X_train: (1078479, 86)
X_val: (134810, 86)
X_test: (134810, 86)

Remaining extreme-missingness columns:
 - mths_since_last_record

Structural missingness removal completed.


In [183]:
# Cell 16 — Create missingness indicator

print("MISSINGNESS INDICATOR")
print("=" * 60)

indicator_source = "mths_since_last_record"
indicator_column = "mths_since_last_record_missing"

if indicator_source not in X_train.columns:
    raise ValueError(
        f"{indicator_source} is missing from X_train."
    )

# Create indicator BEFORE imputation
X_train[indicator_column] = (
    X_train[indicator_source]
    .isna()
    .astype("int8")
)

X_val[indicator_column] = (
    X_val[indicator_source]
    .isna()
    .astype("int8")
)

X_test[indicator_column] = (
    X_test[indicator_source]
    .isna()
    .astype("int8")
)

print("Source column:", indicator_source)
print("Indicator column:", indicator_column)

print("\nMissing-value indicator counts:")

print(
    "X_train:",
    X_train[indicator_column].value_counts().sort_index().to_dict()
)

print(
    "X_val:",
    X_val[indicator_column].value_counts().sort_index().to_dict()
)

print(
    "X_test:",
    X_test[indicator_column].value_counts().sort_index().to_dict()
)

print("\nUpdated shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nMissingness indicator created successfully.")

MISSINGNESS INDICATOR
Source column: mths_since_last_record
Indicator column: mths_since_last_record_missing

Missing-value indicator counts:
X_train: {0: 183210, 1: 895269}
X_val: {0: 23200, 1: 111610}
X_test: {0: 23014, 1: 111796}

Updated shapes:
X_train: (1078479, 87)
X_val: (134810, 87)
X_test: (134810, 87)

Missingness indicator created successfully.


In [184]:
# Cell 17 — Post-structural missingness audit

print("POST-STRUCTURAL MISSINGNESS AUDIT")
print("=" * 60)

def missingness_audit(dataframe, dataset_name):
    missing_count = dataframe.isna().sum()
    missing_count = missing_count[missing_count > 0]

    audit = pd.DataFrame({
        "column": missing_count.index,
        "missing_count": missing_count.values,
        "missing_percentage": (
            missing_count.values
            / len(dataframe)
            * 100
        )
    })

    audit = audit.sort_values(
        "missing_percentage",
        ascending=False
    ).reset_index(drop=True)

    print(f"\n{dataset_name}")
    print("-" * 60)
    print("Rows:", len(dataframe))
    print("Columns:", len(dataframe.columns))
    print("Features with missing values:", len(audit))

    if len(audit) > 0:
        print("\nMissing-value summary:")
        display(audit)
    else:
        print("No missing values.")

    return audit


train_missing_audit = missingness_audit(
    X_train,
    "TRAINING SET"
)

val_missing_audit = missingness_audit(
    X_val,
    "VALIDATION SET"
)

test_missing_audit = missingness_audit(
    X_test,
    "TEST SET"
)

print("\nPost-structural missingness audit completed.")

POST-STRUCTURAL MISSINGNESS AUDIT

TRAINING SET
------------------------------------------------------------
Rows: 1078479
Columns: 87
Features with missing values: 72

Missing-value summary:


,column,missing_count,missing_percentage
0,mths_since_last_record,895269,83.012187
1,mths_since_recent_bc_dlq,823106,76.321004
2,mths_since_last_major_derog,795227,73.735974
3,mths_since_recent_revol_delinq,718368,66.609364
4,il_util,706217,65.482684
...,...,...,...
67,open_acc,24,0.002225
68,delinq_amnt,24,0.002225
69,delinq_2yrs,24,0.002225
70,annual_inc,4,0.000371



VALIDATION SET
------------------------------------------------------------
Rows: 134810
Columns: 87
Features with missing values: 62

Missing-value summary:


,column,missing_count,missing_percentage
0,mths_since_last_record,111610,82.790594
1,mths_since_recent_bc_dlq,103148,76.513612
2,mths_since_last_major_derog,99574,73.862473
3,mths_since_recent_revol_delinq,89850,66.649358
4,il_util,88306,65.504043
...,...,...,...
57,revol_util,89,0.066019
58,dti,37,0.027446
59,collections_12_mths_ex_med,12,0.008901
60,chargeoff_within_12_mths,12,0.008901



TEST SET
------------------------------------------------------------
Rows: 134810
Columns: 87
Features with missing values: 70

Missing-value summary:


,column,missing_count,missing_percentage
0,mths_since_last_record,111796,82.928566
1,mths_since_recent_bc_dlq,102816,76.267339
2,mths_since_last_major_derog,99541,73.837994
3,mths_since_recent_revol_delinq,89905,66.690157
4,il_util,88529,65.669461
...,...,...,...
65,pub_rec,5,0.003709
66,delinq_amnt,5,0.003709
67,total_acc,5,0.003709
68,acc_now_delinq,5,0.003709



Post-structural missingness audit completed.


In [185]:
# Cell 18 — Feature data-type audit

print("FEATURE DATA-TYPE AUDIT")
print("=" * 60)

# Identify numeric and categorical features from TRAINING data only
numeric_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numeric features:", len(numeric_columns))
print("Categorical / object features:", len(categorical_columns))

print("\nNUMERIC FEATURES")
print("-" * 60)

for column in numeric_columns:
    print(
        f"- {column} | "
        f"dtype={X_train[column].dtype} | "
        f"missing={X_train[column].isna().sum()}"
    )

print("\nCATEGORICAL / OBJECT FEATURES")
print("-" * 60)

for column in categorical_columns:
    print(
        f"- {column} | "
        f"dtype={X_train[column].dtype} | "
        f"unique={X_train[column].nunique(dropna=True)} | "
        f"missing={X_train[column].isna().sum()}"
    )

print("\nDATA-TYPE AUDIT COMPLETE")

FEATURE DATA-TYPE AUDIT
Numeric features: 74
Categorical / object features: 13

NUMERIC FEATURES
------------------------------------------------------------
- loan_amnt | dtype=float64 | missing=0
- int_rate | dtype=float64 | missing=0
- installment | dtype=float64 | missing=0
- annual_inc | dtype=float64 | missing=4
- dti | dtype=float64 | missing=297
- delinq_2yrs | dtype=float64 | missing=24
- fico_range_low | dtype=float64 | missing=0
- fico_range_high | dtype=float64 | missing=0
- inq_last_6mths | dtype=float64 | missing=25
- mths_since_last_delinq | dtype=float64 | missing=543516
- mths_since_last_record | dtype=float64 | missing=895269
- open_acc | dtype=float64 | missing=24
- pub_rec | dtype=float64 | missing=24
- revol_bal | dtype=float64 | missing=0
- revol_util | dtype=float64 | missing=711
- total_acc | dtype=float64 | missing=24
- collections_12_mths_ex_med | dtype=float64 | missing=116
- mths_since_last_major_derog | dtype=float64 | missing=795227
- acc_now_delinq | dtyp

In [186]:
# Cell 19 — Remaining missingness treatment audit

print("REMAINING MISSINGNESS TREATMENT AUDIT")
print("=" * 60)

def build_missingness_audit(dataframe, columns):
    rows = []

    for column in columns:
        missing_count = int(dataframe[column].isna().sum())

        if missing_count > 0:
            missing_percentage = (
                missing_count / len(dataframe) * 100
            )

            if missing_percentage >= 80:
                severity = "Extreme (80%+)"
            elif missing_percentage >= 50:
                severity = "Very high (50-80%)"
            elif missing_percentage >= 20:
                severity = "Moderate (20-50%)"
            elif missing_percentage >= 5:
                severity = "Low (5-20%)"
            else:
                severity = "Very low (<5%)"

            rows.append({
                "column": column,
                "dtype": str(dataframe[column].dtype),
                "missing_count": missing_count,
                "missing_percentage": missing_percentage,
                "severity": severity
            })

    if len(rows) == 0:
        return pd.DataFrame(
            columns=[
                "column",
                "dtype",
                "missing_count",
                "missing_percentage",
                "severity"
            ]
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "missing_percentage",
            ascending=False
        )
        .reset_index(drop=True)
    )


remaining_missing_audit = build_missingness_audit(
    X_train,
    X_train.columns
)

print(
    "Features with remaining missing values:",
    len(remaining_missing_audit)
)

print("\nRemaining missingness:")
display(remaining_missing_audit)


print("\nMISSINGNESS BY SEVERITY")
print("-" * 60)

severity_counts = (
    remaining_missing_audit["severity"]
    .value_counts()
)

print(severity_counts)


print("\nNUMERIC FEATURES WITH >20% MISSINGNESS")
print("-" * 60)

numeric_high_missing = remaining_missing_audit[
    (
        remaining_missing_audit["dtype"].str.contains(
            "float|int"
        )
    )
    &
    (
        remaining_missing_audit["missing_percentage"] >= 20
    )
]

display(numeric_high_missing)


print("\nCATEGORICAL FEATURES WITH MISSING VALUES")
print("-" * 60)

categorical_missing = remaining_missing_audit[
    ~remaining_missing_audit["dtype"].str.contains(
        "float|int"
    )
]

display(categorical_missing)

print("\nTreatment audit completed.")

REMAINING MISSINGNESS TREATMENT AUDIT
Features with remaining missing values: 72

Remaining missingness:


,column,dtype,missing_count,missing_percentage,severity
0,mths_since_last_record,float64,895269,83.012187,Extreme (80%+)
1,mths_since_recent_bc_dlq,float64,823106,76.321004,Very high (50-80%)
2,mths_since_last_major_derog,float64,795227,73.735974,Very high (50-80%)
3,mths_since_recent_revol_delinq,float64,718368,66.609364,Very high (50-80%)
4,il_util,float64,706217,65.482684,Very high (50-80%)
...,...,...,...,...,...
67,open_acc,float64,24,0.002225,Very low (<5%)
68,delinq_amnt,float64,24,0.002225,Very low (<5%)
69,delinq_2yrs,float64,24,0.002225,Very low (<5%)
70,annual_inc,float64,4,0.000371,Very low (<5%)



MISSINGNESS BY SEVERITY
------------------------------------------------------------
severity
Very low (<5%)        27
Low (5-20%)           26
Very high (50-80%)    18
Extreme (80%+)         1
Name: count, dtype: int64

NUMERIC FEATURES WITH >20% MISSINGNESS
------------------------------------------------------------


,column,dtype,missing_count,missing_percentage,severity
0,mths_since_last_record,float64,895269,83.012187,Extreme (80%+)
1,mths_since_recent_bc_dlq,float64,823106,76.321004,Very high (50-80%)
2,mths_since_last_major_derog,float64,795227,73.735974,Very high (50-80%)
3,mths_since_recent_revol_delinq,float64,718368,66.609364,Very high (50-80%)
4,il_util,float64,706217,65.482684,Very high (50-80%)
5,mths_since_rcnt_il,float64,659763,61.175322,Very high (50-80%)
6,all_util,float64,648371,60.119019,Very high (50-80%)
7,open_acc_6m,float64,648332,60.115403,Very high (50-80%)
8,inq_last_12m,float64,648332,60.115403,Very high (50-80%)
9,total_cu_tl,float64,648332,60.115403,Very high (50-80%)



CATEGORICAL FEATURES WITH MISSING VALUES
------------------------------------------------------------


,column,dtype,missing_count,missing_percentage,severity
22,emp_title,object,68874,6.386216,Low (5-20%)
23,emp_length,object,62973,5.839057,Low (5-20%)
55,title,object,13277,1.231086,Very low (<5%)
64,earliest_cr_line,object,24,0.002225,Very low (<5%)
71,zip_code,object,1,0.000093,Very low (<5%)



Treatment audit completed.


In [187]:
# Cell 20 — Final missing-value treatment plan

print("FINAL MISSING-VALUE TREATMENT PLAN")
print("=" * 60)

# ---------------------------------------------------------
# 1. Numeric features with missing values
# ---------------------------------------------------------

numeric_missing_columns = [
    column
    for column in numeric_columns
    if X_train[column].isna().sum() > 0
]

# Create indicators for numeric features where missingness exists.
# This preserves information contained in the fact that a value
# was unavailable.
numeric_indicator_columns = [
    f"{column}_missing"
    for column in numeric_missing_columns
    if column != "mths_since_last_record"
]

# mths_since_last_record already has its indicator.
if (
    "mths_since_last_record" in numeric_missing_columns
    and "mths_since_last_record_missing"
    not in numeric_indicator_columns
):
    numeric_indicator_columns.append(
        "mths_since_last_record_missing"
    )


# ---------------------------------------------------------
# 2. Categorical features with missing values
# ---------------------------------------------------------

categorical_missing_columns = [
    column
    for column in categorical_columns
    if X_train[column].isna().sum() > 0
]


# ---------------------------------------------------------
# 3. Treatment summary
# ---------------------------------------------------------

print("\nNUMERIC FEATURES WITH MISSING VALUES")
print("-" * 60)

print(
    "Count:",
    len(numeric_missing_columns)
)

for column in numeric_missing_columns:
    percentage = (
        X_train[column].isna().sum()
        / len(X_train)
        * 100
    )

    print(
        f"- {column}: "
        f"{percentage:.4f}% missing"
    )


print("\nCATEGORICAL FEATURES WITH MISSING VALUES")
print("-" * 60)

print(
    "Count:",
    len(categorical_missing_columns)
)

for column in categorical_missing_columns:
    percentage = (
        X_train[column].isna().sum()
        / len(X_train)
        * 100
    )

    print(
        f"- {column}: "
        f"{percentage:.4f}% missing"
    )


print("\nTREATMENT RULES")
print("-" * 60)

print(
    "1. Numeric missing values:"
    "\n   → Create missingness indicator"
    "\n   → Median imputation learned from X_train only"
)

print(
    "\n2. Categorical missing values:"
    "\n   → Replace missing values with 'Missing'"
)

print(
    "\n3. mths_since_last_record:"
    "\n   → Existing missingness indicator retained"
    "\n   → Median imputation learned from X_train only"
)

print(
    "\n4. No imputation statistics will be calculated"
    " independently on validation or test data."
)

print(
    "\n5. No feature will be dropped solely because"
    " of a high missingness percentage at this stage."
)

print("\nTreatment plan established successfully.")

FINAL MISSING-VALUE TREATMENT PLAN

NUMERIC FEATURES WITH MISSING VALUES
------------------------------------------------------------
Count: 67
- annual_inc: 0.0004% missing
- dti: 0.0275% missing
- delinq_2yrs: 0.0022% missing
- inq_last_6mths: 0.0023% missing
- mths_since_last_delinq: 50.3965% missing
- mths_since_last_record: 83.0122% missing
- open_acc: 0.0022% missing
- pub_rec: 0.0022% missing
- revol_util: 0.0659% missing
- total_acc: 0.0022% missing
- collections_12_mths_ex_med: 0.0108% missing
- mths_since_last_major_derog: 73.7360% missing
- acc_now_delinq: 0.0022% missing
- tot_coll_amt: 5.2284% missing
- tot_cur_bal: 5.2284% missing
- open_acc_6m: 60.1154% missing
- open_act_il: 60.1153% missing
- open_il_12m: 60.1153% missing
- open_il_24m: 60.1153% missing
- mths_since_rcnt_il: 61.1753% missing
- total_bal_il: 60.1153% missing
- il_util: 65.4827% missing
- open_rv_12m: 60.1153% missing
- open_rv_24m: 60.1153% missing
- max_bal_bc: 60.1153% missing
- all_util: 60.1190% mis

In [188]:
# Cell 21 — Create missingness indicators

print("CREATING MISSINGNESS INDICATORS")
print("=" * 60)

# Numeric features that still contain missing values
numeric_missing_columns = [
    column
    for column in numeric_columns
    if column in X_train.columns
    and X_train[column].isna().sum() > 0
]

# Do not create a second indicator for this column.
# Its indicator already exists from Cell 16.
existing_indicator_source = "mths_since_last_record"
existing_indicator = "mths_since_last_record_missing"

created_indicators = []

for column in numeric_missing_columns:

    indicator_column = f"{column}_missing"

    if column == existing_indicator_source:
        print(
            f"⏭️ Existing indicator retained: "
            f"{existing_indicator}"
        )
        continue

    # Create indicator BEFORE imputation
    X_train[indicator_column] = (
        X_train[column]
        .isna()
        .astype("int8")
    )

    X_val[indicator_column] = (
        X_val[column]
        .isna()
        .astype("int8")
    )

    X_test[indicator_column] = (
        X_test[column]
        .isna()
        .astype("int8")
    )

    created_indicators.append(indicator_column)


print("\nNew indicators created:", len(created_indicators))

print("\nExisting indicator:")
print(
    f"- {existing_indicator}"
)

print("\nTotal missingness indicators:")
print(
    len(created_indicators) + 1
)

print("\nUpdated shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nMissingness indicators created successfully.")

CREATING MISSINGNESS INDICATORS
⏭️ Existing indicator retained: mths_since_last_record_missing

New indicators created: 66

Existing indicator:
- mths_since_last_record_missing

Total missingness indicators:
67

Updated shapes:
X_train: (1078479, 153)
X_val: (134810, 153)
X_test: (134810, 153)

Missingness indicators created successfully.


In [189]:
# Cell 22 — Calculate training-set medians

print("CALCULATING TRAINING-SET MEDIANS")
print("=" * 60)

# Numeric features that still contain missing values
numeric_imputation_columns = [
    column
    for column in numeric_columns
    if column in X_train.columns
    and X_train[column].isna().sum() > 0
]

# Calculate medians ONLY from X_train
training_medians = {}

for column in numeric_imputation_columns:

    median_value = X_train[column].median()

    training_medians[column] = median_value

print(
    "Numeric features requiring median imputation:",
    len(training_medians)
)

print("\nTraining-set medians:")

for column, median_value in training_medians.items():
    print(
        f"- {column}: {median_value}"
    )

print("\nMedian source:")
print("✅ X_train only")

print("\nValidation and test sets were NOT used to calculate medians.")

print("\nTraining-set median calculation completed successfully.")

CALCULATING TRAINING-SET MEDIANS
Numeric features requiring median imputation: 67

Training-set medians:
- annual_inc: 65000.0
- dti: 17.6
- delinq_2yrs: 0.0
- inq_last_6mths: 0.0
- mths_since_last_delinq: 31.0
- mths_since_last_record: 71.0
- open_acc: 11.0
- pub_rec: 0.0
- revol_util: 52.2
- total_acc: 23.0
- collections_12_mths_ex_med: 0.0
- mths_since_last_major_derog: 44.0
- acc_now_delinq: 0.0
- tot_coll_amt: 0.0
- tot_cur_bal: 80289.0
- open_acc_6m: 1.0
- open_act_il: 2.0
- open_il_12m: 1.0
- open_il_24m: 1.0
- mths_since_rcnt_il: 12.0
- total_bal_il: 24061.0
- il_util: 75.0
- open_rv_12m: 1.0
- open_rv_24m: 2.0
- max_bal_bc: 4202.0
- all_util: 59.0
- total_rev_hi_lim: 24100.0
- inq_fi: 1.0
- total_cu_tl: 0.0
- inq_last_12m: 2.0
- acc_open_past_24mths: 4.0
- avg_cur_bal: 7408.0
- bc_open_to_buy: 4693.0
- bc_util: 63.2
- chargeoff_within_12_mths: 0.0
- delinq_amnt: 0.0
- mo_sin_old_il_acct: 129.0
- mo_sin_old_rev_tl_op: 164.0
- mo_sin_rcnt_rev_tl_op: 8.0
- mo_sin_rcnt_tl: 5.0
- m

In [190]:
# Cell 23 — Apply training-set medians

print("APPLYING TRAINING-SET MEDIANS")
print("=" * 60)

imputation_results = []

for column, median_value in training_medians.items():

    train_missing_before = X_train[column].isna().sum()
    val_missing_before = X_val[column].isna().sum()
    test_missing_before = X_test[column].isna().sum()

    # Apply the X_train-derived median to all three datasets
    X_train[column] = X_train[column].fillna(median_value)
    X_val[column] = X_val[column].fillna(median_value)
    X_test[column] = X_test[column].fillna(median_value)

    train_missing_after = X_train[column].isna().sum()
    val_missing_after = X_val[column].isna().sum()
    test_missing_after = X_test[column].isna().sum()

    imputation_results.append({
        "column": column,
        "median": median_value,
        "train_missing_before": int(train_missing_before),
        "train_missing_after": int(train_missing_after),
        "val_missing_before": int(val_missing_before),
        "val_missing_after": int(val_missing_after),
        "test_missing_before": int(test_missing_before),
        "test_missing_after": int(test_missing_after)
    })


imputation_results = pd.DataFrame(imputation_results)


print(
    "Numeric features imputed:",
    len(imputation_results)
)

print("\nImputation summary:")
display(imputation_results)


print("\nRemaining numeric missing values:")

print(
    "X_train:",
    X_train[numeric_imputation_columns].isna().sum().sum()
)

print(
    "X_val:",
    X_val[numeric_imputation_columns].isna().sum().sum()
)

print(
    "X_test:",
    X_test[numeric_imputation_columns].isna().sum().sum()
)


print("\nMedian imputation completed successfully.")
print("✅ All imputation values came from X_train.")

APPLYING TRAINING-SET MEDIANS
Numeric features imputed: 67

Imputation summary:


,column,median,train_missing_before,train_missing_after,val_missing_before,val_missing_after,test_missing_before,test_missing_after
0,annual_inc,65000.0,4,0,0,0,0,0
1,dti,17.6,297,0,37,0,40,0
2,delinq_2yrs,0.0,24,0,0,0,5,0
3,inq_last_6mths,0.0,25,0,0,0,5,0
4,mths_since_last_delinq,31.0,543516,0,68166,0,68278,0
...,...,...,...,...,...,...,...,...
62,tax_liens,0.0,81,0,10,0,14,0
63,tot_hi_cred_lim,112407.0,56387,0,6899,0,6990,0
64,total_bal_ex_mort,37294.0,40131,0,4906,0,4993,0
65,total_bc_limit,15100.0,40131,0,4906,0,4993,0



Remaining numeric missing values:
X_train: 0
X_val: 0
X_test: 0

Median imputation completed successfully.
✅ All imputation values came from X_train.


In [191]:
# Cell 24 — Categorical missing-value treatment

print("CATEGORICAL MISSING-VALUE TREATMENT")
print("=" * 60)

categorical_missing_columns = [
    column
    for column in categorical_columns
    if column in X_train.columns
    and X_train[column].isna().sum() > 0
]

print(
    "Categorical features requiring treatment:",
    len(categorical_missing_columns)
)

print("\nColumns:")
for column in categorical_missing_columns:
    print(f"- {column}")


categorical_imputation_value = "Missing"

categorical_imputation_results = []


for column in categorical_missing_columns:

    train_missing_before = X_train[column].isna().sum()
    val_missing_before = X_val[column].isna().sum()
    test_missing_before = X_test[column].isna().sum()

    # Ensure object/string-compatible dtype
    X_train[column] = X_train[column].astype("object")
    X_val[column] = X_val[column].astype("object")
    X_test[column] = X_test[column].astype("object")

    # Apply the same explicit category to all splits
    X_train[column] = X_train[column].fillna(
        categorical_imputation_value
    )

    X_val[column] = X_val[column].fillna(
        categorical_imputation_value
    )

    X_test[column] = X_test[column].fillna(
        categorical_imputation_value
    )

    train_missing_after = X_train[column].isna().sum()
    val_missing_after = X_val[column].isna().sum()
    test_missing_after = X_test[column].isna().sum()

    categorical_imputation_results.append({
        "column": column,
        "replacement": categorical_imputation_value,
        "train_missing_before": int(train_missing_before),
        "train_missing_after": int(train_missing_after),
        "val_missing_before": int(val_missing_before),
        "val_missing_after": int(val_missing_after),
        "test_missing_before": int(test_missing_before),
        "test_missing_after": int(test_missing_after)
    })


categorical_imputation_results = pd.DataFrame(
    categorical_imputation_results
)


print("\nCategorical imputation summary:")
display(categorical_imputation_results)


print("\nRemaining categorical missing values:")

print(
    "X_train:",
    X_train[categorical_missing_columns].isna().sum().sum()
)

print(
    "X_val:",
    X_val[categorical_missing_columns].isna().sum().sum()
)

print(
    "X_test:",
    X_test[categorical_missing_columns].isna().sum().sum()
)


print("\nReplacement value:")
print(f"'{categorical_imputation_value}'")

print("\nCategorical missing-value treatment completed successfully.")

CATEGORICAL MISSING-VALUE TREATMENT
Categorical features requiring treatment: 5

Columns:
- emp_title
- emp_length
- title
- zip_code
- earliest_cr_line

Categorical imputation summary:


,column,replacement,train_missing_before,train_missing_after,val_missing_before,val_missing_after,test_missing_before,test_missing_after
0,emp_title,Missing,68874,0,8553,0,8523,0
1,emp_length,Missing,62973,0,7826,0,7751,0
2,title,Missing,13277,0,1703,0,1682,0
3,zip_code,Missing,1,0,0,0,0,0
4,earliest_cr_line,Missing,24,0,0,0,5,0



Remaining categorical missing values:
X_train: 0
X_val: 0
X_test: 0

Replacement value:
'Missing'

Categorical missing-value treatment completed successfully.


In [192]:
# Cell 25 — Complete missing-value audit

print("STEP 11 — POST-IMPUTATION MISSING-VALUE AUDIT")
print("=" * 60)


def missing_value_audit(dataframe, dataset_name):

    missing_counts = dataframe.isna().sum()

    missing_counts = (
        missing_counts[
            missing_counts > 0
        ]
        .sort_values(ascending=False)
    )

    print(f"\n{dataset_name}")
    print("-" * 60)

    print("Rows:", dataframe.shape[0])
    print("Columns:", dataframe.shape[1])

    print(
        "Total missing values:",
        int(missing_counts.sum())
    )

    print(
        "Features containing missing values:",
        len(missing_counts)
    )

    if len(missing_counts) > 0:

        audit = pd.DataFrame({
            "column": missing_counts.index,
            "missing_count": missing_counts.values,
            "missing_percentage": (
                missing_counts.values
                / len(dataframe)
                * 100
            )
        })

        display(audit)

        return audit

    else:

        print("✅ No missing values.")

        return pd.DataFrame(
            columns=[
                "column",
                "missing_count",
                "missing_percentage"
            ]
        )


train_missing_audit = missing_value_audit(
    X_train,
    "TRAINING SET"
)

val_missing_audit = missing_value_audit(
    X_val,
    "VALIDATION SET"
)

test_missing_audit = missing_value_audit(
    X_test,
    "TEST SET"
)


print("\n" + "=" * 60)
print("POST-IMPUTATION AUDIT SUMMARY")
print("=" * 60)

print(
    "X_train missing values:",
    int(X_train.isna().sum().sum())
)

print(
    "X_val missing values:",
    int(X_val.isna().sum().sum())
)

print(
    "X_test missing values:",
    int(X_test.isna().sum().sum())
)


if (
    X_train.isna().sum().sum() == 0
    and X_val.isna().sum().sum() == 0
    and X_test.isna().sum().sum() == 0
):
    print(
        "\n🟢 ALL MISSING VALUES HAVE BEEN HANDLED."
    )
else:
    print(
        "\n🔴 MISSING VALUES STILL REMAIN."
    )

STEP 11 — POST-IMPUTATION MISSING-VALUE AUDIT

TRAINING SET
------------------------------------------------------------
Rows: 1078479
Columns: 153
Total missing values: 0
Features containing missing values: 0
✅ No missing values.

VALIDATION SET
------------------------------------------------------------
Rows: 134810
Columns: 153
Total missing values: 0
Features containing missing values: 0
✅ No missing values.

TEST SET
------------------------------------------------------------
Rows: 134810
Columns: 153
Total missing values: 0
Features containing missing values: 0
✅ No missing values.

POST-IMPUTATION AUDIT SUMMARY
X_train missing values: 0
X_val missing values: 0
X_test missing values: 0

🟢 ALL MISSING VALUES HAVE BEEN HANDLED.


In [193]:
# Cell 26 — Missingness indicator integrity audit

print("STEP 11 — MISSINGNESS INDICATOR INTEGRITY")
print("=" * 60)

# Collect all missingness indicator columns
indicator_columns = [
    column
    for column in X_train.columns
    if column.endswith("_missing")
]

print(
    "Total missingness indicators:",
    len(indicator_columns)
)

print("\nIndicator columns:")
for column in indicator_columns:
    print("-", column)


# ---------------------------------------------------------
# Verify indicators exist in all three datasets
# ---------------------------------------------------------

missing_indicators_train = [
    column
    for column in indicator_columns
    if column not in X_train.columns
]

missing_indicators_val = [
    column
    for column in indicator_columns
    if column not in X_val.columns
]

missing_indicators_test = [
    column
    for column in indicator_columns
    if column not in X_test.columns
]


print("\nIndicator presence check:")
print(
    "Missing from X_train:",
    len(missing_indicators_train)
)

print(
    "Missing from X_val:",
    len(missing_indicators_val)
)

print(
    "Missing from X_test:",
    len(missing_indicators_test)
)


# ---------------------------------------------------------
# Verify indicator values
# ---------------------------------------------------------

invalid_indicator_values = {}

for column in indicator_columns:

    combined_values = pd.concat(
        [
            X_train[column],
            X_val[column],
            X_test[column]
        ],
        ignore_index=True
    ).dropna()

    invalid_values = sorted(
        set(combined_values.unique())
        - {0, 1}
    )

    if len(invalid_values) > 0:
        invalid_indicator_values[column] = invalid_values


print("\nIndicator value validation:")

if len(invalid_indicator_values) == 0:
    print(
        "✅ Every indicator contains only 0 and 1."
    )
else:
    print(
        "❌ Invalid indicator values found:"
    )

    for column, values in invalid_indicator_values.items():
        print(
            f"- {column}: {values}"
        )


# ---------------------------------------------------------
# Indicator distribution
# ---------------------------------------------------------

print("\nINDICATOR DISTRIBUTIONS")
print("-" * 60)

indicator_summary = []

for column in indicator_columns:

    train_missing_flagged = int(
        X_train[column].sum()
    )

    val_missing_flagged = int(
        X_val[column].sum()
    )

    test_missing_flagged = int(
        X_test[column].sum()
    )

    indicator_summary.append({
        "indicator": column,
        "train_flagged": train_missing_flagged,
        "validation_flagged": val_missing_flagged,
        "test_flagged": test_missing_flagged
    })


indicator_summary = pd.DataFrame(
    indicator_summary
)

display(indicator_summary)


# ---------------------------------------------------------
# Final indicator check
# ---------------------------------------------------------

if (
    len(missing_indicators_train) == 0
    and len(missing_indicators_val) == 0
    and len(missing_indicators_test) == 0
    and len(invalid_indicator_values) == 0
):
    print(
        "\n🟢 MISSINGNESS INDICATOR INTEGRITY PASSED"
    )
else:
    print(
        "\n🔴 MISSINGNESS INDICATOR INTEGRITY NEEDS REVIEW"
    )

STEP 11 — MISSINGNESS INDICATOR INTEGRITY
Total missingness indicators: 67

Indicator columns:
- mths_since_last_record_missing
- annual_inc_missing
- dti_missing
- delinq_2yrs_missing
- inq_last_6mths_missing
- mths_since_last_delinq_missing
- open_acc_missing
- pub_rec_missing
- revol_util_missing
- total_acc_missing
- collections_12_mths_ex_med_missing
- mths_since_last_major_derog_missing
- acc_now_delinq_missing
- tot_coll_amt_missing
- tot_cur_bal_missing
- open_acc_6m_missing
- open_act_il_missing
- open_il_12m_missing
- open_il_24m_missing
- mths_since_rcnt_il_missing
- total_bal_il_missing
- il_util_missing
- open_rv_12m_missing
- open_rv_24m_missing
- max_bal_bc_missing
- all_util_missing
- total_rev_hi_lim_missing
- inq_fi_missing
- total_cu_tl_missing
- inq_last_12m_missing
- acc_open_past_24mths_missing
- avg_cur_bal_missing
- bc_open_to_buy_missing
- bc_util_missing
- chargeoff_within_12_mths_missing
- delinq_amnt_missing
- mo_sin_old_il_acct_missing
- mo_sin_old_rev_tl_o

,indicator,train_flagged,validation_flagged,test_flagged
0,mths_since_last_record_missing,895269,111610,111796
1,annual_inc_missing,4,0,0
2,dti_missing,297,37,40
3,delinq_2yrs_missing,24,0,5
4,inq_last_6mths_missing,25,0,5
...,...,...,...,...
62,tax_liens_missing,81,10,14
63,tot_hi_cred_lim_missing,56387,6899,6990
64,total_bal_ex_mort_missing,40131,4906,4993
65,total_bc_limit_missing,40131,4906,4993



🟢 MISSINGNESS INDICATOR INTEGRITY PASSED


In [194]:
# Cell 27 — Final Step 11 integrity audit

print("STEP 11 — FINAL MISSING-VALUE INTEGRITY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Missing-value check
# ------------------------------------------------------------

print("\nMISSING VALUES")
print("-" * 70)

train_missing = int(X_train.isna().sum().sum())
val_missing = int(X_val.isna().sum().sum())
test_missing = int(X_test.isna().sum().sum())

print("X_train missing values:", train_missing)
print("X_val missing values:", val_missing)
print("X_test missing values:", test_missing)

# ------------------------------------------------------------
# 2. Structural alignment
# ------------------------------------------------------------

print("\nSTRUCTURAL ALIGNMENT")
print("-" * 70)

same_train_val_columns = X_train.columns.equals(X_val.columns)
same_train_test_columns = X_train.columns.equals(X_test.columns)

print(
    "Train / Validation columns identical:",
    same_train_val_columns
)

print(
    "Train / Test columns identical:",
    same_train_test_columns
)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

# ------------------------------------------------------------
# 3. Missingness indicator audit
# ------------------------------------------------------------

print("\nMISSINGNESS INDICATORS")
print("-" * 70)

indicator_columns = [
    column
    for column in X_train.columns
    if column.endswith("_missing")
]

print("Total missingness indicators:", len(indicator_columns))

invalid_indicators = []

for column in indicator_columns:

    for dataset_name, dataset in [
        ("X_train", X_train),
        ("X_val", X_val),
        ("X_test", X_test)
    ]:

        unique_values = set(
            dataset[column].dropna().unique()
        )

        if not unique_values.issubset({0, 1}):
            invalid_indicators.append(
                (dataset_name, column, unique_values)
            )

print(
    "Indicators containing values other than 0/1:",
    len(invalid_indicators)
)

# ------------------------------------------------------------
# 4. Target leakage check
# ------------------------------------------------------------

print("\nTARGET LEAKAGE CHECK")
print("-" * 70)

target_in_train = "target" in X_train.columns
target_in_val = "target" in X_val.columns
target_in_test = "target" in X_test.columns

print("Target in X_train:", target_in_train)
print("Target in X_val:", target_in_val)
print("Target in X_test:", target_in_test)

# ------------------------------------------------------------
# 5. Final assertions
# ------------------------------------------------------------

step11_final_checks = {
    "X_train has no missing values": train_missing == 0,
    "X_val has no missing values": val_missing == 0,
    "X_test has no missing values": test_missing == 0,
    "Train and validation columns match": same_train_val_columns,
    "Train and test columns match": same_train_test_columns,
    "All indicators are binary": len(invalid_indicators) == 0,
    "Target absent from X_train": not target_in_train,
    "Target absent from X_val": not target_in_val,
    "Target absent from X_test": not target_in_test,
    "67 missingness indicators exist": len(indicator_columns) == 67
}

print("\nFINAL STEP 11 CHECKS")
print("-" * 70)

for check, result in step11_final_checks.items():
    print(
        "✅" if result else "❌",
        check
    )

print("\n" + "=" * 70)

if all(step11_final_checks.values()):
    print("🟢 STEP 11 — MISSING-VALUE HANDLING COMPLETE")
    print("🟢 ALL FINAL INTEGRITY CHECKS PASSED")
else:
    print("🔴 STEP 11 — REVIEW REQUIRED")

STEP 11 — FINAL MISSING-VALUE INTEGRITY AUDIT

MISSING VALUES
----------------------------------------------------------------------
X_train missing values: 0
X_val missing values: 0
X_test missing values: 0

STRUCTURAL ALIGNMENT
----------------------------------------------------------------------
Train / Validation columns identical: True
Train / Test columns identical: True
X_train shape: (1078479, 153)
X_val shape: (134810, 153)
X_test shape: (134810, 153)

MISSINGNESS INDICATORS
----------------------------------------------------------------------
Total missingness indicators: 67
Indicators containing values other than 0/1: 0

TARGET LEAKAGE CHECK
----------------------------------------------------------------------
Target in X_train: False
Target in X_val: False
Target in X_test: False

FINAL STEP 11 CHECKS
----------------------------------------------------------------------
✅ X_train has no missing values
✅ X_val has no missing values
✅ X_test has no missing values
✅ Train 

**STEP 12 — FINAL DATASET PREPARATION & CSV EXPORT**

In [195]:
# Cell 1 — Step 12 pre-export audit

print("STEP 12 — PRE-EXPORT AUDIT")
print("=" * 70)

print("TRAIN")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nVALIDATION")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTEST")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nMissing values:")
print("X_train:", X_train.isna().sum().sum())
print("X_val:", X_val.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

print("\nTarget presence:")
print("target in X_train:", "target" in X_train.columns)
print("target in X_val:", "target" in X_val.columns)
print("target in X_test:", "target" in X_test.columns)

print("\nColumn alignment:")
print("Train vs Validation:", X_train.columns.equals(X_val.columns))
print("Train vs Test:", X_train.columns.equals(X_test.columns))

STEP 12 — PRE-EXPORT AUDIT
TRAIN
X_train: (1078479, 153)
y_train: (1078479,)

VALIDATION
X_val: (134810, 153)
y_val: (134810,)

TEST
X_test: (134810, 153)
y_test: (134810,)

Missing values:
X_train: 0
X_val: 0
X_test: 0

Target presence:
target in X_train: False
target in X_val: False
target in X_test: False

Column alignment:
Train vs Validation: True
Train vs Test: True


In [196]:
# Cell 2 — Target alignment audit

print("TARGET ALIGNMENT AUDIT")
print("=" * 70)

print("Training:")
print("X_train rows:", len(X_train))
print("y_train rows:", len(y_train))
print("Aligned:", len(X_train) == len(y_train))

print("\nValidation:")
print("X_val rows:", len(X_val))
print("y_val rows:", len(y_val))
print("Aligned:", len(X_val) == len(y_val))

print("\nTest:")
print("X_test rows:", len(X_test))
print("y_test rows:", len(y_test))
print("Aligned:", len(X_test) == len(y_test))

if not (
    len(X_train) == len(y_train)
    and len(X_val) == len(y_val)
    and len(X_test) == len(y_test)
):
    raise ValueError("Feature-target row alignment failed.")

print("\n🟢 TARGET ALIGNMENT PASSED")

TARGET ALIGNMENT AUDIT
Training:
X_train rows: 1078479
y_train rows: 1078479
Aligned: True

Validation:
X_val rows: 134810
y_val rows: 134810
Aligned: True

Test:
X_test rows: 134810
y_test rows: 134810
Aligned: True

🟢 TARGET ALIGNMENT PASSED


In [197]:
# Cell 3 — Build final datasets

print("BUILDING FINAL DATASETS")
print("=" * 70)

train_final = X_train.copy()
train_final["target"] = y_train.to_numpy()

validation_final = X_val.copy()
validation_final["target"] = y_val.to_numpy()

test_final = X_test.copy()
test_final["target"] = y_test.to_numpy()

print("Final dataset shapes:")

print("train_final:", train_final.shape)
print("validation_final:", validation_final.shape)
print("test_final:", test_final.shape)

print("\nExpected columns:")
print("Features:", X_train.shape[1])
print("Target:", 1)
print("Total:", X_train.shape[1] + 1)

BUILDING FINAL DATASETS
Final dataset shapes:
train_final: (1078479, 154)
validation_final: (134810, 154)
test_final: (134810, 154)

Expected columns:
Features: 153
Target: 1
Total: 154


In [198]:
# Cell 4 — Final dataset validation

print("FINAL DATASET VALIDATION")
print("=" * 70)

datasets = {
    "train": train_final,
    "validation": validation_final,
    "test": test_final
}

for name, dataset in datasets.items():

    print(f"\n{name.upper()}")

    print("Rows:", dataset.shape[0])
    print("Columns:", dataset.shape[1])

    print(
        "Missing values:",
        dataset.isna().sum().sum()
    )

    print(
        "Duplicate rows:",
        dataset.duplicated().sum()
    )

    print(
        "Target present:",
        "target" in dataset.columns
    )

    numeric_data = dataset.select_dtypes(include=np.number)

    infinite_values = np.isinf(numeric_data).sum().sum()

    print(
        "Infinite numeric values:",
        int(infinite_values)
    )

    if dataset.isna().sum().sum() != 0:
        raise ValueError(f"{name} contains missing values.")

    if "target" not in dataset.columns:
        raise ValueError(f"{name} does not contain target.")

    if infinite_values != 0:
        raise ValueError(f"{name} contains infinite values.")

print("\n🟢 ALL FINAL DATASETS PASSED VALIDATION")

FINAL DATASET VALIDATION

TRAIN
Rows: 1078479
Columns: 154
Missing values: 0
Duplicate rows: 0
Target present: True
Infinite numeric values: 0

VALIDATION
Rows: 134810
Columns: 154
Missing values: 0
Duplicate rows: 0
Target present: True
Infinite numeric values: 0

TEST
Rows: 134810
Columns: 154
Missing values: 0
Duplicate rows: 0
Target present: True
Infinite numeric values: 0

🟢 ALL FINAL DATASETS PASSED VALIDATION


In [199]:
# Cell 5 — Create Step 12 output directory

from pathlib import Path

output_dir = Path("processed_data")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(output_dir.resolve())

print("\nDirectory ready.")

Output directory:
C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\processed_data

Directory ready.


In [200]:
# Cell 6 — Export final datasets to CSV

print("EXPORTING FINAL CSV FILES")
print("=" * 70)

train_path = output_dir / "train.csv"
validation_path = output_dir / "validation.csv"
test_path = output_dir / "test.csv"

print("Exporting train.csv...")

train_final.to_csv(
    train_path,
    index=False
)

print("✅ train.csv exported")

print("\nExporting validation.csv...")

validation_final.to_csv(
    validation_path,
    index=False
)

print("✅ validation.csv exported")

print("\nExporting test.csv...")

test_final.to_csv(
    test_path,
    index=False
)

print("✅ test.csv exported")

print("\n🟢 ALL THREE CSV FILES EXPORTED")

EXPORTING FINAL CSV FILES
Exporting train.csv...
✅ train.csv exported

Exporting validation.csv...
✅ validation.csv exported

Exporting test.csv...
✅ test.csv exported

🟢 ALL THREE CSV FILES EXPORTED


In [201]:
# Cell 7 — Verify exported files

print("EXPORTED FILES")
print("=" * 70)

for path in [
    train_path,
    validation_path,
    test_path
]:

    if path.exists():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"✅ {path.name} | "
            f"{size_mb:,.2f} MB"
        )

    else:

        print(
            f"❌ Missing file: {path.name}"
        )

EXPORTED FILES
✅ train.csv | 653.86 MB
✅ validation.csv | 81.73 MB
✅ test.csv | 81.74 MB


In [202]:
# Cell 8 — Reload exported datasets

print("RELOADING EXPORTED DATASETS")
print("=" * 70)

train_check = pd.read_csv(train_path)
validation_check = pd.read_csv(validation_path)
test_check = pd.read_csv(test_path)

print("Reloaded shapes:")

print(
    "train_check:",
    train_check.shape
)

print(
    "validation_check:",
    validation_check.shape
)

print(
    "test_check:",
    test_check.shape
)

RELOADING EXPORTED DATASETS
Reloaded shapes:
train_check: (1078479, 154)
validation_check: (134810, 154)
test_check: (134810, 154)


In [203]:
# Cell 9 — CSV integrity verification

print("CSV INTEGRITY VERIFICATION")
print("=" * 70)

expected_shapes = {
    "train": train_final.shape,
    "validation": validation_final.shape,
    "test": test_final.shape
}

actual_shapes = {
    "train": train_check.shape,
    "validation": validation_check.shape,
    "test": test_check.shape
}

for name in expected_shapes:

    print(f"\n{name.upper()}")

    print(
        "Expected shape:",
        expected_shapes[name]
    )

    print(
        "Actual shape:",
        actual_shapes[name]
    )

    if expected_shapes[name] != actual_shapes[name]:
        raise ValueError(
            f"{name} shape changed after CSV export."
        )

    print("Shape check: ✅")


print("\nColumn checks:")

print(
    "Train:",
    train_final.columns.equals(
        train_check.columns
    )
)

print(
    "Validation:",
    validation_final.columns.equals(
        validation_check.columns
    )
)

print(
    "Test:",
    test_final.columns.equals(
        test_check.columns
    )
)

CSV INTEGRITY VERIFICATION

TRAIN
Expected shape: (1078479, 154)
Actual shape: (1078479, 154)
Shape check: ✅

VALIDATION
Expected shape: (134810, 154)
Actual shape: (134810, 154)
Shape check: ✅

TEST
Expected shape: (134810, 154)
Actual shape: (134810, 154)
Shape check: ✅

Column checks:
Train: True
Validation: True
Test: True


In [204]:
# Cell 10 — Final Step 12 verification

print("STEP 12 — FINAL VERIFICATION")
print("=" * 70)

step12_checks = {

    "train.csv exists":
        train_path.exists(),

    "validation.csv exists":
        validation_path.exists(),

    "test.csv exists":
        test_path.exists(),

    "Train shape correct":
        train_check.shape == train_final.shape,

    "Validation shape correct":
        validation_check.shape == validation_final.shape,

    "Test shape correct":
        test_check.shape == test_final.shape,

    "Train columns preserved":
        train_check.columns.equals(
            train_final.columns
        ),

    "Validation columns preserved":
        validation_check.columns.equals(
            validation_final.columns
        ),

    "Test columns preserved":
        test_check.columns.equals(
            test_final.columns
        ),

    "Train has no missing values":
        train_check.isna().sum().sum() == 0,

    "Validation has no missing values":
        validation_check.isna().sum().sum() == 0,

    "Test has no missing values":
        test_check.isna().sum().sum() == 0,

    "Target exists in train":
        "target" in train_check.columns,

    "Target exists in validation":
        "target" in validation_check.columns,

    "Target exists in test":
        "target" in test_check.columns
}

for check, result in step12_checks.items():

    print(
        "✅" if result else "❌",
        check
    )

if all(step12_checks.values()):

    print(
        "\n🟢 STEP 12 — CSV EXPORT AND "
        "FINAL DATASET VERIFICATION PASSED"
    )

else:

    print(
        "\n🔴 STEP 12 NEEDS REVIEW"
    )

STEP 12 — FINAL VERIFICATION
✅ train.csv exists
✅ validation.csv exists
✅ test.csv exists
✅ Train shape correct
✅ Validation shape correct
✅ Test shape correct
✅ Train columns preserved
✅ Validation columns preserved
✅ Test columns preserved
✅ Train has no missing values
✅ Validation has no missing values
✅ Test has no missing values
✅ Target exists in train
✅ Target exists in validation
✅ Target exists in test

🟢 STEP 12 — CSV EXPORT AND FINAL DATASET VERIFICATION PASSED


**STEP 13 — FEATURE ENCODING & REPRESENTATION**

In [205]:
# Cell 1 — Step 13 setup

print("STEP 13 — FEATURE ENCODING & REPRESENTATION")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nDtypes:")
print(X_train.dtypes.value_counts())

print("\nStep 13 objective:")
print("Convert categorical features into model-ready representations.")
print("All encoding decisions will be learned from X_train only.")

STEP 13 — FEATURE ENCODING & REPRESENTATION
X_train: (1078479, 153)
X_val: (134810, 153)
X_test: (134810, 153)

Dtypes:
float64    73
int8       67
object     13
Name: count, dtype: int64

Step 13 objective:
Convert categorical features into model-ready representations.
All encoding decisions will be learned from X_train only.


In [206]:
# Cell 2 — Identify categorical features

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_feature_columns = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Categorical features:", len(categorical_columns))
print("Numeric features:", len(numeric_feature_columns))

print("\nCategorical features:")
for column in categorical_columns:
    print("-", column)

Categorical features: 13
Numeric features: 140

Categorical features:
- term
- grade
- sub_grade
- emp_title
- emp_length
- home_ownership
- verification_status
- purpose
- title
- zip_code
- addr_state
- earliest_cr_line
- application_type


In [207]:
# Cell 3 — Categorical cardinality audit

print("CATEGORICAL CARDINALITY AUDIT")
print("=" * 70)

categorical_cardinality = (
    X_train[categorical_columns]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)

for column, count in categorical_cardinality.items():
    missing_count = X_train[column].isna().sum()

    print(
        f"{column}: "
        f"{count:,} unique values | "
        f"missing={missing_count:,}"
    )

CATEGORICAL CARDINALITY AUDIT
emp_title: 317,883 unique values | missing=0
title: 52,135 unique values | missing=0
zip_code: 936 unique values | missing=0
earliest_cr_line: 733 unique values | missing=0
addr_state: 51 unique values | missing=0
sub_grade: 35 unique values | missing=0
purpose: 14 unique values | missing=0
emp_length: 12 unique values | missing=0
grade: 7 unique values | missing=0
home_ownership: 6 unique values | missing=0
verification_status: 3 unique values | missing=0
term: 2 unique values | missing=0
application_type: 2 unique values | missing=0


In [208]:
# Cell 4 — Categorize categorical features

low_cardinality_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False) <= 50
]

high_cardinality_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False) > 50
]

print("LOW-CARDINALITY CATEGORICAL FEATURES")
print("-" * 70)

for column in low_cardinality_columns:
    print(
        f"- {column}: "
        f"{X_train[column].nunique(dropna=False)} unique values"
    )

print("\nHIGH-CARDINALITY CATEGORICAL FEATURES")
print("-" * 70)

for column in high_cardinality_columns:
    print(
        f"- {column}: "
        f"{X_train[column].nunique(dropna=False):,} unique values"
    )

LOW-CARDINALITY CATEGORICAL FEATURES
----------------------------------------------------------------------
- term: 2 unique values
- grade: 7 unique values
- sub_grade: 35 unique values
- emp_length: 12 unique values
- home_ownership: 6 unique values
- verification_status: 3 unique values
- purpose: 14 unique values
- application_type: 2 unique values

HIGH-CARDINALITY CATEGORICAL FEATURES
----------------------------------------------------------------------
- emp_title: 317,883 unique values
- title: 52,135 unique values
- zip_code: 936 unique values
- addr_state: 51 unique values
- earliest_cr_line: 733 unique values


In [209]:
# Cell 5 — Inspect low-cardinality categories

print("LOW-CARDINALITY CATEGORY VALUES")
print("=" * 70)

for column in low_cardinality_columns:

    print(f"\n{column}")
    print("-" * 50)

    values = (
        X_train[column]
        .value_counts(dropna=False)
        .head(50)
    )

    print(values)

LOW-CARDINALITY CATEGORY VALUES

term
--------------------------------------------------
term
36 months    818698
60 months    259781
Name: count, dtype: int64

grade
--------------------------------------------------
grade
B    314313
C    305639
A    188308
D    161510
E     75297
F     25901
G      7511
Name: count, dtype: int64

sub_grade
--------------------------------------------------
sub_grade
C1    68542
B4    66711
B5    66122
B3    65428
C2    63235
C3    60056
C4    59667
B2    59021
B1    57031
C5    54139
A5    51166
A4    41838
D1    41271
D2    35959
A1    34898
D3    31653
A3    30542
A2    29864
D4    28639
D5    23988
E1    19135
E2    17235
E3    14736
E4    12576
E5    11615
F1     7983
F2     5837
F3     4922
F4     3948
F5     3211
G1     2443
G2     1756
G3     1318
G4     1055
G5      939
Name: count, dtype: int64

emp_length
--------------------------------------------------
emp_length
10+ years    353980
2 years       97744
< 1 year      86885
3 years       

In [210]:
# Cell 6 — Inspect high-cardinality fields

print("HIGH-CARDINALITY FEATURE REVIEW")
print("=" * 70)

for column in high_cardinality_columns:

    print(f"\n{column}")
    print("-" * 50)

    print("Unique values:", X_train[column].nunique(dropna=False))
    print("Missing:", X_train[column].isna().sum())

    print("\nMost frequent values:")
    print(
        X_train[column]
        .value_counts(dropna=False)
        .head(10)
    )

HIGH-CARDINALITY FEATURE REVIEW

emp_title
--------------------------------------------------
Unique values: 317883
Missing: 0

Most frequent values:
emp_title
Missing             68874
Teacher             17030
Manager             15682
Owner                8273
Registered Nurse     6972
RN                   6797
Supervisor           6636
Driver               6033
Sales                5964
Project Manager      5154
Name: count, dtype: int64

title
--------------------------------------------------
Unique values: 52135
Missing: 0

Most frequent values:
title
Debt consolidation         528613
Credit card refinancing    199893
Home improvement            60291
Other                       53370
Major purchase              19216
Missing                     13277
Debt Consolidation          12687
Medical expenses            10867
Business                     9484
Car financing                8983
Name: count, dtype: int64

zip_code
--------------------------------------------------
Unique v

In [211]:
# Cell 7 — Rare-category audit

print("RARE CATEGORY AUDIT")
print("=" * 70)

rare_category_summary = {}

for column in categorical_columns:

    value_counts = X_train[column].value_counts(
        dropna=False
    )

    rare_count = (
        value_counts <= 10
    ).sum()

    rare_category_summary[column] = int(rare_count)

    print(
        f"{column}: "
        f"{rare_count:,} categories with <= 10 observations"
    )

RARE CATEGORY AUDIT
term: 0 categories with <= 10 observations
grade: 0 categories with <= 10 observations
sub_grade: 0 categories with <= 10 observations
emp_title: 310,234 categories with <= 10 observations
emp_length: 0 categories with <= 10 observations
home_ownership: 0 categories with <= 10 observations
verification_status: 0 categories with <= 10 observations
purpose: 0 categories with <= 10 observations
title: 51,076 categories with <= 10 observations
zip_code: 68 categories with <= 10 observations
addr_state: 0 categories with <= 10 observations
earliest_cr_line: 101 categories with <= 10 observations
application_type: 0 categories with <= 10 observations


In [212]:
# Cell 8 — Encoding treatment groups

# Low-cardinality categorical features.
# These will use one-hot encoding.

one_hot_columns = low_cardinality_columns.copy()

# High-cardinality features require separate handling.
# We will not one-hot encode them blindly.

high_cardinality_review_columns = high_cardinality_columns.copy()

print("ONE-HOT ENCODING GROUP")
print("=" * 70)

for column in one_hot_columns:
    print("-", column)

print("\nHIGH-CARDINALITY REVIEW GROUP")
print("=" * 70)

for column in high_cardinality_review_columns:
    print("-", column)

ONE-HOT ENCODING GROUP
- term
- grade
- sub_grade
- emp_length
- home_ownership
- verification_status
- purpose
- application_type

HIGH-CARDINALITY REVIEW GROUP
- emp_title
- title
- zip_code
- addr_state
- earliest_cr_line


In [213]:
# Cell 9 — Feature representation confirmation

print("FEATURE REPRESENTATION CONFIRMATION")
print("=" * 70)

representation_plan = {}

for column in one_hot_columns:
    representation_plan[column] = "one_hot"

for column in high_cardinality_review_columns:
    representation_plan[column] = "high_cardinality_review"

for column, treatment in representation_plan.items():
    print(f"{column}: {treatment}")

FEATURE REPRESENTATION CONFIRMATION
term: one_hot
grade: one_hot
sub_grade: one_hot
emp_length: one_hot
home_ownership: one_hot
verification_status: one_hot
purpose: one_hot
application_type: one_hot
emp_title: high_cardinality_review
title: high_cardinality_review
zip_code: high_cardinality_review
addr_state: high_cardinality_review
earliest_cr_line: high_cardinality_review


In [214]:
# Cell 10 — Category consistency audit

print("CATEGORY CONSISTENCY AUDIT")
print("=" * 70)

for column in categorical_columns:

    train_values = set(
        X_train[column]
        .dropna()
        .unique()
    )

    val_values = set(
        X_val[column]
        .dropna()
        .unique()
    )

    test_values = set(
        X_test[column]
        .dropna()
        .unique()
    )

    unseen_val = val_values - train_values
    unseen_test = test_values - train_values

    print(f"\n{column}")
    print("Train categories:", len(train_values))
    print("Validation-only categories:", len(unseen_val))
    print("Test-only categories:", len(unseen_test))

CATEGORY CONSISTENCY AUDIT

term
Train categories: 2
Validation-only categories: 0
Test-only categories: 0

grade
Train categories: 7
Validation-only categories: 0
Test-only categories: 0

sub_grade
Train categories: 35
Validation-only categories: 0
Test-only categories: 0

emp_title
Train categories: 317883
Validation-only categories: 31199
Test-only categories: 31617

emp_length
Train categories: 12
Validation-only categories: 0
Test-only categories: 0

home_ownership
Train categories: 6
Validation-only categories: 0
Test-only categories: 0

verification_status
Train categories: 3
Validation-only categories: 0
Test-only categories: 0

purpose
Train categories: 14
Validation-only categories: 0
Test-only categories: 0

title
Train categories: 52135
Validation-only categories: 5510
Test-only categories: 5604

zip_code
Train categories: 936
Validation-only categories: 4
Test-only categories: 6

addr_state
Train categories: 51
Validation-only categories: 0
Test-only categories: 0

earlies

In [215]:
# Cell 11 — Freeze Step 13 representation decisions

step13_plan = {
    "one_hot_columns": one_hot_columns,
    "high_cardinality_columns": high_cardinality_review_columns,
    "numeric_columns": numeric_feature_columns
}

print("STEP 13 REPRESENTATION PLAN")
print("=" * 70)

print("\nOne-hot columns:")
for column in step13_plan["one_hot_columns"]:
    print("-", column)

print("\nHigh-cardinality columns:")
for column in step13_plan["high_cardinality_columns"]:
    print("-", column)

print("\nNumeric columns:", len(step13_plan["numeric_columns"]))

STEP 13 REPRESENTATION PLAN

One-hot columns:
- term
- grade
- sub_grade
- emp_length
- home_ownership
- verification_status
- purpose
- application_type

High-cardinality columns:
- emp_title
- title
- zip_code
- addr_state
- earliest_cr_line

Numeric columns: 140


In [216]:
# Cell 12 — Final Step 13 representation audit

print("STEP 13 — FINAL REPRESENTATION AUDIT")
print("=" * 70)

all_feature_columns = (
    set(numeric_feature_columns)
    | set(categorical_columns)
)

covered_columns = (
    set(numeric_feature_columns)
    | set(one_hot_columns)
    | set(high_cardinality_review_columns)
)

missing_from_plan = (
    all_feature_columns - covered_columns
)

unexpected_in_plan = (
    covered_columns - all_feature_columns
)

print("Total original feature columns:", len(all_feature_columns))
print("Columns covered by representation plan:", len(covered_columns))

print("\nMissing from representation plan:")
print(missing_from_plan)

print("\nUnexpected columns in representation plan:")
print(unexpected_in_plan)

print("\nStep 13 representation coverage:",
      len(missing_from_plan) == 0)

if len(missing_from_plan) == 0 and len(unexpected_in_plan) == 0:
    print("\n🟢 STEP 13 REPRESENTATION AUDIT PASSED")
else:
    print("\n🔴 STEP 13 NEEDS REVIEW")

STEP 13 — FINAL REPRESENTATION AUDIT
Total original feature columns: 153
Columns covered by representation plan: 153

Missing from representation plan:
set()

Unexpected columns in representation plan:
set()

Step 13 representation coverage: True

🟢 STEP 13 REPRESENTATION AUDIT PASSED


In [217]:
# Cell 13 — Encoding imports

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

print("Encoding libraries loaded successfully.")

Encoding libraries loaded successfully.


In [218]:
# Cell 14 — Define categorical encoding groups

# Low-cardinality categorical features
one_hot_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False) <= 50
]

# High-cardinality categorical features
high_cardinality_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False) > 50
]

print("One-hot columns:")
for column in one_hot_columns:
    print("-", column)

print("\nHigh-cardinality columns:")
for column in high_cardinality_columns:
    print("-", column)

One-hot columns:
- term
- grade
- sub_grade
- emp_length
- home_ownership
- verification_status
- purpose
- application_type

High-cardinality columns:
- emp_title
- title
- zip_code
- addr_state
- earliest_cr_line


In [219]:
# Cell 15 — High-cardinality frequency audit

print("HIGH-CARDINALITY FREQUENCY AUDIT")
print("=" * 70)

for column in high_cardinality_columns:

    counts = X_train[column].value_counts(
        dropna=False
    )

    print(f"\n{column}")
    print("Unique:", len(counts))
    print("Top 10:")
    print(counts.head(10))

HIGH-CARDINALITY FREQUENCY AUDIT

emp_title
Unique: 317883
Top 10:
emp_title
Missing             68874
Teacher             17030
Manager             15682
Owner                8273
Registered Nurse     6972
RN                   6797
Supervisor           6636
Driver               6033
Sales                5964
Project Manager      5154
Name: count, dtype: int64

title
Unique: 52135
Top 10:
title
Debt consolidation         528613
Credit card refinancing    199893
Home improvement            60291
Other                       53370
Major purchase              19216
Missing                     13277
Debt Consolidation          12687
Medical expenses            10867
Business                     9484
Car financing                8983
Name: count, dtype: int64

zip_code
Unique: 936
Top 10:
zip_code
945xx    12060
750xx    11648
112xx    11083
606xx     9965
300xx     9784
331xx     9134
070xx     8613
100xx     8558
891xx     8509
770xx     8467
Name: count, dtype: int64

addr_state
Unique: 5

In [220]:
# Cell 16 — Define rare-category threshold

RARE_CATEGORY_MIN_COUNT = 50

print(
    "Rare-category threshold:",
    RARE_CATEGORY_MIN_COUNT
)

print(
    "\nCategories occurring fewer than",
    RARE_CATEGORY_MIN_COUNT,
    "times will be grouped into '__RARE__'."
)

Rare-category threshold: 50

Categories occurring fewer than 50 times will be grouped into '__RARE__'.


In [221]:
# Cell 17 — Learn rare-category mappings from X_train only

rare_category_maps = {}

for column in categorical_columns:

    value_counts = X_train[column].value_counts(
        dropna=False
    )

    frequent_values = set(
        value_counts[
            value_counts >= RARE_CATEGORY_MIN_COUNT
        ].index
    )

    rare_category_maps[column] = frequent_values

print("Rare-category mappings learned from X_train only.")

for column, frequent_values in rare_category_maps.items():
    print(
        f"{column}: "
        f"{len(frequent_values):,} retained categories"
    )

Rare-category mappings learned from X_train only.
term: 2 retained categories
grade: 7 retained categories
sub_grade: 35 retained categories
emp_title: 1,739 retained categories
emp_length: 12 retained categories
home_ownership: 5 retained categories
verification_status: 3 retained categories
purpose: 14 retained categories
title: 260 retained categories
zip_code: 841 retained categories
addr_state: 50 retained categories
earliest_cr_line: 561 retained categories
application_type: 2 retained categories


In [222]:
# Cell 18 — Apply rare-category mapping

def apply_rare_category_mapping(
    dataframe,
    categorical_columns,
    rare_category_maps
):
    dataframe = dataframe.copy()

    for column in categorical_columns:

        frequent_values = rare_category_maps[column]

        dataframe[column] = dataframe[column].where(
            dataframe[column].isin(frequent_values),
            "__RARE__"
        )

        dataframe[column] = dataframe[column].fillna(
            "__MISSING__"
        )

        dataframe[column] = dataframe[column].astype(str)

    return dataframe


X_train_encoded_source = apply_rare_category_mapping(
    X_train,
    categorical_columns,
    rare_category_maps
)

X_val_encoded_source = apply_rare_category_mapping(
    X_val,
    categorical_columns,
    rare_category_maps
)

X_test_encoded_source = apply_rare_category_mapping(
    X_test,
    categorical_columns,
    rare_category_maps
)

print("Rare-category mapping applied successfully.")

Rare-category mapping applied successfully.


In [223]:
# Cell 19 — Verify cardinality reduction

print("CARDINALITY AFTER RARE-CATEGORY GROUPING")
print("=" * 70)

for column in categorical_columns:

    train_unique = (
        X_train_encoded_source[column]
        .nunique(dropna=False)
    )

    val_unique = (
        X_val_encoded_source[column]
        .nunique(dropna=False)
    )

    test_unique = (
        X_test_encoded_source[column]
        .nunique(dropna=False)
    )

    print(
        f"{column}: "
        f"train={train_unique:,}, "
        f"val={val_unique:,}, "
        f"test={test_unique:,}"
    )

CARDINALITY AFTER RARE-CATEGORY GROUPING
term: train=2, val=2, test=2
grade: train=7, val=7, test=7
sub_grade: train=35, val=35, test=35
emp_title: train=1,740, val=1,740, test=1,739
emp_length: train=12, val=12, test=12
home_ownership: train=6, val=6, test=6
verification_status: train=3, val=3, test=3
purpose: train=14, val=14, test=14
title: train=261, val=261, test=261
zip_code: train=842, val=842, test=842
addr_state: train=51, val=51, test=51
earliest_cr_line: train=562, val=562, test=562
application_type: train=2, val=2, test=2


In [224]:
# Cell 20 — Final categorical groups after rare grouping

final_one_hot_columns = [
    column
    for column in categorical_columns
    if column in one_hot_columns
]

print("Final one-hot columns:")
for column in final_one_hot_columns:
    print("-", column)

print(
    "\nNumber of one-hot source columns:",
    len(final_one_hot_columns)
)

Final one-hot columns:
- term
- grade
- sub_grade
- emp_length
- home_ownership
- verification_status
- purpose
- application_type

Number of one-hot source columns: 8


In [225]:
# Cell 21 — Build one-hot encoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

print("OneHotEncoder configured.")
print("Unknown categories: ignored")
print("Encoder will be fitted on X_train only.")

OneHotEncoder configured.
Unknown categories: ignored
Encoder will be fitted on X_train only.


In [226]:
# Cell 22 — Fit encoder on X_train only

encoder.fit(
    X_train_encoded_source[
        final_one_hot_columns
    ]
)

print("Encoder fitted successfully.")
print("Fit source: X_train only.")

Encoder fitted successfully.
Fit source: X_train only.


In [227]:
# Cell 23 — Transform categorical features

X_train_cat = encoder.transform(
    X_train_encoded_source[
        final_one_hot_columns
    ]
)

X_val_cat = encoder.transform(
    X_val_encoded_source[
        final_one_hot_columns
    ]
)

X_test_cat = encoder.transform(
    X_test_encoded_source[
        final_one_hot_columns
    ]
)

print("Categorical transformation complete.")

print("\nTrain categorical matrix:", X_train_cat.shape)
print("Validation categorical matrix:", X_val_cat.shape)
print("Test categorical matrix:", X_test_cat.shape)

Categorical transformation complete.

Train categorical matrix: (1078479, 81)
Validation categorical matrix: (134810, 81)
Test categorical matrix: (134810, 81)


In [228]:
# Cell 24 — Prepare numeric feature matrices

numeric_columns_final = [
    column
    for column in X_train.columns
    if column not in categorical_columns
]

print("Numeric feature count:", len(numeric_columns_final))

X_train_num = X_train[
    numeric_columns_final
].to_numpy(dtype=np.float32)

X_val_num = X_val[
    numeric_columns_final
].to_numpy(dtype=np.float32)

X_test_num = X_test[
    numeric_columns_final
].to_numpy(dtype=np.float32)

print("X_train numeric:", X_train_num.shape)
print("X_val numeric:", X_val_num.shape)
print("X_test numeric:", X_test_num.shape)

Numeric feature count: 140
X_train numeric: (1078479, 140)
X_val numeric: (134810, 140)
X_test numeric: (134810, 140)


In [229]:
# Cell 25 — Combine numeric and categorical features

from scipy import sparse

X_train_final = sparse.hstack(
    [
        sparse.csr_matrix(X_train_num),
        X_train_cat
    ],
    format="csr"
)

X_val_final = sparse.hstack(
    [
        sparse.csr_matrix(X_val_num),
        X_val_cat
    ],
    format="csr"
)

X_test_final = sparse.hstack(
    [
        sparse.csr_matrix(X_test_num),
        X_test_cat
    ],
    format="csr"
)

print("FINAL MODEL-READY MATRICES")
print("=" * 70)

print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

FINAL MODEL-READY MATRICES
X_train_final: (1078479, 221)
X_val_final: (134810, 221)
X_test_final: (134810, 221)


In [230]:
# Cell 26 — Verify feature alignment

print("FEATURE MATRIX ALIGNMENT")
print("=" * 70)

print(
    "Train / Validation feature count:",
    X_train_final.shape[1] == X_val_final.shape[1]
)

print(
    "Train / Test feature count:",
    X_train_final.shape[1] == X_test_final.shape[1]
)

print(
    "Train rows:",
    X_train_final.shape[0]
)

print(
    "Validation rows:",
    X_val_final.shape[0]
)

print(
    "Test rows:",
    X_test_final.shape[0]
)

FEATURE MATRIX ALIGNMENT
Train / Validation feature count: True
Train / Test feature count: True
Train rows: 1078479
Validation rows: 134810
Test rows: 134810


In [231]:
# Cell 27 — Final finite-value audit

print("FINITE VALUE AUDIT")
print("=" * 70)

print(
    "X_train finite:",
    np.isfinite(X_train_final.data).all()
)

print(
    "X_val finite:",
    np.isfinite(X_val_final.data).all()
)

print(
    "X_test finite:",
    np.isfinite(X_test_final.data).all()
)

FINITE VALUE AUDIT
X_train finite: True
X_val finite: True
X_test finite: True


In [232]:
# Cell 28 — Target separation audit

print("TARGET SEPARATION AUDIT")
print("=" * 70)

TARGET_NAME = "target"

print("Target column:", TARGET_NAME)

print("\nTarget vector sizes:")
print("y_train:", len(y_train))
print("y_val:", len(y_val))
print("y_test:", len(y_test))

print("\nFeature matrix sizes:")
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

print("\nTarget / feature alignment:")

train_aligned = (
    X_train_final.shape[0] == len(y_train)
)

val_aligned = (
    X_val_final.shape[0] == len(y_val)
)

test_aligned = (
    X_test_final.shape[0] == len(y_test)
)

print(
    "Train X/y aligned:",
    train_aligned
)

print(
    "Validation X/y aligned:",
    val_aligned
)

print(
    "Test X/y aligned:",
    test_aligned
)

print("\nTarget is stored separately from feature matrices.")

if train_aligned and val_aligned and test_aligned:
    print("\n🟢 TARGET SEPARATION CHECK PASSED")
else:
    print("\n🔴 TARGET SEPARATION CHECK FAILED")

TARGET SEPARATION AUDIT
Target column: target

Target vector sizes:
y_train: 1078479
y_val: 134810
y_test: 134810

Feature matrix sizes:
X_train_final: (1078479, 221)
X_val_final: (134810, 221)
X_test_final: (134810, 221)

Target / feature alignment:
Train X/y aligned: True
Validation X/y aligned: True
Test X/y aligned: True

Target is stored separately from feature matrices.

🟢 TARGET SEPARATION CHECK PASSED


In [233]:
# Cell 29 — Save Step 13 encoder

import joblib
from pathlib import Path

step13_dir = Path("step13_artifacts")
step13_dir.mkdir(exist_ok=True)

encoder_path = step13_dir / "categorical_encoder.joblib"
mapping_path = step13_dir / "rare_category_maps.joblib"

joblib.dump(
    encoder,
    encoder_path
)

joblib.dump(
    rare_category_maps,
    mapping_path
)

print("STEP 13 ARTIFACTS SAVED")
print("=" * 70)

print("Encoder:")
print(encoder_path)

print("\nRare-category mappings:")
print(mapping_path)

STEP 13 ARTIFACTS SAVED
Encoder:
step13_artifacts\categorical_encoder.joblib

Rare-category mappings:
step13_artifacts\rare_category_maps.joblib


In [234]:
# Cell 30 — FINAL STEP 13 AUDIT

print("STEP 13 — FINAL FEATURE ENCODING AUDIT")
print("=" * 70)

step13_checks = {
    "Categorical features identified": (
        len(categorical_columns) > 0
    ),

    "Encoder fitted": (
        hasattr(encoder, "categories_")
    ),

    "Train categorical matrix created": (
        X_train_cat is not None
    ),

    "Validation categorical matrix created": (
        X_val_cat is not None
    ),

    "Test categorical matrix created": (
        X_test_cat is not None
    ),

    "Train feature count matches validation": (
        X_train_final.shape[1] == X_val_final.shape[1]
    ),

    "Train feature count matches test": (
        X_train_final.shape[1] == X_test_final.shape[1]
    ),

    "Train and target aligned": (
        X_train_final.shape[0] == len(y_train)
    ),

    "Validation and target aligned": (
        X_val_final.shape[0] == len(y_val)
    ),

    "Test and target aligned": (
        X_test_final.shape[0] == len(y_test)
    ),

    "Train finite": (
        np.isfinite(X_train_final.data).all()
    ),

    "Validation finite": (
        np.isfinite(X_val_final.data).all()
    ),

    "Test finite": (
        np.isfinite(X_test_final.data).all()
    ),

    "Unknown categories handled safely": (
        encoder.handle_unknown == "ignore"
    )
}

for check, result in step13_checks.items():

    print(
        "✅" if result else "❌",
        check
    )

print("\n" + "=" * 70)

if all(step13_checks.values()):
    print("🟢 STEP 13 — FEATURE ENCODING COMPLETE")
else:
    print("🔴 STEP 13 — NEEDS REVIEW")

STEP 13 — FINAL FEATURE ENCODING AUDIT
✅ Categorical features identified
✅ Encoder fitted
✅ Train categorical matrix created
✅ Validation categorical matrix created
✅ Test categorical matrix created
✅ Train feature count matches validation
✅ Train feature count matches test
✅ Train and target aligned
✅ Validation and target aligned
✅ Test and target aligned
✅ Train finite
✅ Validation finite
✅ Test finite
✅ Unknown categories handled safely

🟢 STEP 13 — FEATURE ENCODING COMPLETE


**STEP 14 Numeric feature audit and outlier treatment**

In [235]:
# Cell 1 — STEP 14 SETUP

import numpy as np
import pandas as pd

print("STEP 14 — NUMERIC FEATURE AND OUTLIER AUDIT")
print("=" * 70)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("\nNumeric feature count:", len(numeric_columns))

print("\nStep 14 objective:")
print("- Audit numerical distributions")
print("- Detect extreme values")
print("- Check invalid/infinite values")
print("- Quantify potential outliers")
print("- Apply treatment only where justified")

STEP 14 — NUMERIC FEATURE AND OUTLIER AUDIT
X_train shape: (1078479, 153)
X_val shape: (134810, 153)
X_test shape: (134810, 153)

Numeric feature count: 74

Step 14 objective:
- Audit numerical distributions
- Detect extreme values
- Check invalid/infinite values
- Quantify potential outliers
- Apply treatment only where justified


In [236]:
# Cell 2 — Infinite-value audit

print("INFINITE VALUE AUDIT")
print("=" * 70)

train_inf = np.isinf(
    X_train[numeric_columns].to_numpy(
        dtype=np.float64
    )
).sum()

val_inf = np.isinf(
    X_val[numeric_columns].to_numpy(
        dtype=np.float64
    )
).sum()

test_inf = np.isinf(
    X_test[numeric_columns].to_numpy(
        dtype=np.float64
    )
).sum()

print("X_train infinite values:", train_inf)
print("X_val infinite values:", val_inf)
print("X_test infinite values:", test_inf)

if train_inf == 0 and val_inf == 0 and test_inf == 0:
    print("\n🟢 No infinite numerical values found.")
else:
    print("\n🔴 Infinite values require treatment.")

INFINITE VALUE AUDIT
X_train infinite values: 0
X_val infinite values: 0
X_test infinite values: 0

🟢 No infinite numerical values found.


In [237]:
# Cell 3 — Numerical summary

numeric_summary = X_train[numeric_columns].describe(
    percentiles=[
        0.001,
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        0.999
    ]
).T

numeric_summary["missing"] = (
    X_train[numeric_columns]
    .isna()
    .sum()
)

print("NUMERICAL FEATURE SUMMARY")
print("=" * 70)

display(numeric_summary)

NUMERICAL FEATURE SUMMARY


,count,mean,std,min,0.1%,1%,5%,25%,50%,75%,95%,99%,99.9%,max,missing
loan_amnt,1078479.0,14412.209672,8719.450335,500.00,1000.000,1500.00,3200.00,7975.00,12000.00,20000.00,32900.00,35000.00,4.000000e+04,40000.00,0
int_rate,1078479.0,13.241886,4.767231,5.31,5.320,5.32,6.49,9.75,12.74,15.99,22.15,26.30,3.089000e+01,30.99,0
installment,1078479.0,437.895672,261.572035,4.93,32.500,52.76,108.49,248.30,375.07,580.41,963.66,1222.31,1.368660e+03,1719.83,0
annual_inc,1078479.0,76262.066355,69690.391864,0.00,9519.296,18000.00,28000.00,45783.00,65000.00,90000.00,155000.00,252000.00,5.770440e+05,9550000.00,0
dti,1078479.0,18.271768,11.159898,-1.00,0.210,1.76,4.96,11.79,17.60,24.04,32.98,38.46,6.596566e+01,999.00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tot_hi_cred_lim,1078479.0,171112.065597,173799.999254,0.00,3600.000,8800.00,19400.00,51739.50,112407.00,242836.50,490252.50,755621.30,1.369145e+06,9999999.00,0
total_bal_ex_mort,1078479.0,49200.498001,46771.476541,0.00,101.000,2416.00,7026.00,21499.00,37294.00,61109.00,131089.10,231545.22,4.285714e+05,2921551.00,0
total_bc_limit,1078479.0,21382.602146,21179.245642,0.00,0.000,0.00,2300.00,8000.00,15100.00,27400.00,62200.00,102600.00,1.691000e+05,1105500.00,0
total_il_high_credit_limit,1078479.0,41591.818867,42117.850276,0.00,0.000,0.00,0.00,15540.00,31685.50,54700.00,117670.10,200829.10,3.568775e+05,2000000.00,0


In [238]:
# Cell 4 — Negative-value audit

print("NEGATIVE VALUE AUDIT")
print("=" * 70)

negative_counts = {}

for column in numeric_columns:

    count = (
        X_train[column] < 0
    ).sum()

    if count > 0:
        negative_counts[column] = count

if len(negative_counts) == 0:

    print("🟢 No negative values found.")

else:

    negative_df = (
        pd.Series(
            negative_counts,
            name="negative_count"
        )
        .sort_values(ascending=False)
    )

    print(
        "Features containing negative values:"
    )

    display(negative_df)

NEGATIVE VALUE AUDIT
Features containing negative values:


dti    1
Name: negative_count, dtype: int64

In [239]:
# Cell 5 — IQR outlier audit

print("IQR OUTLIER AUDIT")
print("=" * 70)

iqr_results = []

for column in numeric_columns:

    series = X_train[column].dropna()

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    outlier_count = outlier_mask.sum()

    outlier_percentage = (
        outlier_count / len(series) * 100
    )

    iqr_results.append({
        "feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": outlier_percentage
    })

iqr_audit = pd.DataFrame(iqr_results)

iqr_audit = iqr_audit.sort_values(
    "outlier_percentage",
    ascending=False
)

display(iqr_audit)

IQR OUTLIER AUDIT


,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percentage
9,mths_since_last_delinq,31.0,31.0,0.0,31.00,31.00,526910,48.856770
26,total_bal_il,24061.0,24061.0,0.0,24061.00,24061.00,430139,39.883855
30,max_bal_bc,4202.0,4202.0,0.0,4202.00,4202.00,430111,39.881259
31,all_util,59.0,59.0,0.0,59.00,59.00,421895,39.119445
25,mths_since_rcnt_il,12.0,12.0,0.0,12.00,12.00,403195,37.385522
...,...,...,...,...,...,...,...,...
62,num_tl_30dpd,0.0,0.0,0.0,0.00,0.00,3263,0.302556
61,num_tl_120dpd_2m,0.0,0.0,0.0,0.00,0.00,763,0.070748
39,bc_util,39.7,83.9,44.2,-26.60,150.20,68,0.006305
14,revol_util,33.5,70.8,37.3,-22.45,126.75,58,0.005378


In [240]:
# Cell 6 — Extreme percentile audit

print("EXTREME PERCENTILE AUDIT")
print("=" * 70)

extreme_results = []

for column in numeric_columns:

    series = X_train[column].dropna()

    p001 = series.quantile(0.001)
    p01 = series.quantile(0.01)
    p99 = series.quantile(0.99)
    p999 = series.quantile(0.999)

    extreme_results.append({
        "feature": column,
        "p0.1%": p001,
        "p1%": p01,
        "p99%": p99,
        "p99.9%": p999,
        "min": series.min(),
        "max": series.max()
    })

extreme_audit = pd.DataFrame(
    extreme_results
)

display(extreme_audit)

EXTREME PERCENTILE AUDIT


,feature,p0.1%,p1%,p99%,p99.9%,min,max
0,loan_amnt,1000.000,1500.00,35000.00,4.000000e+04,500.00,40000.00
1,int_rate,5.320,5.32,26.30,3.089000e+01,5.31,30.99
2,installment,32.500,52.76,1222.31,1.368660e+03,4.93,1719.83
3,annual_inc,9519.296,18000.00,252000.00,5.770440e+05,0.00,9550000.00
4,dti,0.210,1.76,38.46,6.596566e+01,-1.00,999.00
...,...,...,...,...,...,...,...
69,tot_hi_cred_lim,3600.000,8800.00,755621.30,1.369145e+06,0.00,9999999.00
70,total_bal_ex_mort,101.000,2416.00,231545.22,4.285714e+05,0.00,2921551.00
71,total_bc_limit,0.000,0.00,102600.00,1.691000e+05,0.00,1105500.00
72,total_il_high_credit_limit,0.000,0.00,200829.10,3.568775e+05,0.00,2000000.00


In [241]:
# Cell 7 — Identify potentially extreme features

print("POTENTIALLY EXTREME FEATURES")
print("=" * 70)

potential_extreme_features = (
    iqr_audit[
        iqr_audit["outlier_percentage"] >= 5
    ]["feature"]
    .tolist()
)

print(
    "Features with >= 5% IQR-defined outliers:"
)

for column in potential_extreme_features:
    print("-", column)

print(
    "\nCount:",
    len(potential_extreme_features)
)

POTENTIALLY EXTREME FEATURES
Features with >= 5% IQR-defined outliers:
- mths_since_last_delinq
- total_bal_il
- max_bal_bc
- all_util
- mths_since_rcnt_il
- il_util
- mths_since_recent_revol_delinq
- inq_last_12m
- open_rv_24m
- open_act_il
- inq_fi
- open_rv_12m
- open_il_24m
- open_il_12m
- open_acc_6m
- mths_since_last_major_derog
- mths_since_recent_bc_dlq
- num_accts_ever_120_pd
- delinq_2yrs
- total_cu_tl
- mths_since_last_record_missing
- pub_rec
- mths_since_last_record
- tot_coll_amt
- pub_rec_bankruptcies
- mths_since_recent_bc
- bc_open_to_buy
- mo_sin_rcnt_rev_tl_op
- mo_sin_old_il_acct
- pct_tl_nvr_dlq
- total_bc_limit
- total_rev_hi_lim
- total_bal_ex_mort
- revol_bal
- num_il_tl
- avg_cur_bal
- mo_sin_rcnt_tl
- total_il_high_credit_limit
- inq_last_6mths
- num_tl_90g_dpd_24m

Count: 40


In [242]:
# Cell 8 — Inspect suspicious numerical ranges

print("SUSPICIOUS RANGE INSPECTION")
print("=" * 70)

for column in potential_extreme_features:

    print(f"\n{column}")

    print(
        X_train[column]
        .describe()
    )

SUSPICIOUS RANGE INSPECTION

mths_since_last_delinq
count    1.078479e+06
mean     3.264102e+01
std      1.553605e+01
min      0.000000e+00
25%      3.100000e+01
50%      3.100000e+01
75%      3.100000e+01
max      2.020000e+02
Name: mths_since_last_delinq, dtype: float64

total_bal_il
count    1.078479e+06
mean     2.875848e+04
std      2.765814e+04
min      0.000000e+00
25%      2.406100e+04
50%      2.406100e+04
75%      2.406100e+04
max      1.547285e+06
Name: total_bal_il, dtype: float64

max_bal_bc
count    1.078479e+06
mean     4.735888e+03
std      3.523564e+03
min      0.000000e+00
25%      4.202000e+03
50%      4.202000e+03
75%      4.202000e+03
max      7.768430e+05
Name: max_bal_bc, dtype: float64

all_util
count    1.078479e+06
mean     5.867905e+01
std      1.321904e+01
min      0.000000e+00
25%      5.900000e+01
50%      5.900000e+01
75%      5.900000e+01
max      1.980000e+02
Name: all_util, dtype: float64

mths_since_rcnt_il
count    1.078479e+06
mean     1.494221e+01


In [243]:
# Cell 9 — Domain sanity checks

print("DOMAIN SANITY CHECKS")
print("=" * 70)

domain_checks = {}

# Percent-like features
percent_features = [
    "int_rate",
    "revol_util",
    "bc_util",
    "il_util",
    "all_util",
    "pct_tl_nvr_dlq",
    "percent_bc_gt_75"
]

for column in percent_features:

    if column in X_train.columns:

        invalid = (
            (X_train[column] < 0) |
            (X_train[column] > 100)
        ).sum()

        domain_checks[column] = invalid


# FICO range
for column in [
    "fico_range_low",
    "fico_range_high"
]:

    if column in X_train.columns:

        invalid = (
            (X_train[column] < 300) |
            (X_train[column] > 850)
        ).sum()

        domain_checks[column] = invalid


print("Invalid-value counts:")

for column, count in domain_checks.items():

    print(
        f"{column}: {count}"
    )

DOMAIN SANITY CHECKS
Invalid-value counts:
int_rate: 0
revol_util: 3784
bc_util: 17081
il_util: 19355
all_util: 5232
pct_tl_nvr_dlq: 0
percent_bc_gt_75: 0
fico_range_low: 0
fico_range_high: 0


In [244]:
# Cell 10 — Outlier treatment decision

print("OUTLIER TREATMENT DECISION")
print("=" * 70)

print("""
Outliers are NOT automatically removed.

Reason:
Credit data naturally contains applicants with:
- unusually high income
- unusually large loans
- high revolving balances
- unusual credit histories
- unusual utilization levels

These observations may contain legitimate credit-risk information.

Therefore:

1. Do not delete rows because of IQR outliers.
2. Do not clip features solely because they have outliers.
3. Invalid domain values will be treated separately.
4. Legitimate extreme observations will be retained.
5. Any transformation must be learned from X_train only.
""")

print("🟢 Decision: retain legitimate numerical extremes.")

OUTLIER TREATMENT DECISION

Outliers are NOT automatically removed.

Reason:
Credit data naturally contains applicants with:
- unusually high income
- unusually large loans
- high revolving balances
- unusual credit histories
- unusual utilization levels

These observations may contain legitimate credit-risk information.

Therefore:

1. Do not delete rows because of IQR outliers.
2. Do not clip features solely because they have outliers.
3. Invalid domain values will be treated separately.
4. Legitimate extreme observations will be retained.
5. Any transformation must be learned from X_train only.

🟢 Decision: retain legitimate numerical extremes.


In [245]:
# Cell 11 — Final numerical treatment policy

step14_numeric_policy = {
    "infinite_values": "none_expected",
    "invalid_domain_values": "audit_required",
    "row_deletion_for_outliers": False,
    "automatic_iqr_clipping": False,
    "legitimate_extremes": "retain",
    "transformation_source": "X_train_only"
}

print("STEP 14 NUMERIC TREATMENT POLICY")
print("=" * 70)

for key, value in step14_numeric_policy.items():
    print(f"{key}: {value}")

STEP 14 NUMERIC TREATMENT POLICY
infinite_values: none_expected
invalid_domain_values: audit_required
row_deletion_for_outliers: False
automatic_iqr_clipping: False
legitimate_extremes: retain
transformation_source: X_train_only


In [246]:
# Cell 12 — Numerical integrity recheck

print("NUMERICAL INTEGRITY RECHECK")
print("=" * 70)

train_numeric = X_train[numeric_columns]

val_numeric = X_val[numeric_columns]

test_numeric = X_test[numeric_columns]

checks = {
    "Train infinite values == 0":
        not np.isinf(
            train_numeric.to_numpy(
                dtype=np.float64
            )
        ).any(),

    "Validation infinite values == 0":
        not np.isinf(
            val_numeric.to_numpy(
                dtype=np.float64
            )
        ).any(),

    "Test infinite values == 0":
        not np.isinf(
            test_numeric.to_numpy(
                dtype=np.float64
            )
        ).any()
}

for name, result in checks.items():

    print(
        "✅" if result else "❌",
        name
    )

NUMERICAL INTEGRITY RECHECK
✅ Train infinite values == 0
✅ Validation infinite values == 0
✅ Test infinite values == 0


In [247]:
# Cell 13 — Verify row preservation

print("ROW PRESERVATION AUDIT")
print("=" * 70)

print(
    "X_train rows:",
    len(X_train)
)

print(
    "X_val rows:",
    len(X_val)
)

print(
    "X_test rows:",
    len(X_test)
)

print("\nNo rows will be removed for legitimate outliers.")

print(
    "\n🟢 Row preservation policy confirmed."
)

ROW PRESERVATION AUDIT
X_train rows: 1078479
X_val rows: 134810
X_test rows: 134810

No rows will be removed for legitimate outliers.

🟢 Row preservation policy confirmed.


In [248]:
# Cell 14 — FINAL STEP 14 AUDIT

print("STEP 14 — FINAL NUMERICAL / OUTLIER AUDIT")
print("=" * 70)

step14_checks = {

    "Numeric features identified":
        len(numeric_columns) > 0,

    "Train has no infinite values":
        not np.isinf(
            X_train[numeric_columns]
            .to_numpy(dtype=np.float64)
        ).any(),

    "Validation has no infinite values":
        not np.isinf(
            X_val[numeric_columns]
            .to_numpy(dtype=np.float64)
        ).any(),

    "Test has no infinite values":
        not np.isinf(
            X_test[numeric_columns]
            .to_numpy(dtype=np.float64)
        ).any(),

    "IQR audit completed":
        len(iqr_audit) == len(numeric_columns),

    "Extreme percentile audit completed":
        len(extreme_audit) == len(numeric_columns),

    "No rows removed for outliers":
        True,

    "Legitimate extreme values retained":
        True,

    "Numerical treatment policy established":
        True
}

for check, result in step14_checks.items():

    print(
        "✅" if result else "❌",
        check
    )

print("\n" + "=" * 70)

if all(step14_checks.values()):

    print(
        "🟢 STEP 14 — NUMERICAL / OUTLIER AUDIT COMPLETE"
    )

else:

    print(
        "🔴 STEP 14 — NEEDS REVIEW"
    )

STEP 14 — FINAL NUMERICAL / OUTLIER AUDIT
✅ Numeric features identified
✅ Train has no infinite values
✅ Validation has no infinite values
✅ Test has no infinite values
✅ IQR audit completed
✅ Extreme percentile audit completed
✅ No rows removed for outliers
✅ Legitimate extreme values retained
✅ Numerical treatment policy established

🟢 STEP 14 — NUMERICAL / OUTLIER AUDIT COMPLETE


In [249]:
# Cell 15 — Inspect invalid utilization values

print("INVALID UTILIZATION VALUE INVESTIGATION")
print("=" * 70)

utilization_features = [
    "revol_util",
    "bc_util",
    "il_util",
    "all_util"
]

for column in utilization_features:

    if column not in X_train.columns:
        continue

    invalid_mask = (
        (X_train[column] < 0) |
        (X_train[column] > 100)
    )

    invalid_values = X_train.loc[
        invalid_mask,
        column
    ]

    print(f"\n{column}")
    print("-" * 50)
    print("Invalid count:", len(invalid_values))

    if len(invalid_values) > 0:
        print("Minimum invalid value:", invalid_values.min())
        print("Maximum invalid value:", invalid_values.max())

        print("\nMost common invalid values:")
        print(
            invalid_values
            .value_counts()
            .head(20)
        )

INVALID UTILIZATION VALUE INVESTIGATION

revol_util
--------------------------------------------------
Invalid count: 3784
Minimum invalid value: 100.1
Maximum invalid value: 892.3

Most common invalid values:
revol_util
101.0    199
100.1    194
100.2    190
100.6    177
100.5    169
100.7    162
100.4    158
100.3    157
100.8    130
100.9    129
101.2    104
101.3    103
101.1     94
101.5     93
101.6     92
102.0     89
101.4     85
101.7     70
101.9     59
101.8     57
Name: count, dtype: int64

bc_util
--------------------------------------------------
Invalid count: 17081
Minimum invalid value: 100.1
Maximum invalid value: 339.6

Most common invalid values:
bc_util
100.1    946
101.0    944
100.2    896
100.3    863
100.4    824
100.5    771
100.6    764
100.7    698
100.9    672
100.8    647
101.1    574
101.2    509
101.3    505
101.4    469
101.6    407
102.0    406
101.5    400
101.7    330
101.8    309
101.9    292
Name: count, dtype: int64

il_util
----------------------

In [250]:
# Cell 16 — Cross-split invalid-value audit

print("CROSS-SPLIT INVALID UTILIZATION AUDIT")
print("=" * 70)

for dataset_name, dataset in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test)
]:

    print(f"\n{dataset_name}")
    print("-" * 50)

    for column in utilization_features:

        if column not in dataset.columns:
            continue

        invalid_mask = (
            (dataset[column] < 0) |
            (dataset[column] > 100)
        )

        print(
            f"{column}:",
            invalid_mask.sum()
        )

CROSS-SPLIT INVALID UTILIZATION AUDIT

X_train
--------------------------------------------------
revol_util: 3784
bc_util: 17081
il_util: 19355
all_util: 5232

X_val
--------------------------------------------------
revol_util: 460
bc_util: 2076
il_util: 2397
all_util: 649

X_test
--------------------------------------------------
revol_util: 470
bc_util: 2104
il_util: 2519
all_util: 715


In [251]:
# Cell 17 — Inspect exact invalid rows

print("EXACT INVALID UTILIZATION ROWS")
print("=" * 70)

for column in utilization_features:

    invalid_mask = (
        (X_train[column] < 0) |
        (X_train[column] > 100)
    )

    if invalid_mask.sum() == 0:
        continue

    print(f"\n{column}")
    print("-" * 50)

    display(
        X_train.loc[
            invalid_mask,
            [column]
        ].head(30)
    )

EXACT INVALID UTILIZATION ROWS

revol_util
--------------------------------------------------


,revol_util
374225,101.8
706976,101.6
102455,100.4
1345806,100.7
112344,126.1
1315706,102.0
125138,100.6
1292851,109.9
573259,100.3
701069,101.0



bc_util
--------------------------------------------------


,bc_util
330409,101.3
374225,101.8
706976,100.3
183751,115.3
102455,100.4
367759,100.1
559944,101.4
1297484,100.4
459773,101.0
1128017,108.0



il_util
--------------------------------------------------


,il_util
898227,105.0
589118,101.0
917356,149.0
419978,101.0
415680,109.0
416458,123.0
1322051,105.0
525081,110.0
434619,104.0
1282786,107.0



all_util
--------------------------------------------------


,all_util
1322051,103.0
525081,103.0
473738,122.0
1272854,102.0
962142,106.0
424005,123.0
972100,113.0
406866,108.0
582336,106.0
442319,104.0


In [252]:
# Cell 18 — Check whether invalid values are already present
# in the current preprocessing data

print("INVALID VALUE ORIGIN CHECK")
print("=" * 70)

for column in utilization_features:

    if column not in accepted_eligible.columns:
        continue

    original_values = accepted_eligible[column]

    invalid_mask = (
        (original_values < 0) |
        (original_values > 100)
    )

    print(
        f"{column}:",
        invalid_mask.sum(),
        "invalid values in accepted_eligible"
    )

    if invalid_mask.sum() > 0:

        print(
            "Range of invalid values:",
            original_values[invalid_mask].min(),
            "to",
            original_values[invalid_mask].max()
        )

INVALID VALUE ORIGIN CHECK
revol_util: 4714 invalid values in accepted_eligible
Range of invalid values: 100.1 to 892.3
bc_util: 21261 invalid values in accepted_eligible
Range of invalid values: 100.1 to 339.6
il_util: 24271 invalid values in accepted_eligible
Range of invalid values: 101.0 to 558.0
all_util: 6596 invalid values in accepted_eligible
Range of invalid values: 101.0 to 198.0


In [253]:
# Cell 18 — Check whether invalid values are already present
# in the current preprocessing data

print("INVALID VALUE ORIGIN CHECK")
print("=" * 70)

for column in utilization_features:

    if column not in accepted_eligible.columns:
        continue

    original_values = accepted_eligible[column]

    invalid_mask = (
        (original_values < 0) |
        (original_values > 100)
    )

    print(
        f"{column}:",
        invalid_mask.sum(),
        "invalid values in accepted_eligible"
    )

    if invalid_mask.sum() > 0:

        print(
            "Range of invalid values:",
            original_values[invalid_mask].min(),
            "to",
            original_values[invalid_mask].max()
        )

INVALID VALUE ORIGIN CHECK
revol_util: 4714 invalid values in accepted_eligible
Range of invalid values: 100.1 to 892.3
bc_util: 21261 invalid values in accepted_eligible
Range of invalid values: 100.1 to 339.6
il_util: 24271 invalid values in accepted_eligible
Range of invalid values: 101.0 to 558.0
all_util: 6596 invalid values in accepted_eligible
Range of invalid values: 101.0 to 198.0


In [254]:
# Cell 20 — Step 14 decision checkpoint

print("STEP 14 — DOMAIN DECISION CHECKPOINT")
print("=" * 70)

print("""
No values have been modified by this checkpoint.

The previous cells only:
1. Identified invalid utilization values.
2. Checked train/validation/test.
3. Inspected their exact ranges.
4. Checked whether they were already present
   in accepted_eligible.

Treatment will only be applied after reviewing
these results.
""")

print("🟡 WAITING FOR DOMAIN-VALUE REVIEW")

STEP 14 — DOMAIN DECISION CHECKPOINT

No values have been modified by this checkpoint.

The previous cells only:
1. Identified invalid utilization values.
2. Checked train/validation/test.
3. Inspected their exact ranges.
4. Checked whether they were already present
   in accepted_eligible.

Treatment will only be applied after reviewing
these results.

🟡 WAITING FOR DOMAIN-VALUE REVIEW


In [255]:
# Cell 21 — Utilization consistency audit

print("UTILIZATION CONSISTENCY AUDIT")
print("=" * 70)

checks = [
    ("revol_util", "revol_bal", "total_rev_hi_lim"),
    ("bc_util", "total_bc_limit", "bc_open_to_buy"),
    ("il_util", "total_bal_il", "total_il_high_credit_limit"),
]

for util_col, balance_col, limit_col in checks:

    print(f"\n{util_col}")
    print("-" * 60)

    if not all(
        col in X_train.columns
        for col in [util_col, balance_col, limit_col]
    ):
        print("Required columns unavailable.")
        continue

    subset = X_train[
        [util_col, balance_col, limit_col]
    ].dropna()

    print("Rows available:", len(subset))

    print("\nUtilization summary:")
    print(subset[util_col].describe())

    print("\nBalance summary:")
    print(subset[balance_col].describe())

    print("\nLimit summary:")
    print(subset[limit_col].describe())

UTILIZATION CONSISTENCY AUDIT

revol_util
------------------------------------------------------------
Rows available: 1078479

Utilization summary:
count    1.078479e+06
mean     5.183073e+01
std      2.452974e+01
min      0.000000e+00
25%      3.350000e+01
50%      5.220000e+01
75%      7.080000e+01
max      8.923000e+02
Name: revol_util, dtype: float64

Balance summary:
count    1.078479e+06
mean     1.627198e+04
std      2.240174e+04
min      0.000000e+00
25%      5.940000e+03
50%      1.113000e+04
75%      1.975800e+04
max      2.904836e+06
Name: revol_bal, dtype: float64

Limit summary:
count    1.078479e+06
mean     3.232286e+04
std      3.519116e+04
min      0.000000e+00
25%      1.450000e+04
50%      2.410000e+04
75%      3.930000e+04
max      9.999999e+06
Name: total_rev_hi_lim, dtype: float64

bc_util
------------------------------------------------------------
Rows available: 1078479

Utilization summary:
count    1.078479e+06
mean     6.011941e+01
std      2.761554e+01
min

In [256]:
# Cell 22 — Reconstructed utilization comparison

print("RECONSTRUCTED UTILIZATION COMPARISON")
print("=" * 70)

reconstruction_specs = [
    ("revol_util", "revol_bal", "total_rev_hi_lim"),
    ("il_util", "total_bal_il", "total_il_high_credit_limit"),
]

for util_col, balance_col, limit_col in reconstruction_specs:

    print(f"\n{util_col}")
    print("-" * 60)

    subset = X_train[
        [util_col, balance_col, limit_col]
    ].dropna()

    valid_limit = subset[limit_col] > 0

    subset = subset.loc[valid_limit].copy()

    subset["_reconstructed_util"] = (
        subset[balance_col]
        / subset[limit_col]
        * 100
    )

    subset["_difference"] = (
        subset[util_col]
        - subset["_reconstructed_util"]
    )

    print(
        "Correlation:",
        subset[util_col].corr(
            subset["_reconstructed_util"]
        )
    )

    print(
        "Median absolute difference:",
        subset["_difference"]
        .abs()
        .median()
    )

    print(
        "95th percentile absolute difference:",
        subset["_difference"]
        .abs()
        .quantile(0.95)
    )

    print("\nExamples with highest differences:")

    display(
        subset.sort_values(
            "_difference",
            key=lambda x: x.abs(),
            ascending=False
        ).head(10)
    )

RECONSTRUCTED UTILIZATION COMPARISON

revol_util
------------------------------------------------------------
Correlation: 0.7948471897412919
Median absolute difference: 0.028571428571432023
95th percentile absolute difference: 14.047578599122645

Examples with highest differences:


,revol_util,revol_bal,total_rev_hi_lim,_reconstructed_util,_difference
959402,99.0,952013.0,24100.0,3950.261411,-3851.261411
959136,68.4,602519.0,24100.0,2500.078838,-2431.678838
959273,91.7,508961.0,24100.0,2111.871369,-2020.171369
958853,12.2,487589.0,24100.0,2023.190871,-2010.990871
958652,89.4,465731.0,24100.0,1932.493776,-1843.093776
960422,20.8,407794.0,24100.0,1692.091286,-1671.291286
958737,89.3,423189.0,24100.0,1755.970954,-1666.670954
960469,20.7,401941.0,24100.0,1667.804979,-1647.104979
959286,77.6,394107.0,24100.0,1635.298755,-1557.698755
959268,89.6,374487.0,24100.0,1553.887967,-1464.287967



il_util
------------------------------------------------------------
Correlation: 0.09162516518448707
Median absolute difference: 11.94156145182842
95th percentile absolute difference: 151.4564705882353

Examples with highest differences:


,il_util,total_bal_il,total_il_high_credit_limit,_reconstructed_util,_difference
1024415,75.0,24061.0,59.0,40781.355932,-40706.355932
1167620,75.0,24061.0,94.0,25596.808511,-25521.808511
334766,75.0,24061.0,100.0,24061.000000,-23986.000000
883029,75.0,24061.0,161.0,14944.720497,-14869.720497
757556,75.0,24061.0,172.0,13988.953488,-13913.953488
115865,75.0,24061.0,173.0,13908.092486,-13833.092486
689861,75.0,24061.0,183.0,13148.087432,-13073.087432
260388,75.0,24061.0,183.0,13148.087432,-13073.087432
1040251,75.0,24061.0,200.0,12030.500000,-11955.500000
721718,75.0,24061.0,200.0,12030.500000,-11955.500000


In [257]:
# Cell 23 — Corrected extreme utilization record inspection

print("EXTREME UTILIZATION RECORDS")
print("=" * 70)

utilization_features = [
    "revol_util",
    "bc_util",
    "il_util",
    "all_util"
]

supporting_features = [
    "revol_bal",
    "total_rev_hi_lim",
    "total_bc_limit",
    "bc_open_to_buy",
    "total_bal_il",
    "total_il_high_credit_limit"
]

# Remove duplicate column names while preserving order
available_supporting = []

for column in supporting_features:
    if column in X_train.columns and column not in available_supporting:
        available_supporting.append(column)

for column in utilization_features:

    if column not in X_train.columns:
        continue

    print(f"\n{column}")
    print("-" * 60)

    # Identify extreme values
    extreme_mask = (
        X_train[column].notna()
        & (X_train[column] > 100)
    )

    extreme_count = extreme_mask.sum()

    print("Values > 100:", extreme_count)

    if extreme_count == 0:
        print("No values above 100.")
        continue

    # Build unique column list
    display_columns = [column]

    for supporting_column in available_supporting:
        if supporting_column != column:
            display_columns.append(supporting_column)

    # Remove duplicate labels from DataFrame safely
    display_columns = list(dict.fromkeys(display_columns))

    extreme_data = X_train.loc[
        extreme_mask,
        display_columns
    ].copy()

    # Sort using the Series directly.
    # This avoids the duplicate-label sort_values issue.
    extreme_data["_sort_value"] = X_train.loc[
        extreme_mask,
        column
    ].to_numpy()

    extreme_data = (
        extreme_data
        .sort_values(
            "_sort_value",
            ascending=False
        )
        .drop(columns="_sort_value")
    )

    print("\nTop 20 extreme observations:")

    display(
        extreme_data.head(20)
    )

EXTREME UTILIZATION RECORDS

revol_util
------------------------------------------------------------
Values > 100: 3784

Top 20 extreme observations:


,revol_util,revol_bal,total_rev_hi_lim,total_bc_limit,bc_open_to_buy,total_bal_il,total_il_high_credit_limit
835912,892.3,2677.0,300.0,0.0,4693.0,24061.0,25644.0
799628,366.6,2566.0,700.0,500.0,134.0,24061.0,17697.0
180764,193.0,17770.0,9200.0,8800.0,4400.0,24061.0,13929.0
369578,184.6,266444.0,305000.0,5000.0,0.0,24061.0,46627.0
95152,182.8,14262.0,7800.0,7800.0,0.0,24061.0,51912.0
731538,180.3,5408.0,3000.0,3000.0,0.0,24061.0,0.0
754221,177.7,7107.0,4000.0,3000.0,0.0,24061.0,48822.0
645052,172.0,66385.0,38600.0,34600.0,11533.0,8234.0,15974.0
626986,165.8,28183.0,17000.0,17000.0,0.0,9026.0,20465.0
1327361,162.0,68960.0,42579.0,42579.0,7097.0,4427.0,10127.0



bc_util
------------------------------------------------------------
Values > 100: 17081

Top 20 extreme observations:


,bc_util,revol_bal,total_rev_hi_lim,total_bc_limit,bc_open_to_buy,total_bal_il,total_il_high_credit_limit
1028201,339.6,3114.0,9600.0,700.0,0.0,24061.0,5000.0
356008,318.2,10311.0,10700.0,800.0,0.0,24061.0,15087.0
752292,255.2,3747.0,6900.0,500.0,0.0,24061.0,21474.0
1004722,252.3,3787.0,5200.0,1500.0,0.0,21880.0,25125.0
311251,243.8,2226.0,1900.0,600.0,0.0,24061.0,29476.0
68563,235.3,2243.0,1900.0,600.0,0.0,24061.0,41876.0
361212,204.6,13167.0,11500.0,3000.0,0.0,24061.0,32194.0
754221,202.9,7107.0,4000.0,3000.0,0.0,24061.0,48822.0
180764,202.0,17770.0,9200.0,8800.0,4400.0,24061.0,13929.0
679630,197.0,8324.0,6200.0,2500.0,0.0,24061.0,106069.0



il_util
------------------------------------------------------------
Values > 100: 19355

Top 20 extreme observations:


,il_util,revol_bal,total_rev_hi_lim,total_bc_limit,bc_open_to_buy,total_bal_il,total_il_high_credit_limit
611880,558.0,5110.0,14300.0,4500.0,1271.0,1567.0,281.0
964601,376.0,2369.0,14900.0,4900.0,4144.0,23523.0,6250.0
1274810,311.0,9413.0,23600.0,17200.0,9501.0,41195.0,13250.0
972487,303.0,2571.0,7400.0,7400.0,4829.0,20144.0,6650.0
898048,293.0,18757.0,25550.0,23250.0,3875.0,136271.0,46577.0
660248,291.0,11105.0,11500.0,10000.0,325.0,37536.0,52572.0
628179,274.0,4827.0,16200.0,14000.0,9369.0,10347.0,3778.0
913891,272.0,1497.0,11100.0,2200.0,1957.0,14136.0,5199.0
982272,268.0,6237.0,14300.0,7100.0,3868.0,38783.0,42819.0
632384,265.0,38040.0,85800.0,82600.0,44571.0,38440.0,14532.0



all_util
------------------------------------------------------------
Values > 100: 5232

Top 20 extreme observations:


,all_util,revol_bal,total_rev_hi_lim,total_bc_limit,bc_open_to_buy,total_bal_il,total_il_high_credit_limit
660248,198.0,11105.0,11500.0,10000.0,325.0,37536.0,52572.0
1254035,184.0,1621.0,2100.0,2100.0,479.0,35258.0,18000.0
523386,173.0,22113.0,22300.0,17200.0,695.0,301096.0,149913.0
645052,172.0,66385.0,38600.0,34600.0,11533.0,8234.0,15974.0
1301574,172.0,2198.0,2500.0,2500.0,302.0,124749.0,71269.0
442045,170.0,4107.0,10700.0,8000.0,5233.0,53426.0,23250.0
577962,169.0,25917.0,25600.0,23600.0,36.0,392137.0,317111.0
532295,165.0,4117.0,5200.0,3500.0,352.0,28849.0,14750.0
630246,165.0,4063.0,5900.0,5900.0,1837.0,30364.0,15000.0
972487,162.0,2571.0,7400.0,7400.0,4829.0,20144.0,6650.0


In [258]:
# Cell 24 — Impossible-value audit

print("IMPOSSIBLE VALUE AUDIT")
print("=" * 70)

impossible_checks = {}

for column in [
    "revol_util",
    "bc_util",
    "il_util",
    "all_util"
]:

    if column not in X_train.columns:
        continue

    series = X_train[column]

    impossible = (
        series < 0
    ).sum()

    impossible_checks[column] = impossible

    print(
        f"{column}: negative values = {impossible}"
    )

print("\nInterpretation:")
print("""
Values above 100% are not automatically treated as invalid.
Negative utilization is structurally invalid.

No values will be modified in this cell.
""")

IMPOSSIBLE VALUE AUDIT
revol_util: negative values = 0
bc_util: negative values = 0
il_util: negative values = 0
all_util: negative values = 0

Interpretation:

Values above 100% are not automatically treated as invalid.
Negative utilization is structurally invalid.

No values will be modified in this cell.



In [259]:
# Cell 25 — Final Step 14 decision

print("STEP 14 — FINAL OUTLIER / DOMAIN DECISION")
print("=" * 70)

negative_utilization_total = 0

for column in [
    "revol_util",
    "bc_util",
    "il_util",
    "all_util"
]:

    if column in X_train.columns:

        negative_utilization_total += (
            X_train[column] < 0
        ).sum()

print(
    "Negative utilization values:",
    negative_utilization_total
)

print("""
Treatment decision:

1. Legitimate extreme numerical observations are retained.
2. IQR-defined outliers are not deleted.
3. Utilization values above 100% are retained.
4. Negative utilization values, if present, require correction.
5. No rows are deleted because of outliers.
6. No validation/test statistics are used.
""")

STEP 14 — FINAL OUTLIER / DOMAIN DECISION
Negative utilization values: 0

Treatment decision:

1. Legitimate extreme numerical observations are retained.
2. IQR-defined outliers are not deleted.
3. Utilization values above 100% are retained.
4. Negative utilization values, if present, require correction.
5. No rows are deleted because of outliers.
6. No validation/test statistics are used.



In [260]:
# Cell 26 — FINAL STEP 14 INTEGRITY AUDIT

print("STEP 14 — FINAL INTEGRITY AUDIT")
print("=" * 70)

checks = {}

for name, dataset in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test)
]:

    checks[f"{name} finite"] = np.isfinite(
        dataset[numeric_columns]
        .to_numpy(dtype=np.float64)
    ).all()

checks["IQR audit complete"] = (
    len(iqr_audit) == len(numeric_columns)
)

checks["Extreme percentile audit complete"] = (
    len(extreme_audit) == len(numeric_columns)
)

checks["No outlier row deletion"] = True

checks["Utilization >100% retained as observed"] = True

for name, result in checks.items():

    print(
        "✅" if result else "❌",
        name
    )

print("\n" + "=" * 70)

if all(checks.values()):
    print(
        "🟢 STEP 14 — NUMERICAL / OUTLIER AUDIT COMPLETE"
    )
else:
    print(
        "🔴 STEP 14 — REVIEW REQUIRED"
    )

STEP 14 — FINAL INTEGRITY AUDIT
✅ X_train finite
✅ X_val finite
✅ X_test finite
✅ IQR audit complete
✅ Extreme percentile audit complete
✅ No outlier row deletion
✅ Utilization >100% retained as observed

🟢 STEP 14 — NUMERICAL / OUTLIER AUDIT COMPLETE


**STEP 15 — FINAL MODEL-READY DATASET**

In [261]:
# Cell 1 — STEP 15 SETUP

print("STEP 15 — FINAL MODEL-READY DATASET")
print("=" * 70)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

STEP 15 — FINAL MODEL-READY DATASET
X_train shape: (1078479, 153)
X_val shape: (134810, 153)
X_test shape: (134810, 153)

Target shapes:
y_train: (1078479,)
y_val: (134810,)
y_test: (134810,)


In [262]:
# Cell 2 — X/Y ROW ALIGNMENT

print("X / Y ROW ALIGNMENT")
print("=" * 70)

alignment_checks = {
    "Train": len(X_train) == len(y_train),
    "Validation": len(X_val) == len(y_val),
    "Test": len(X_test) == len(y_test)
}

for dataset, result in alignment_checks.items():
    print(
        "✅" if result else "❌",
        dataset,
        "X/Y alignment"
    )

if not all(alignment_checks.values()):
    raise ValueError(
        "X/Y row alignment failed."
    )

print("\n🟢 X/Y row alignment verified.")

X / Y ROW ALIGNMENT
✅ Train X/Y alignment
✅ Validation X/Y alignment
✅ Test X/Y alignment

🟢 X/Y row alignment verified.


In [263]:
# Cell 3 — TARGET VALUE AUDIT

print("TARGET VALUE AUDIT")
print("=" * 70)

for name, target in [
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
]:

    print(f"\n{name}")
    print("-" * 50)

    print(
        "Unique values:",
        sorted(
            pd.Series(target)
            .dropna()
            .unique()
            .tolist()
        )
    )

    print(
        "Missing:",
        pd.Series(target).isna().sum()
    )

    print(
        "Distribution:"
    )

    print(
        pd.Series(target)
        .value_counts()
        .sort_index()
    )

TARGET VALUE AUDIT

y_train
--------------------------------------------------
Unique values: [0.0, 1.0]
Missing: 0
Distribution:
target
0.0    862991
1.0    215488
Name: count, dtype: int64

y_val
--------------------------------------------------
Unique values: [0.0, 1.0]
Missing: 0
Distribution:
target
0.0    107874
1.0     26936
Name: count, dtype: int64

y_test
--------------------------------------------------
Unique values: [0.0, 1.0]
Missing: 0
Distribution:
target
0.0    107874
1.0     26936
Name: count, dtype: int64


In [264]:
# Cell 4 — BINARY TARGET VALIDATION

print("BINARY TARGET VALIDATION")
print("=" * 70)

for name, target in [
    ("y_train", y_train),
    ("y_val", y_val),
    ("y_test", y_test)
]:

    unique_values = set(
        pd.Series(target)
        .dropna()
        .unique()
    )

    print(
        name,
        "unique values:",
        unique_values
    )

    if not unique_values.issubset({0, 1}):
        raise ValueError(
            f"{name} contains values outside binary {0, 1}."
        )

print("\n🟢 All targets are binary.")

BINARY TARGET VALIDATION
y_train unique values: {np.float64(0.0), np.float64(1.0)}
y_val unique values: {np.float64(0.0), np.float64(1.0)}
y_test unique values: {np.float64(0.0), np.float64(1.0)}

🟢 All targets are binary.


In [265]:
# Cell 5 — FEATURE COLUMN ALIGNMENT

print("FEATURE COLUMN ALIGNMENT")
print("=" * 70)

train_columns = list(X_train.columns)
val_columns = list(X_val.columns)
test_columns = list(X_test.columns)

print(
    "Train == Validation:",
    train_columns == val_columns
)

print(
    "Train == Test:",
    train_columns == test_columns
)

if train_columns != val_columns:
    raise ValueError(
        "Train and validation feature columns differ."
    )

if train_columns != test_columns:
    raise ValueError(
        "Train and test feature columns differ."
    )

print("\n🟢 Feature columns and ordering match.")

FEATURE COLUMN ALIGNMENT
Train == Validation: True
Train == Test: True

🟢 Feature columns and ordering match.


In [266]:
# Cell 6 — DUPLICATE FEATURE NAME AUDIT

print("DUPLICATE FEATURE NAME AUDIT")
print("=" * 70)

duplicate_columns = (
    X_train.columns[
        X_train.columns.duplicated()
    ]
    .tolist()
)

print(
    "Duplicate feature names:",
    len(duplicate_columns)
)

if duplicate_columns:
    print("\nDuplicates found:")

    for column in duplicate_columns:
        print("-", column)

    raise ValueError(
        "Duplicate feature names exist. "
        "Resolve them before model training."
    )

print(
    "\n🟢 No duplicate feature names."
)

DUPLICATE FEATURE NAME AUDIT
Duplicate feature names: 0

🟢 No duplicate feature names.


In [267]:
# Cell 7 — FINAL MISSING VALUE CHECK

print("FINAL MISSING-VALUE CHECK")
print("=" * 70)

missing_counts = {
    "X_train": X_train.isna().sum().sum(),
    "X_val": X_val.isna().sum().sum(),
    "X_test": X_test.isna().sum().sum()
}

for dataset, count in missing_counts.items():

    print(
        dataset,
        "missing values:",
        count
    )

if any(
    count != 0
    for count in missing_counts.values()
):
    raise ValueError(
        "Missing values remain in the model-ready datasets."
    )

print(
    "\n🟢 No missing values remain."
)

FINAL MISSING-VALUE CHECK
X_train missing values: 0
X_val missing values: 0
X_test missing values: 0

🟢 No missing values remain.


In [268]:
# Cell 8 — FINAL INFINITE VALUE CHECK

print("FINAL INFINITE VALUE CHECK")
print("=" * 70)

for name, dataset in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test)
]:

    numeric_data = dataset.select_dtypes(
        include=np.number
    )

    infinite_count = np.isinf(
        numeric_data.to_numpy(
            dtype=np.float64
        )
    ).sum()

    print(
        name,
        "infinite values:",
        infinite_count
    )

    if infinite_count != 0:
        raise ValueError(
            f"Infinite values found in {name}."
        )

print("\n🟢 No infinite values remain.")

FINAL INFINITE VALUE CHECK
X_train infinite values: 0
X_val infinite values: 0
X_test infinite values: 0

🟢 No infinite values remain.


In [269]:
# Cell 9 — MODEL-READY DATA TYPE AUDIT

print("MODEL-READY DATA TYPE AUDIT")
print("=" * 70)

print(
    "Non-numeric columns in X_train:",
    X_train.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

print(
    "\nNon-numeric columns in X_val:",
    X_val.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

print(
    "\nNon-numeric columns in X_test:",
    X_test.select_dtypes(
        exclude=np.number
    ).columns.tolist()
)

MODEL-READY DATA TYPE AUDIT
Non-numeric columns in X_train: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'application_type']

Non-numeric columns in X_val: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'application_type']

Non-numeric columns in X_test: ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'application_type']


In [270]:
# Cell 10 — Identify Remaining Categorical Features

print("REMAINING CATEGORICAL FEATURES")
print("=" * 70)

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
for column in categorical_features:
    print(
        f"- {column} | "
        f"dtype={X_train[column].dtype} | "
        f"unique={X_train[column].nunique(dropna=False)}"
    )

print("\n" + "=" * 70)

if categorical_features:
    print(
        "🟡 Categorical features still exist."
    )
    print(
        "Encoding is required before the final model matrix."
    )
else:
    print(
        "🟢 No categorical features remain."
    )

REMAINING CATEGORICAL FEATURES
Numeric features: 140
Categorical features: 13

Categorical features:
- term | dtype=object | unique=2
- grade | dtype=object | unique=7
- sub_grade | dtype=object | unique=35
- emp_title | dtype=object | unique=317883
- emp_length | dtype=object | unique=12
- home_ownership | dtype=object | unique=6
- verification_status | dtype=object | unique=3
- purpose | dtype=object | unique=14
- title | dtype=object | unique=52135
- zip_code | dtype=object | unique=936
- addr_state | dtype=object | unique=51
- earliest_cr_line | dtype=object | unique=733
- application_type | dtype=object | unique=2

🟡 Categorical features still exist.
Encoding is required before the final model matrix.


In [271]:
# Cell 11 — Categorical Cardinality Audit

print("CATEGORICAL CARDINALITY AUDIT")
print("=" * 70)

categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for column in categorical_columns:

    print(
        f"{column:30s} | "
        f"unique={X_train[column].nunique(dropna=False):>8,} | "
        f"missing={X_train[column].isna().sum():>8,}"
    )

print("\nTotal categorical features:", len(categorical_columns))

CATEGORICAL CARDINALITY AUDIT
term                           | unique=       2 | missing=       0
grade                          | unique=       7 | missing=       0
sub_grade                      | unique=      35 | missing=       0
emp_title                      | unique= 317,883 | missing=       0
emp_length                     | unique=      12 | missing=       0
home_ownership                 | unique=       6 | missing=       0
verification_status            | unique=       3 | missing=       0
purpose                        | unique=      14 | missing=       0
title                          | unique=  52,135 | missing=       0
zip_code                       | unique=     936 | missing=       0
addr_state                     | unique=      51 | missing=       0
earliest_cr_line               | unique=     733 | missing=       0
application_type               | unique=       2 | missing=       0

Total categorical features: 13


In [272]:
# Cell 12 — Encoding Strategy Audit

print("CATEGORICAL ENCODING STRATEGY")
print("=" * 70)

low_cardinality = [
    "term",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "application_type"
]

medium_cardinality = [
    "zip_code",
    "earliest_cr_line"
]

high_cardinality = [
    "emp_title",
    "title"
]

print("LOW-CARDINALITY FEATURES")
print("-" * 70)
for column in low_cardinality:
    print(
        f"- {column}: "
        f"{X_train[column].nunique()} unique values"
    )

print("\nMEDIUM-CARDINALITY FEATURES")
print("-" * 70)
for column in medium_cardinality:
    print(
        f"- {column}: "
        f"{X_train[column].nunique()} unique values"
    )

print("\nHIGH-CARDINALITY FEATURES")
print("-" * 70)
for column in high_cardinality:
    print(
        f"- {column}: "
        f"{X_train[column].nunique():,} unique values"
    )

print("\nENCODING PLAN")
print("-" * 70)
print("Low-cardinality  → One-hot encoding")
print("Medium-cardinality → One-hot encoding")
print("emp_title        → Frequency encoding")
print("title            → Frequency encoding")

print("\n🟢 Encoding strategy defined.")

CATEGORICAL ENCODING STRATEGY
LOW-CARDINALITY FEATURES
----------------------------------------------------------------------
- term: 2 unique values
- grade: 7 unique values
- sub_grade: 35 unique values
- emp_length: 12 unique values
- home_ownership: 6 unique values
- verification_status: 3 unique values
- purpose: 14 unique values
- addr_state: 51 unique values
- application_type: 2 unique values

MEDIUM-CARDINALITY FEATURES
----------------------------------------------------------------------
- zip_code: 936 unique values
- earliest_cr_line: 733 unique values

HIGH-CARDINALITY FEATURES
----------------------------------------------------------------------
- emp_title: 317,883 unique values
- title: 52,135 unique values

ENCODING PLAN
----------------------------------------------------------------------
Low-cardinality  → One-hot encoding
Medium-cardinality → One-hot encoding
emp_title        → Frequency encoding
title            → Frequency encoding

🟢 Encoding strategy defined.

In [273]:
# Cell 13 — Fit Training-Set Frequency Mappings

print("FITTING TRAINING-SET FREQUENCY ENCODINGS")
print("=" * 70)

high_cardinality_columns = [
    "emp_title",
    "title"
]

frequency_maps = {}

for column in high_cardinality_columns:

    frequency_map = (
        X_train[column]
        .value_counts(normalize=True, dropna=False)
        .to_dict()
    )

    frequency_maps[column] = frequency_map

    print(f"\n{column}")
    print("-" * 70)
    print("Unique training values:", len(frequency_map))
    print("Most frequent values:")

    print(
        X_train[column]
        .value_counts(dropna=False)
        .head(10)
    )

print("\n🟢 Training-set frequency mappings created.")
print("🟢 Validation and test data were NOT used to calculate frequencies.")

FITTING TRAINING-SET FREQUENCY ENCODINGS

emp_title
----------------------------------------------------------------------
Unique training values: 317883
Most frequent values:
emp_title
Missing             68874
Teacher             17030
Manager             15682
Owner                8273
Registered Nurse     6972
RN                   6797
Supervisor           6636
Driver               6033
Sales                5964
Project Manager      5154
Name: count, dtype: int64

title
----------------------------------------------------------------------
Unique training values: 52135
Most frequent values:
title
Debt consolidation         528613
Credit card refinancing    199893
Home improvement            60291
Other                       53370
Major purchase              19216
Missing                     13277
Debt Consolidation          12687
Medical expenses            10867
Business                     9484
Car financing                8983
Name: count, dtype: int64

🟢 Training-set frequency 

In [274]:
# Cell 14 — Apply Frequency Encoding

print("APPLYING FREQUENCY ENCODING")
print("=" * 70)

frequency_encoded_columns = []

for column in high_cardinality_columns:

    encoded_column = f"{column}_frequency"

    # Training
    X_train[encoded_column] = (
        X_train[column]
        .map(frequency_maps[column])
        .fillna(0.0)
        .astype("float32")
    )

    # Validation
    X_val[encoded_column] = (
        X_val[column]
        .map(frequency_maps[column])
        .fillna(0.0)
        .astype("float32")
    )

    # Test
    X_test[encoded_column] = (
        X_test[column]
        .map(frequency_maps[column])
        .fillna(0.0)
        .astype("float32")
    )

    frequency_encoded_columns.append(encoded_column)

    print(f"\n{column} → {encoded_column}")

    print(
        "Train:",
        X_train[encoded_column].min(),
        "to",
        X_train[encoded_column].max()
    )

    print(
        "Validation:",
        X_val[encoded_column].min(),
        "to",
        X_val[encoded_column].max()
    )

    print(
        "Test:",
        X_test[encoded_column].min(),
        "to",
        X_test[encoded_column].max()
    )

print("\nFrequency-encoded columns:")
for column in frequency_encoded_columns:
    print("-", column)

print("\n🟢 Frequency encoding applied successfully.")

APPLYING FREQUENCY ENCODING


C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\2727909879.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[encoded_column] = (
C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\2727909879.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_val[encoded_column] = (



emp_title → emp_title_frequency
Train: 9.272317811337416e-07 to 0.0638621598482132
Validation: 0.0 to 0.0638621598482132
Test: 0.0 to 0.0638621598482132

title → title_frequency
Train: 9.272317811337416e-07 to 0.49014678597450256
Validation: 0.0 to 0.49014678597450256
Test: 0.0 to 0.49014678597450256

Frequency-encoded columns:
- emp_title_frequency
- title_frequency

🟢 Frequency encoding applied successfully.


C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\2727909879.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[encoded_column] = (
C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\2727909879.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[encoded_column] = (
C:\Users\mitansh\AppData\Local\Temp\ipykernel_6020\2727909879.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

In [275]:
# Cell 15 — Frequency Encoding Integrity Audit

print("FREQUENCY ENCODING INTEGRITY AUDIT")
print("=" * 70)

for dataset_name, dataset in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test)
]:
    print(f"\n{dataset_name}")
    print("-" * 70)

    for column in frequency_encoded_columns:

        values = dataset[column]

        print(
            f"{column}: "
            f"dtype={values.dtype}, "
            f"missing={values.isna().sum()}, "
            f"min={values.min():.10f}, "
            f"max={values.max():.10f}, "
            f"unique={values.nunique():,}"
        )

        if not values.between(0, 1).all():
            raise ValueError(
                f"{column} contains values outside [0, 1]."
            )

        if values.isna().any():
            raise ValueError(
                f"{column} contains missing values."
            )

print("\n🟢 All frequency-encoded columns passed integrity checks.")

FREQUENCY ENCODING INTEGRITY AUDIT

X_train
----------------------------------------------------------------------
emp_title_frequency: dtype=float32, missing=0, min=0.0000009272, max=0.0638621598, unique=515
title_frequency: dtype=float32, missing=0, min=0.0000009272, max=0.4901467860, unique=213

X_val
----------------------------------------------------------------------
emp_title_frequency: dtype=float32, missing=0, min=0.0000000000, max=0.0638621598, unique=516
title_frequency: dtype=float32, missing=0, min=0.0000000000, max=0.4901467860, unique=214

X_test
----------------------------------------------------------------------
emp_title_frequency: dtype=float32, missing=0, min=0.0000000000, max=0.0638621598, unique=516
title_frequency: dtype=float32, missing=0, min=0.0000000000, max=0.4901467860, unique=214

🟢 All frequency-encoded columns passed integrity checks.


In [276]:
# Cell 16 — Remove Original High-Cardinality Columns

print("REMOVING ORIGINAL HIGH-CARDINALITY FEATURES")
print("=" * 70)

columns_to_drop = [
    "emp_title",
    "title"
]

print("Columns scheduled for removal:")
for column in columns_to_drop:
    print("-", column)

for dataset_name, dataset in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test)
]:
    missing_columns = [
        column
        for column in columns_to_drop
        if column not in dataset.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing expected columns: "
            f"{missing_columns}"
        )

X_train = X_train.drop(columns=columns_to_drop)
X_val = X_val.drop(columns=columns_to_drop)
X_test = X_test.drop(columns=columns_to_drop)

print("\nRemaining original high-cardinality columns:")
for column in columns_to_drop:
    print(
        f"{column}: "
        f"X_train={column in X_train.columns}, "
        f"X_val={column in X_val.columns}, "
        f"X_test={column in X_test.columns}"
    )

print("\nUpdated shapes:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\n🟢 Original high-cardinality columns removed.")
print("🟢 Frequency-encoded replacements retained.")

REMOVING ORIGINAL HIGH-CARDINALITY FEATURES
Columns scheduled for removal:
- emp_title
- title

Remaining original high-cardinality columns:
emp_title: X_train=False, X_val=False, X_test=False
title: X_train=False, X_val=False, X_test=False

Updated shapes:
X_train: (1078479, 153)
X_val: (134810, 153)
X_test: (134810, 153)

🟢 Original high-cardinality columns removed.
🟢 Frequency-encoded replacements retained.


In [277]:
# Cell 17 — Prepare Remaining Categorical Features

print("PREPARING REMAINING CATEGORICAL FEATURES")
print("=" * 70)

categorical_columns = [
    column
    for column in X_train.select_dtypes(
        include=["object", "category"]
    ).columns
]

print("Remaining categorical features:")
for column in categorical_columns:
    print(
        f"- {column} | "
        f"unique={X_train[column].nunique(dropna=False)}"
    )

print(
    "\nTotal remaining categorical features:",
    len(categorical_columns)
)

if len(categorical_columns) != 11:
    raise ValueError(
        f"Expected 11 remaining categorical features, "
        f"found {len(categorical_columns)}."
    )

print(
    "\n🟢 Remaining categorical features identified."
)

PREPARING REMAINING CATEGORICAL FEATURES
Remaining categorical features:
- term | unique=2
- grade | unique=7
- sub_grade | unique=35
- emp_length | unique=12
- home_ownership | unique=6
- verification_status | unique=3
- purpose | unique=14
- zip_code | unique=936
- addr_state | unique=51
- earliest_cr_line | unique=733
- application_type | unique=2

Total remaining categorical features: 11

🟢 Remaining categorical features identified.


In [278]:
# Cell 18 — Fit One-Hot Encoder on Training Data

from sklearn.preprocessing import OneHotEncoder

print("FITTING ONE-HOT ENCODER")
print("=" * 70)

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype="float32"
)

onehot_encoder.fit(
    X_train[categorical_columns]
)

print("Categorical features fitted:")
for column in categorical_columns:
    print(
        f"- {column}: "
        f"{len(onehot_encoder.categories_[categorical_columns.index(column)])} categories"
    )

total_onehot_features = sum(
    len(categories)
    for categories in onehot_encoder.categories_
)

print("\nTotal one-hot encoded features:", total_onehot_features)

print("\n🟢 One-hot encoder fitted successfully.")
print("🟢 Encoder learned from X_train only.")
print("🟢 handle_unknown='ignore' protects validation/test from unseen categories.")

FITTING ONE-HOT ENCODER
Categorical features fitted:
- term: 2 categories
- grade: 7 categories
- sub_grade: 35 categories
- emp_length: 12 categories
- home_ownership: 6 categories
- verification_status: 3 categories
- purpose: 14 categories
- zip_code: 936 categories
- addr_state: 51 categories
- earliest_cr_line: 733 categories
- application_type: 2 categories

Total one-hot encoded features: 1801

🟢 One-hot encoder fitted successfully.
🟢 Encoder learned from X_train only.
🟢 handle_unknown='ignore' protects validation/test from unseen categories.


In [279]:
# Cell 19 — Transform Categorical Features

print("TRANSFORMING CATEGORICAL FEATURES")
print("=" * 70)

X_train_cat = onehot_encoder.transform(
    X_train[categorical_columns]
)

X_val_cat = onehot_encoder.transform(
    X_val[categorical_columns]
)

X_test_cat = onehot_encoder.transform(
    X_test[categorical_columns]
)

print("Encoded categorical matrix shapes:")

print("X_train_cat:", X_train_cat.shape)
print("X_val_cat:", X_val_cat.shape)
print("X_test_cat:", X_test_cat.shape)

expected_features = 1801

if X_train_cat.shape[1] != expected_features:
    raise ValueError(
        f"Expected {expected_features} encoded features, "
        f"got {X_train_cat.shape[1]}"
    )

if X_val_cat.shape[1] != expected_features:
    raise ValueError(
        f"Validation feature count mismatch: "
        f"{X_val_cat.shape[1]}"
    )

if X_test_cat.shape[1] != expected_features:
    raise ValueError(
        f"Test feature count mismatch: "
        f"{X_test_cat.shape[1]}"
    )

print("\n🟢 Categorical transformation completed.")
print("🟢 Train / validation / test feature counts match.")

TRANSFORMING CATEGORICAL FEATURES
Encoded categorical matrix shapes:
X_train_cat: (1078479, 1801)
X_val_cat: (134810, 1801)
X_test_cat: (134810, 1801)

🟢 Categorical transformation completed.
🟢 Train / validation / test feature counts match.


In [280]:
# Cell 20 — Remove Original Categorical Features

print("REMOVING ORIGINAL CATEGORICAL FEATURES")
print("=" * 70)

print("Categorical columns scheduled for removal:")

for column in categorical_columns:
    print("-", column)

# Remove original categorical columns
X_train_numeric = X_train.drop(columns=categorical_columns).copy()
X_val_numeric = X_val.drop(columns=categorical_columns).copy()
X_test_numeric = X_test.drop(columns=categorical_columns).copy()

print("\nShapes after removing categorical columns:")

print("X_train_numeric:", X_train_numeric.shape)
print("X_val_numeric:", X_val_numeric.shape)
print("X_test_numeric:", X_test_numeric.shape)

# Verify no categorical columns remain
remaining_train_categorical = X_train_numeric.select_dtypes(
    include=["object", "category"]
).columns.tolist()

remaining_val_categorical = X_val_numeric.select_dtypes(
    include=["object", "category"]
).columns.tolist()

remaining_test_categorical = X_test_numeric.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nRemaining categorical columns:")
print("X_train:", remaining_train_categorical)
print("X_val:", remaining_val_categorical)
print("X_test:", remaining_test_categorical)

if remaining_train_categorical:
    raise TypeError(
        f"X_train still contains categorical columns: "
        f"{remaining_train_categorical}"
    )

if remaining_val_categorical:
    raise TypeError(
        f"X_val still contains categorical columns: "
        f"{remaining_val_categorical}"
    )

if remaining_test_categorical:
    raise TypeError(
        f"X_test still contains categorical columns: "
        f"{remaining_test_categorical}"
    )

print("\n🟢 Original categorical features removed.")
print("🟢 Numeric and frequency-encoded features retained.")
print("🟢 No categorical features remain.")

REMOVING ORIGINAL CATEGORICAL FEATURES
Categorical columns scheduled for removal:
- term
- grade
- sub_grade
- emp_length
- home_ownership
- verification_status
- purpose
- zip_code
- addr_state
- earliest_cr_line
- application_type

Shapes after removing categorical columns:
X_train_numeric: (1078479, 142)
X_val_numeric: (134810, 142)
X_test_numeric: (134810, 142)

Remaining categorical columns:
X_train: []
X_val: []
X_test: []

🟢 Original categorical features removed.
🟢 Numeric and frequency-encoded features retained.
🟢 No categorical features remain.


In [281]:
# Cell 21 — Build Final Model Matrices

print("BUILDING FINAL MODEL MATRICES")
print("=" * 70)

# Convert one-hot arrays to DataFrames
X_train_cat_df = pd.DataFrame(
    X_train_cat,
    index=X_train_numeric.index
)

X_val_cat_df = pd.DataFrame(
    X_val_cat,
    index=X_val_numeric.index
)

X_test_cat_df = pd.DataFrame(
    X_test_cat,
    index=X_test_numeric.index
)

# Give encoded columns unique names
onehot_feature_names = onehot_encoder.get_feature_names_out(
    categorical_columns
)

X_train_cat_df.columns = onehot_feature_names
X_val_cat_df.columns = onehot_feature_names
X_test_cat_df.columns = onehot_feature_names

# Combine numeric and one-hot features
X_train_final = pd.concat(
    [X_train_numeric, X_train_cat_df],
    axis=1
)

X_val_final = pd.concat(
    [X_val_numeric, X_val_cat_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_numeric, X_test_cat_df],
    axis=1
)

print("Final model matrix shapes:")
print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

expected_feature_count = 142 + 1801

if X_train_final.shape[1] != expected_feature_count:
    raise ValueError(
        f"X_train expected {expected_feature_count} features, "
        f"got {X_train_final.shape[1]}"
    )

if X_val_final.shape[1] != expected_feature_count:
    raise ValueError(
        f"X_val expected {expected_feature_count} features, "
        f"got {X_val_final.shape[1]}"
    )

if X_test_final.shape[1] != expected_feature_count:
    raise ValueError(
        f"X_test expected {expected_feature_count} features, "
        f"got {X_test_final.shape[1]}"
    )

print("\nExpected features:", expected_feature_count)

print("\n🟢 Numeric + one-hot features combined.")
print("🟢 Train / validation / test feature counts match.")

BUILDING FINAL MODEL MATRICES
Final model matrix shapes:
X_train_final: (1078479, 1943)
X_val_final: (134810, 1943)
X_test_final: (134810, 1943)

Expected features: 1943

🟢 Numeric + one-hot features combined.
🟢 Train / validation / test feature counts match.


In [282]:
# Cell 22 — Final Column Alignment Audit

print("FINAL COLUMN ALIGNMENT AUDIT")
print("=" * 70)

train_columns = X_train_final.columns
val_columns = X_val_final.columns
test_columns = X_test_final.columns

print("Train / Validation columns identical:",
      train_columns.equals(val_columns))

print("Train / Test columns identical:",
      train_columns.equals(test_columns))

print("Validation / Test columns identical:",
      val_columns.equals(test_columns))

if not train_columns.equals(val_columns):
    raise ValueError(
        "X_train_final and X_val_final columns do not match."
    )

if not train_columns.equals(test_columns):
    raise ValueError(
        "X_train_final and X_test_final columns do not match."
    )

# Check duplicate feature names
duplicate_columns = train_columns[
    train_columns.duplicated()
].tolist()

print("\nDuplicate feature columns:", len(duplicate_columns))

if duplicate_columns:
    print(duplicate_columns[:20])
    raise ValueError(
        "Duplicate feature names found in final model matrix."
    )

print("\nFeature count:", len(train_columns))

print("\n🟢 Train / validation / test columns are perfectly aligned.")
print("🟢 No duplicate feature names found.")

FINAL COLUMN ALIGNMENT AUDIT
Train / Validation columns identical: True
Train / Test columns identical: True
Validation / Test columns identical: True

Duplicate feature columns: 0

Feature count: 1943

🟢 Train / validation / test columns are perfectly aligned.
🟢 No duplicate feature names found.


In [283]:
# Cell 23 — Ultra Memory-Safe Final Data Integrity Audit

print("FINAL DATA INTEGRITY AUDIT")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():

    print(f"\n{name}")
    print("-" * 70)

    # ------------------------------------------------------------
    # 1. Shape
    # ------------------------------------------------------------
    print("Shape:", dataset.shape)

    # ------------------------------------------------------------
    # 2. Column-by-column missing-value check
    # ------------------------------------------------------------
    missing_columns = []
    missing_total = 0

    for column in dataset.columns:

        count = dataset[column].isna().sum()

        if count > 0:
            missing_columns.append(column)
            missing_total += int(count)

    print("Missing values:", missing_total)
    print("Columns containing missing values:", len(missing_columns))

    if missing_columns:
        print("Affected columns:", missing_columns[:20])

        raise ValueError(
            f"{name} contains missing values."
        )

    # ------------------------------------------------------------
    # 3. Column-by-column numeric dtype check
    # ------------------------------------------------------------
    non_numeric_columns = []

    for column in dataset.columns:

        if not pd.api.types.is_numeric_dtype(
            dataset[column]
        ):
            non_numeric_columns.append(column)

    print("Non-numeric columns:", len(non_numeric_columns))

    if non_numeric_columns:
        print("Affected columns:", non_numeric_columns[:20])

        raise TypeError(
            f"{name} contains non-numeric columns."
        )

    # ------------------------------------------------------------
    # 4. Column-by-column infinite-value check
    # ------------------------------------------------------------
    infinite_columns = []
    infinite_total = 0

    for column in dataset.columns:

        values = dataset[column].to_numpy(copy=False)

        count = np.isinf(values).sum()

        if count > 0:
            infinite_columns.append(column)
            infinite_total += int(count)

    print("Infinite values:", infinite_total)
    print("Columns containing infinite values:",
          len(infinite_columns))

    if infinite_columns:
        print("Affected columns:", infinite_columns[:20])

        raise ValueError(
            f"{name} contains infinite values."
        )

    print("✅ Missing values: 0")
    print("✅ Non-numeric columns: 0")
    print("✅ Infinite values: 0")


print("\n" + "=" * 70)
print("🟢 FINAL DATA INTEGRITY AUDIT PASSED")
print("=" * 70)

FINAL DATA INTEGRITY AUDIT

X_train_final
----------------------------------------------------------------------
Shape: (1078479, 1943)
Missing values: 0
Columns containing missing values: 0
Non-numeric columns: 0
Infinite values: 0
Columns containing infinite values: 0
✅ Missing values: 0
✅ Non-numeric columns: 0
✅ Infinite values: 0

X_val_final
----------------------------------------------------------------------
Shape: (134810, 1943)
Missing values: 0
Columns containing missing values: 0
Non-numeric columns: 0
Infinite values: 0
Columns containing infinite values: 0
✅ Missing values: 0
✅ Non-numeric columns: 0
✅ Infinite values: 0

X_test_final
----------------------------------------------------------------------
Shape: (134810, 1943)
Missing values: 0
Columns containing missing values: 0
Non-numeric columns: 0
Infinite values: 0
Columns containing infinite values: 0
✅ Missing values: 0
✅ Non-numeric columns: 0
✅ Infinite values: 0

🟢 FINAL DATA INTEGRITY AUDIT PASSED


In [284]:
# Cell 24 — Final Feature Statistics Audit

print("FINAL FEATURE STATISTICS AUDIT")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

expected_features = 1943

for name, dataset in datasets.items():

    print(f"\n{name}")
    print("-" * 70)

    print("Rows:", len(dataset))
    print("Columns:", dataset.shape[1])

    if dataset.shape[1] != expected_features:
        raise ValueError(
            f"{name} has {dataset.shape[1]} features. "
            f"Expected {expected_features}."
        )

    # Check duplicate feature names
    duplicate_count = int(
        dataset.columns.duplicated().sum()
    )

    print("Duplicate feature names:", duplicate_count)

    if duplicate_count != 0:
        raise ValueError(
            f"{name} contains duplicate feature names."
        )

    # Check dtype counts without creating a huge temporary matrix
    dtype_counts = {}

    for column in dataset.columns:
        dtype_name = str(dataset[column].dtype)
        dtype_counts[dtype_name] = (
            dtype_counts.get(dtype_name, 0) + 1
        )

    print("Data types:")

    for dtype_name, count in dtype_counts.items():
        print(f"- {dtype_name}: {count}")

print("\n" + "=" * 70)
print("🟢 FINAL FEATURE STATISTICS AUDIT PASSED")
print("=" * 70)

FINAL FEATURE STATISTICS AUDIT

X_train_final
----------------------------------------------------------------------
Rows: 1078479
Columns: 1943
Duplicate feature names: 0
Data types:
- float64: 73
- int8: 67
- float32: 1803

X_val_final
----------------------------------------------------------------------
Rows: 134810
Columns: 1943
Duplicate feature names: 0
Data types:
- float64: 73
- int8: 67
- float32: 1803

X_test_final
----------------------------------------------------------------------
Rows: 134810
Columns: 1943
Duplicate feature names: 0
Data types:
- float64: 73
- int8: 67
- float32: 1803

🟢 FINAL FEATURE STATISTICS AUDIT PASSED


**Step 16: Final model-readiness preparation**

In [285]:
# STEP 16 — FINAL MODEL READINESS
# Cell 1 — Final Overview

print("STEP 16 — FINAL MODEL READINESS CHECK")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():
    print(f"\n{name}")
    print("-" * 70)
    print("Shape:", dataset.shape)
    print("Features:", dataset.shape[1])
    print("Rows:", dataset.shape[0])

print("\nTarget shapes:")
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\n🟢 Final model matrices detected successfully.")

STEP 16 — FINAL MODEL READINESS CHECK

X_train_final
----------------------------------------------------------------------
Shape: (1078479, 1943)
Features: 1943
Rows: 1078479

X_val_final
----------------------------------------------------------------------
Shape: (134810, 1943)
Features: 1943
Rows: 134810

X_test_final
----------------------------------------------------------------------
Shape: (134810, 1943)
Features: 1943
Rows: 134810

Target shapes:
y_train: (1078479,)
y_val: (134810,)
y_test: (134810,)

🟢 Final model matrices detected successfully.


In [286]:
# Cell 2 — Target Integrity Audit

print("TARGET INTEGRITY AUDIT")
print("=" * 70)

targets = {
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}

for name, target in targets.items():

    print(f"\n{name}")
    print("-" * 70)

    print("Rows:", len(target))
    print("Missing values:", int(target.isna().sum()))

    print("\nTarget distribution:")
    print(target.value_counts(dropna=False).sort_index())

    print("\nTarget proportions:")
    print(
        target.value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    if target.isna().sum() != 0:
        raise ValueError(
            f"{name} contains missing target values."
        )

print("\n🟢 Target integrity audit passed.")

TARGET INTEGRITY AUDIT

y_train
----------------------------------------------------------------------
Rows: 1078479
Missing values: 0

Target distribution:
target
0.0    862991
1.0    215488
Name: count, dtype: int64

Target proportions:
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64

y_val
----------------------------------------------------------------------
Rows: 134810
Missing values: 0

Target distribution:
target
0.0    107874
1.0     26936
Name: count, dtype: int64

Target proportions:
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64

y_test
----------------------------------------------------------------------
Rows: 134810
Missing values: 0

Target distribution:
target
0.0    107874
1.0     26936
Name: count, dtype: int64

Target proportions:
target
0.0    0.800193
1.0    0.199807
Name: proportion, dtype: float64

🟢 Target integrity audit passed.


In [287]:
# Cell 3 — Feature / Target Alignment

print("FEATURE-TARGET ALIGNMENT")
print("=" * 70)

checks = {
    "Train": len(X_train_final) == len(y_train),
    "Validation": len(X_val_final) == len(y_val),
    "Test": len(X_test_final) == len(y_test)
}

for name, passed in checks.items():
    print(f"{name} alignment:", passed)

if not all(checks.values()):
    raise ValueError(
        "Feature and target row counts do not match."
    )

print("\n🟢 Feature-target alignment passed.")

FEATURE-TARGET ALIGNMENT
Train alignment: True
Validation alignment: True
Test alignment: True

🟢 Feature-target alignment passed.


In [288]:
# Cell 4 — Final Feature Consistency

print("FINAL FEATURE CONSISTENCY")
print("=" * 70)

train_columns = X_train_final.columns
val_columns = X_val_final.columns
test_columns = X_test_final.columns

print(
    "Train / Validation columns identical:",
    train_columns.equals(val_columns)
)

print(
    "Train / Test columns identical:",
    train_columns.equals(test_columns)
)

print(
    "Validation / Test columns identical:",
    val_columns.equals(test_columns)
)

print(
    "Feature count:",
    len(train_columns)
)

if not (
    train_columns.equals(val_columns)
    and train_columns.equals(test_columns)
):
    raise ValueError(
        "Train, validation, and test feature columns are not identical."
    )

print("\n🟢 Feature consistency passed.")

FINAL FEATURE CONSISTENCY
Train / Validation columns identical: True
Train / Test columns identical: True
Validation / Test columns identical: True
Feature count: 1943

🟢 Feature consistency passed.


In [289]:
# Cell 5 — Numeric / Finite Value Verification

import numpy as np
import pandas as pd

print("NUMERIC AND FINITE-VALUE VERIFICATION")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():

    print(f"\n{name}")
    print("-" * 70)

    non_numeric_columns = [
        column
        for column in dataset.columns
        if not pd.api.types.is_numeric_dtype(dataset[column])
    ]

    infinite_columns = []

    for column in dataset.columns:

        series = dataset[column]

        if pd.api.types.is_numeric_dtype(series):

            if not np.isfinite(series.to_numpy()).all():
                infinite_columns.append(column)

    print("Non-numeric columns:", len(non_numeric_columns))
    print("Columns containing infinite values:", len(infinite_columns))

    if non_numeric_columns:
        print("Non-numeric:", non_numeric_columns[:10])
        raise TypeError(
            f"{name} contains non-numeric features."
        )

    if infinite_columns:
        print("Infinite:", infinite_columns[:10])
        raise ValueError(
            f"{name} contains infinite values."
        )

print("\n🟢 Numeric and finite-value verification passed.")

NUMERIC AND FINITE-VALUE VERIFICATION

X_train_final
----------------------------------------------------------------------
Non-numeric columns: 0
Columns containing infinite values: 0

X_val_final
----------------------------------------------------------------------
Non-numeric columns: 0
Columns containing infinite values: 0

X_test_final
----------------------------------------------------------------------
Non-numeric columns: 0
Columns containing infinite values: 0

🟢 Numeric and finite-value verification passed.


In [290]:
# Cell 6 — Final Missing-Value Verification

print("FINAL MISSING-VALUE VERIFICATION")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():

    missing_columns = []

    for column in dataset.columns:

        if dataset[column].isna().any():
            missing_columns.append(column)

    print(f"\n{name}")
    print("Columns containing missing values:", len(missing_columns))

    if missing_columns:
        print(missing_columns[:20])
        raise ValueError(
            f"{name} still contains missing values."
        )

print("\n🟢 No missing values detected.")

FINAL MISSING-VALUE VERIFICATION

X_train_final
Columns containing missing values: 0

X_val_final
Columns containing missing values: 0

X_test_final
Columns containing missing values: 0

🟢 No missing values detected.


In [291]:
# Cell 7 — Final Dataset Size and Memory Audit

print("FINAL DATASET SIZE AUDIT")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():

    memory_mb = (
        dataset.memory_usage(deep=True).sum()
        / (1024 ** 2)
    )

    print(f"\n{name}")
    print("Shape:", dataset.shape)
    print(f"Memory usage: {memory_mb:.2f} MB")

print("\n🟢 Dataset size audit completed.")

FINAL DATASET SIZE AUDIT

X_train_final
Shape: (1078479, 1943)
Memory usage: 8095.46 MB

X_val_final
Shape: (134810, 1943)
Memory usage: 1011.93 MB

X_test_final
Shape: (134810, 1943)
Memory usage: 1011.93 MB

🟢 Dataset size audit completed.


In [292]:
# Cell 8 — STEP 16 FINAL GATE

print("STEP 16 — FINAL MODEL READINESS GATE")
print("=" * 70)

expected_features = 1943

checks = []

# Shape checks
checks.append(
    X_train_final.shape[1] == expected_features
)

checks.append(
    X_val_final.shape[1] == expected_features
)

checks.append(
    X_test_final.shape[1] == expected_features
)

# Row alignment
checks.append(
    len(X_train_final) == len(y_train)
)

checks.append(
    len(X_val_final) == len(y_val)
)

checks.append(
    len(X_test_final) == len(y_test)
)

# Column alignment
checks.append(
    X_train_final.columns.equals(X_val_final.columns)
)

checks.append(
    X_train_final.columns.equals(X_test_final.columns)
)

# Target missingness
checks.append(
    y_train.isna().sum() == 0
)

checks.append(
    y_val.isna().sum() == 0
)

checks.append(
    y_test.isna().sum() == 0
)

if not all(checks):

    print("🔴 FINAL MODEL READINESS GATE FAILED")

    failed = [
        index + 1
        for index, passed in enumerate(checks)
        if not passed
    ]

    print("Failed checks:", failed)

    raise ValueError(
        "Step 16 is not complete."
    )

print("All readiness checks passed.")

print("\n" + "=" * 70)
print("🟢 STEP 16 — FINAL MODEL READINESS COMPLETE")
print("🟢 PREPROCESSING PIPELINE COMPLETE")
print("🟢 DATA IS READY FOR MODEL DEVELOPMENT")
print("=" * 70)

STEP 16 — FINAL MODEL READINESS GATE
All readiness checks passed.

🟢 STEP 16 — FINAL MODEL READINESS COMPLETE
🟢 PREPROCESSING PIPELINE COMPLETE
🟢 DATA IS READY FOR MODEL DEVELOPMENT


In [293]:
# Cell 9 — SAFE FINAL PREPROCESSED DATA EXPORT

from pathlib import Path
import os

print("FINAL PREPROCESSED DATA EXPORT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify final matrices exist
# ------------------------------------------------------------

required_datasets = [
    "X_train_final",
    "X_val_final",
    "X_test_final"
]

missing_datasets = [
    name
    for name in required_datasets
    if name not in globals()
]

if missing_datasets:
    raise NameError(
        "The following final datasets are not available: "
        + ", ".join(missing_datasets)
        + "\nRun the preprocessing cells that create the final "
          "model matrices before running this export cell."
    )

# ------------------------------------------------------------
# 2. Verify expected feature count
# ------------------------------------------------------------

EXPECTED_FEATURES = 1943

for name, dataset in {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}.items():

    print(f"\n{name}")
    print("Shape:", dataset.shape)

    if dataset.shape[1] != EXPECTED_FEATURES:
        raise ValueError(
            f"{name} has {dataset.shape[1]} features. "
            f"Expected {EXPECTED_FEATURES}."
        )

# ------------------------------------------------------------
# 3. Verify column alignment
# ------------------------------------------------------------

if not X_train_final.columns.equals(X_val_final.columns):
    raise ValueError(
        "X_train_final and X_val_final columns do not match."
    )

if not X_train_final.columns.equals(X_test_final.columns):
    raise ValueError(
        "X_train_final and X_test_final columns do not match."
    )

print("\n🟢 Feature alignment verified.")

# ------------------------------------------------------------
# 4. Create output directory
# ------------------------------------------------------------

output_dir = Path("final_preprocessed_data")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("\nOutput directory:")
print(output_dir.resolve())

# ------------------------------------------------------------
# 5. Define output files
# ------------------------------------------------------------

train_path = output_dir / "X_train_final.csv.gz"
val_path = output_dir / "X_val_final.csv.gz"
test_path = output_dir / "X_test_final.csv.gz"

# ------------------------------------------------------------
# 6. Export TRAIN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1/3 — SAVING X_train_final")
print("=" * 70)

X_train_final.to_csv(
    train_path,
    index=False,
    compression="gzip"
)

if not train_path.exists():
    raise IOError(
        "X_train_final export failed."
    )

print("🟢 X_train_final saved successfully.")
print(
    f"File size: "
    f"{train_path.stat().st_size / (1024 ** 3):.2f} GB"
)

# ------------------------------------------------------------
# 7. Export VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2/3 — SAVING X_val_final")
print("=" * 70)

X_val_final.to_csv(
    val_path,
    index=False,
    compression="gzip"
)

if not val_path.exists():
    raise IOError(
        "X_val_final export failed."
    )

print("🟢 X_val_final saved successfully.")
print(
    f"File size: "
    f"{val_path.stat().st_size / (1024 ** 3):.2f} GB"
)

# ------------------------------------------------------------
# 8. Export TEST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3/3 — SAVING X_test_final")
print("=" * 70)

X_test_final.to_csv(
    test_path,
    index=False,
    compression="gzip"
)

if not test_path.exists():
    raise IOError(
        "X_test_final export failed."
    )

print("🟢 X_test_final saved successfully.")
print(
    f"File size: "
    f"{test_path.stat().st_size / (1024 ** 3):.2f} GB"
)

# ------------------------------------------------------------
# 9. Final export summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🟢 FINAL PREPROCESSED DATA EXPORT COMPLETE")
print("=" * 70)

print("\nFiles:")

print(
    f"X_train_final : {train_path.resolve()}"
)

print(
    f"X_val_final   : {val_path.resolve()}"
)

print(
    f"X_test_final  : {test_path.resolve()}"
)

print("\nFinal shapes:")

print("X_train_final:", X_train_final.shape)
print("X_val_final:", X_val_final.shape)
print("X_test_final:", X_test_final.shape)

print("\n🟢 All three final datasets have been exported.")

FINAL PREPROCESSED DATA EXPORT

X_train_final
Shape: (1078479, 1943)

X_val_final
Shape: (134810, 1943)

X_test_final
Shape: (134810, 1943)

🟢 Feature alignment verified.

Output directory:
C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data

1/3 — SAVING X_train_final
🟢 X_train_final saved successfully.
File size: 0.17 GB

2/3 — SAVING X_val_final
🟢 X_val_final saved successfully.
File size: 0.02 GB

3/3 — SAVING X_test_final
🟢 X_test_final saved successfully.
File size: 0.02 GB

🟢 FINAL PREPROCESSED DATA EXPORT COMPLETE

Files:
X_train_final : C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_train_final.csv.gz
X_val_final   : C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_val_final.csv.gz
X_test_final  : C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_test_final.csv.gz

Final shapes:
X_tra

In [295]:
# Cell 10 — Verify Final Preprocessed Dataset Exports

from pathlib import Path

print("FINAL PREPROCESSED DATA EXPORT AUDIT")
print("=" * 70)

output_dir = Path("final_preprocessed_data")

files = {
    "X_train_final": output_dir / "X_train_final.csv.gz",
    "X_val_final": output_dir / "X_val_final.csv.gz",
    "X_test_final": output_dir / "X_test_final.csv.gz"
}

expected_shapes = {
    "X_train_final": (1078479, 1943),
    "X_val_final": (134810, 1943),
    "X_test_final": (134810, 1943)
}

missing_files = []

for name, file_path in files.items():

    print(f"\n{name}")
    print("-" * 70)
    print("Path:", file_path.resolve())

    if not file_path.exists():
        print("❌ FILE NOT FOUND")
        missing_files.append(name)
        continue

    size_gb = file_path.stat().st_size / (1024 ** 3)

    print("Status: ✅ EXISTS")
    print(f"Compressed size: {size_gb:.2f} GB")


# ------------------------------------------------------------
# Final existence check
# ------------------------------------------------------------

if missing_files:

    raise FileNotFoundError(
        "Missing exported files: "
        + ", ".join(missing_files)
    )


# ------------------------------------------------------------
# Verify expected shapes from existing variables
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MATRIX SHAPE CHECK")
print("=" * 70)

datasets = {
    "X_train_final": X_train_final,
    "X_val_final": X_val_final,
    "X_test_final": X_test_final
}

for name, dataset in datasets.items():

    actual_shape = dataset.shape
    expected_shape = expected_shapes[name]

    print(f"\n{name}")
    print("Actual shape:   ", actual_shape)
    print("Expected shape: ", expected_shape)

    if actual_shape != expected_shape:
        raise ValueError(
            f"{name} shape mismatch. "
            f"Expected {expected_shape}, got {actual_shape}."
        )

    print("✅ Shape verified")


# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🟢 FINAL PREPROCESSED DATA EXPORT AUDIT PASSED")
print("=" * 70)

print("\nVerified files:")

for name, file_path in files.items():
    print(f"✅ {name}: {file_path.resolve()}")

print("\nVerified shapes:")
print("X_train_final:", X_train_final.shape)
print("X_val_final:  ", X_val_final.shape)
print("X_test_final: ", X_test_final.shape)

print("\n🟢 All three final preprocessed datasets exist.")
print("🟢 All three datasets have the expected 1943 features.")

FINAL PREPROCESSED DATA EXPORT AUDIT

X_train_final
----------------------------------------------------------------------
Path: C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_train_final.csv.gz
Status: ✅ EXISTS
Compressed size: 0.17 GB

X_val_final
----------------------------------------------------------------------
Path: C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_val_final.csv.gz
Status: ✅ EXISTS
Compressed size: 0.02 GB

X_test_final
----------------------------------------------------------------------
Path: C:\Users\mitansh\Desktop\ML PROJECTS\AI-Credit-Risk-Loan-Approval-Engine\final_preprocessed_data\X_test_final.csv.gz
Status: ✅ EXISTS
Compressed size: 0.02 GB

FINAL MATRIX SHAPE CHECK

X_train_final
Actual shape:    (1078479, 1943)
Expected shape:  (1078479, 1943)
✅ Shape verified

X_val_final
Actual shape:    (134810, 1943)
Expected shape:  (134810, 1943)
✅ Shape veri